# Week 7 — Prescriptive Analytics, RAG, and Evaluation

## Student Skill Gap Analysis and Career Recommendation System

This notebook extends the completed Week 4, Week 5, and Week 6 analysis.

### Week 7 objectives

1. Develop prescriptive skill-gap analysis.
2. Prioritize missing skills using O*NET importance, required level,
   Canadian labour-market demand, cross-occupation relevance, and
   career/education alignment.
3. Generate actionable skill-development recommendations.
4. Build a Retrieval-Augmented Generation (RAG) knowledge base.
5. Retrieve evidence for career explanations.
6. Evaluate skill extraction.
7. Evaluate occupation recommendation performance.
8. Evaluate skill-gap predictions.
9. Evaluate RAG responses.
10. Compare the contribution of O*NET, Job Bank, and CIP information.
11. Conduct error analysis and document limitations.

### Analytical principle

Week 7 builds on the validated outputs from previous weeks rather than
recreating the Week 4–6 models.

The system is treated as a career recommendation and occupation-ranking
system rather than a supervised career-classification model because the
project does not contain a verified candidate-to-career target variable.

In [3]:
# ============================================
# WEEK 7 — IMPORTS
# ============================================

import os
import re
import ast
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    ndcg_score
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

print("Week 7 libraries imported successfully.")

Week 7 libraries imported successfully.


In [4]:
# ============================================
# PROJECT PATHS
# ============================================

PROJECT_ROOT = Path(r"C:\Users\Admin\Capstone_Project")

OUTPUTS_DIR = PROJECT_ROOT / "Outputs" / "Tables"

WEEK5_DIR = OUTPUTS_DIR / "Week5"
WEEK6_DIR = OUTPUTS_DIR / "Week6"
WEEK7_DIR = OUTPUTS_DIR / "Week7"

# Create Week 7 output directory if it does not already exist
WEEK7_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)

print("\nOutput tables folder:")
print(OUTPUTS_DIR)

print("\nWeek 5 folder:")
print(WEEK5_DIR)

print("\nWeek 6 folder:")
print(WEEK6_DIR)

print("\nWeek 7 folder:")
print(WEEK7_DIR)

Project root:
C:\Users\Admin\Capstone_Project

Output tables folder:
C:\Users\Admin\Capstone_Project\Outputs\Tables

Week 5 folder:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week5

Week 6 folder:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week6

Week 7 folder:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7


In [15]:
# ============================================
# O*NET / WEEK 4 OUTPUT PATHS
# ============================================

ONET_ABILITIES = OUTPUTS_DIR / "onet_abilities_profile.csv"

ONET_CIP_INTEGRATED = OUTPUTS_DIR / "onet_cip_integrated_occupation_profile.csv"

ONET_EDUCATION = OUTPUTS_DIR / "onet_education_profile.csv"

ONET_EDUCATION_V2 = OUTPUTS_DIR / "onet_education_profile_v2.csv"

ONET_INTEGRATED = OUTPUTS_DIR / "onet_integrated_occupation_profile.csv"

ONET_JOB_ZONE = OUTPUTS_DIR / "onet_job_zone_profile.csv"

ONET_KNOWLEDGE = OUTPUTS_DIR / "onet_knowledge_profile.csv"

ONET_SKILLS = OUTPUTS_DIR / "onet_skills_profile.csv"

ONET_TASKS = OUTPUTS_DIR / "onet_tasks_profile.csv"

ONET_TASKS_V2 = OUTPUTS_DIR / "onet_tasks_profile_v2.csv"

ONET_TRAINING = OUTPUTS_DIR / "onet_training_profile.csv"

VALIDATED_MAPPING = OUTPUTS_DIR / "validated_occupation_onet_mapping.csv"

WEEK4_SUMMARY = OUTPUTS_DIR / "week4_final_summary.csv"

print("O*NET and Week 4 paths defined.")

O*NET and Week 4 paths defined.


In [17]:
# ============================================
# WEEK 5 OUTPUT PATHS
# ============================================

WEEK5_FILES = {
    "candidate_cip_mapping": WEEK5_DIR / "week5_candidate_cip_mapping.csv",
    "candidate_feature_summary": WEEK5_DIR / "week5_candidate_feature_summary.csv",
    "candidate_features": WEEK5_DIR / "week5_candidate_features.csv",
    "candidate_occupation_cip_mapping": WEEK5_DIR / "week5_candidate_occupation_cip_mapping.csv",
    "candidate_occupation_scores": WEEK5_DIR / "week5_candidate_occupation_scores.csv",
    "candidate_onet_skill_matrix": WEEK5_DIR / "week5_candidate_onet_skill_matrix.csv",
    "cip_quality_validation": WEEK5_DIR / "week5_cip_quality_validation_summary.csv",
    "cip_scoring_trace": WEEK5_DIR / "week5_cip_scoring_trace.csv",
    "cip_scoring_trace_full": WEEK5_DIR / "week5_cip_scoring_trace_full.csv",
    "final_cip_summary": WEEK5_DIR / "week5_final_cip_summary.csv",
    "final_top5_cip": WEEK5_DIR / "week5_final_top5_cip_recommendations.csv",
    "validation_summary": WEEK5_DIR / "week5_validation_summary.csv"
}


# ============================================
# WEEK 6 OUTPUT PATHS
# ============================================

WEEK6_FILES = {
    "all_model_components": WEEK6_DIR / "week6_all_model_components.csv",
    "baseline_vs_hybrid": WEEK6_DIR / "week6_baseline_vs_hybrid.csv",
    "baseline_vs_hybrid_comparison": WEEK6_DIR / "week6_baseline_vs_hybrid_comparison.csv",
    "canadian_labour_demand": WEEK6_DIR / "week6_canadian_labour_demand_scores.csv",
    "confidence_distribution": WEEK6_DIR / "week6_confidence_distribution.csv",
    "education_alignment": WEEK6_DIR / "week6_education_alignment_scores.csv",
    "error_analysis": WEEK6_DIR / "week6_error_analysis.csv",
    "final_confidence_distribution": WEEK6_DIR / "week6_final_confidence_distribution.csv",
    "final_confidence_summary": WEEK6_DIR / "week6_final_confidence_summary.csv",
    "final_hybrid_recommendations": WEEK6_DIR / "week6_final_hybrid_recommendations.csv",
    "final_hybrid_top1": WEEK6_DIR / "week6_final_hybrid_top1_recommendations.csv",
    "final_hybrid_top5": WEEK6_DIR / "week6_final_hybrid_top5_recommendations.csv",
    "final_project_completion": WEEK6_DIR / "week6_final_project_completion_summary.csv",
    "final_recommendation_confidence": WEEK6_DIR / "week6_final_recommendation_confidence.csv",
    "final_summary": WEEK6_DIR / "week6_final_summary.csv",
    "final_top1": WEEK6_DIR / "week6_final_top1_recommendations.csv",
    "final_top5": WEEK6_DIR / "week6_final_top5_recommendations.csv",
    "hybrid_recommendation_scores": WEEK6_DIR / "week6_hybrid_recommendation_scores.csv",
    "keyword_baseline": WEEK6_DIR / "week6_keyword_baseline_recommendations.csv",
    "model_comparison": WEEK6_DIR / "week6_model_comparison.csv",
    "occupation_distribution_comparison": WEEK6_DIR / "week6_occupation_distribution_comparison.csv",
    "occupation_recommendation_distribution": WEEK6_DIR / "week6_occupation_recommendation_distribution.csv",
    "recommendation_confidence_analysis": WEEK6_DIR / "week6_recommendation_confidence_analysis.csv",
    "recommendation_transitions": WEEK6_DIR / "week6_recommendation_transitions.csv",
    "semantic_recommendations": WEEK6_DIR / "week6_semantic_recommendations.csv",
    "tfidf_recommendations": WEEK6_DIR / "week6_tfidf_recommendations.csv",
    "top1_occupation_distribution": WEEK6_DIR / "week6_top1_occupation_distribution.csv",
    "validation_summary": WEEK6_DIR / "week6_validation_summary.csv"
}

print("Week 5 and Week 6 paths defined.")
print(f"\nWeek 5 files: {len(WEEK5_FILES)}")
print(f"Week 6 files: {len(WEEK6_FILES)}")

Week 5 and Week 6 paths defined.

Week 5 files: 12
Week 6 files: 28


In [23]:
# ============================================
# INPUT FILE EXISTENCE VALIDATION
# ============================================

all_input_files = {
    "O*NET / Week 4": {
        "onet_abilities": ONET_ABILITIES,
        "onet_cip_integrated": ONET_CIP_INTEGRATED,
        "onet_education": ONET_EDUCATION,
        "onet_education_v2": ONET_EDUCATION_V2,
        "onet_integrated": ONET_INTEGRATED,
        "onet_job_zone": ONET_JOB_ZONE,
        "onet_knowledge": ONET_KNOWLEDGE,
        "onet_skills": ONET_SKILLS,
        "onet_tasks": ONET_TASKS,
        "onet_tasks_v2": ONET_TASKS_V2,
        "onet_training": ONET_TRAINING,
        "validated_mapping": VALIDATED_MAPPING,
        "week4_summary": WEEK4_SUMMARY
    },
    "Week 5": WEEK5_FILES,
    "Week 6": WEEK6_FILES
}

validation_rows = []

for group_name, files_dict in all_input_files.items():
    for file_name, file_path in files_dict.items():
        validation_rows.append({
            "group": group_name,
            "file": file_name,
            "path": str(file_path),
            "exists": file_path.exists()
        })

file_validation_df = pd.DataFrame(validation_rows)

display(file_validation_df)

print("\n============================================")
print("FILE VALIDATION SUMMARY")
print("============================================")

print("Total expected files:", len(file_validation_df))
print("Files found:", file_validation_df["exists"].sum())
print("Files missing:", (~file_validation_df["exists"]).sum())

if file_validation_df["exists"].all():
    print("\nSUCCESS: All expected Week 4–6 input files were found.")
else:
    print("\nWARNING: Some expected files are missing.")
    
    display(
        file_validation_df[
            ~file_validation_df["exists"]
        ]
    )

,group,file,path,exists
0,O*NET / Week 4,onet_abilities,C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_abilities_profile.csv,True
1,O*NET / Week 4,onet_cip_integrated,C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_cip_integrated_occupation_profile.csv,True
2,O*NET / Week 4,onet_education,C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_education_profile.csv,True
3,O*NET / Week 4,onet_education_v2,C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_education_profile_v2.csv,True
4,O*NET / Week 4,onet_integrated,C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_integrated_occupation_profile.csv,True
5,O*NET / Week 4,onet_job_zone,C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_job_zone_profile.csv,True
6,O*NET / Week 4,onet_knowledge,C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_knowledge_profile.csv,True
7,O*NET / Week 4,onet_skills,C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_skills_profile.csv,True
8,O*NET / Week 4,onet_tasks,C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_tasks_profile.csv,True
9,O*NET / Week 4,onet_tasks_v2,C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_tasks_profile_v2.csv,True



FILE VALIDATION SUMMARY
Total expected files: 53
Files found: 53
Files missing: 0

SUCCESS: All expected Week 4–6 input files were found.


In [25]:
# ============================================
# LOAD CORE WEEK 5–6 DATASETS
# ============================================

candidate_features = pd.read_csv(
    WEEK5_FILES["candidate_features"]
)

candidate_onet_skill_matrix = pd.read_csv(
    WEEK5_FILES["candidate_onet_skill_matrix"]
)

candidate_occupation_cip_mapping = pd.read_csv(
    WEEK5_FILES["candidate_occupation_cip_mapping"]
)

week5_top5_cip = pd.read_csv(
    WEEK5_FILES["final_top5_cip"]
)

final_hybrid = pd.read_csv(
    WEEK6_FILES["final_hybrid_recommendations"]
)

final_hybrid_top1 = pd.read_csv(
    WEEK6_FILES["final_hybrid_top1"]
)

final_hybrid_top5 = pd.read_csv(
    WEEK6_FILES["final_hybrid_top5"]
)

hybrid_scores = pd.read_csv(
    WEEK6_FILES["hybrid_recommendation_scores"]
)

labour_demand = pd.read_csv(
    WEEK6_FILES["canadian_labour_demand"]
)

education_alignment = pd.read_csv(
    WEEK6_FILES["education_alignment"]
)

print("Core Week 5–6 datasets loaded successfully.")

Core Week 5–6 datasets loaded successfully.


In [27]:
# ============================================
# LOAD O*NET DATA
# ============================================

onet_skills_profile = pd.read_csv(
    ONET_SKILLS
)

onet_knowledge_profile = pd.read_csv(
    ONET_KNOWLEDGE
)

onet_tasks_profile = pd.read_csv(
    ONET_TASKS
)

onet_education_profile = pd.read_csv(
    ONET_EDUCATION
)

onet_integrated_profile = pd.read_csv(
    ONET_INTEGRATED
)

onet_cip_integrated_profile = pd.read_csv(
    ONET_CIP_INTEGRATED
)

validated_occupation_mapping = pd.read_csv(
    VALIDATED_MAPPING
)

print("Core O*NET/Week 4 datasets loaded successfully.")

Core O*NET/Week 4 datasets loaded successfully.


In [29]:
# ============================================
# INPUT DATASET AUDIT
# ============================================

datasets_to_audit = {
    "candidate_features": candidate_features,
    "candidate_onet_skill_matrix": candidate_onet_skill_matrix,
    "candidate_occupation_cip_mapping": candidate_occupation_cip_mapping,
    "week5_top5_cip": week5_top5_cip,
    "final_hybrid": final_hybrid,
    "final_hybrid_top1": final_hybrid_top1,
    "final_hybrid_top5": final_hybrid_top5,
    "hybrid_scores": hybrid_scores,
    "labour_demand": labour_demand,
    "education_alignment": education_alignment,
    "onet_skills_profile": onet_skills_profile,
    "onet_knowledge_profile": onet_knowledge_profile,
    "onet_tasks_profile": onet_tasks_profile,
    "onet_education_profile": onet_education_profile,
    "onet_integrated_profile": onet_integrated_profile,
    "onet_cip_integrated_profile": onet_cip_integrated_profile,
    "validated_occupation_mapping": validated_occupation_mapping
}

audit_rows = []

for name, df in datasets_to_audit.items():
    audit_rows.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

dataset_audit_df = pd.DataFrame(audit_rows)

display(dataset_audit_df)

,dataset,rows,columns,missing_cells,duplicate_rows
0,candidate_features,63,16,0,0
1,candidate_onet_skill_matrix,63,23,0,0
2,candidate_occupation_cip_mapping,315,8,0,0
3,week5_top5_cip,314,8,0,0
4,final_hybrid,945,31,0,0
5,final_hybrid_top1,63,11,0,0
6,final_hybrid_top5,315,11,0,0
7,hybrid_scores,945,10,0,0
8,labour_demand,15,13,0,0
9,education_alignment,945,4,0,0


In [31]:
# ============================================
# DISPLAY ACTUAL COLUMN NAMES
# ============================================

for name, df in datasets_to_audit.items():
    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)
    print("Shape:", df.shape)
    print("Columns:")
    
    for col in df.columns:
        print(" -", col)


CANDIDATE_FEATURES
Shape: (63, 16)
Columns:
 - candidate_id
 - cv_id
 - word_count
 - paragraph_count
 - clean_text
 - technical_skills
 - technologies
 - databases
 - software_tools
 - professional_skills
 - certifications
 - degree_level
 - field_of_study
 - raw_skills
 - raw_skill_count
 - onet_skill_mapping

CANDIDATE_ONET_SKILL_MATRIX
Shape: (63, 23)
Columns:
 - candidate_id
 - Active Learning
 - Active Learning_evidence_count
 - Active Listening
 - Active Listening_evidence_count
 - Critical Thinking
 - Critical Thinking_evidence_count
 - Learning Strategies
 - Learning Strategies_evidence_count
 - Mathematics
 - Mathematics_evidence_count
 - Monitoring
 - Monitoring_evidence_count
 - Reading Comprehension
 - Reading Comprehension_evidence_count
 - Science
 - Science_evidence_count
 - Speaking
 - Speaking_evidence_count
 - Writing
 - Writing_evidence_count
 - onet_skill_match_count
 - total_onet_evidence_count

CANDIDATE_OCCUPATION_CIP_MAPPING
Shape: (315, 8)
Columns:
 - candida

In [33]:
# ============================================
# CANDIDATE O*NET SKILL EVIDENCE
# ============================================

onet_candidate_skill_columns = [
    col for col in candidate_onet_skill_matrix.columns
    if not col.endswith("_evidence_count")
    and col != "candidate_id"
    and col not in ["onet_skill_match_count", "total_onet_evidence_count"]
]

print("Candidate O*NET skill columns:")
print(onet_candidate_skill_columns)

print("\nNumber of O*NET skill categories:",
      len(onet_candidate_skill_columns))

display(
    candidate_onet_skill_matrix[
        ["candidate_id"] +
        onet_candidate_skill_columns +
        ["onet_skill_match_count", "total_onet_evidence_count"]
    ].head(10)
)

Candidate O*NET skill columns:
['Active Learning', 'Active Listening', 'Critical Thinking', 'Learning Strategies', 'Mathematics', 'Monitoring', 'Reading Comprehension', 'Science', 'Speaking', 'Writing']

Number of O*NET skill categories: 10


,candidate_id,Active Learning,Active Listening,Critical Thinking,Learning Strategies,Mathematics,Monitoring,Reading Comprehension,Science,Speaking,Writing,onet_skill_match_count,total_onet_evidence_count
0,Candidate_001,1,0,1,1,1,0,1,1,0,1,7,19
1,Candidate_002,1,0,1,1,0,0,1,0,0,1,5,12
2,Candidate_003,1,1,1,1,0,1,1,0,1,1,8,21
3,Candidate_004,1,0,1,1,1,1,1,1,0,1,8,19
4,Candidate_005,1,0,1,1,1,0,1,0,0,0,5,18
5,Candidate_006,1,1,1,1,1,0,1,1,1,1,9,24
6,Candidate_007,1,1,1,1,1,1,1,1,1,1,10,30
7,Candidate_008,1,0,1,1,1,1,1,1,0,0,7,14
8,Candidate_009,0,0,0,0,0,0,1,0,0,1,2,6
9,Candidate_010,1,1,1,1,1,1,1,0,1,1,9,35


In [35]:
# ============================================
# CONVERT CANDIDATE O*NET MATRIX TO LONG FORMAT
# ============================================

candidate_skill_long = candidate_onet_skill_matrix.melt(
    id_vars=["candidate_id"],
    value_vars=onet_candidate_skill_columns,
    var_name="skill",
    value_name="candidate_evidence"
)

candidate_skill_long["candidate_evidence"] = pd.to_numeric(
    candidate_skill_long["candidate_evidence"],
    errors="coerce"
).fillna(0)

candidate_skill_long["skill_normalized"] = (
    candidate_skill_long["skill"]
    .str.lower()
    .str.strip()
)

# Keep only skills with actual evidence
candidate_skill_long = candidate_skill_long[
    candidate_skill_long["candidate_evidence"] > 0
].copy()

print("Candidate-skill evidence table shape:",
      candidate_skill_long.shape)

display(candidate_skill_long.head(20))

Candidate-skill evidence table shape: (456, 4)


,candidate_id,skill,candidate_evidence,skill_normalized
0,Candidate_001,Active Learning,1,active learning
1,Candidate_002,Active Learning,1,active learning
2,Candidate_003,Active Learning,1,active learning
3,Candidate_004,Active Learning,1,active learning
4,Candidate_005,Active Learning,1,active learning
5,Candidate_006,Active Learning,1,active learning
6,Candidate_007,Active Learning,1,active learning
7,Candidate_008,Active Learning,1,active learning
9,Candidate_010,Active Learning,1,active learning
10,Candidate_011,Active Learning,1,active learning


In [37]:
# ============================================
# CANDIDATE O*NET COVERAGE
# ============================================

candidate_onet_coverage = (
    candidate_skill_long
    .groupby("candidate_id")
    .agg(
        demonstrated_onet_skill_count=("skill", "nunique"),
        total_onet_evidence=("candidate_evidence", "sum")
    )
    .reset_index()
)

print(
    "Candidates with at least one O*NET-mapped skill:",
    candidate_onet_coverage["candidate_id"].nunique()
)

print(
    "Total candidates:",
    candidate_features["candidate_id"].nunique()
)

display(
    candidate_onet_coverage.describe()
)

Candidates with at least one O*NET-mapped skill: 63
Total candidates: 63


,demonstrated_onet_skill_count,total_onet_evidence
count,63.000000,63.000000
mean,7.238095,7.238095
std,1.729388,1.729388
min,2.000000,2.000000
25%,6.000000,6.000000
50%,7.000000,7.000000
75%,8.000000,8.000000
max,10.000000,10.000000


In [39]:
# ============================================
# O*NET OCCUPATION SKILL PROFILE
# ============================================

onet_required_skills = onet_skills_profile.copy()

onet_required_skills["skill_normalized"] = (
    onet_required_skills["skill"]
    .astype(str)
    .str.lower()
    .str.strip()
)

onet_required_skills["importance"] = pd.to_numeric(
    onet_required_skills["importance"],
    errors="coerce"
)

onet_required_skills["level"] = pd.to_numeric(
    onet_required_skills["level"],
    errors="coerce"
)

print("O*NET occupation-skill records:",
      len(onet_required_skills))

print(
    "Unique occupations:",
    onet_required_skills["selected_occupation"].nunique()
)

print(
    "Unique O*NET skills:",
    onet_required_skills["skill"].nunique()
)

display(
    onet_required_skills[
        [
            "selected_occupation",
            "skill",
            "importance",
            "level",
            "onet_title"
        ]
    ].head(20)
)

O*NET occupation-skill records: 200
Unique occupations: 15
Unique O*NET skills: 10


,selected_occupation,skill,importance,level,onet_title
0,Restaurant Manager,Reading Comprehension,3.75,3.88,Food Service Managers
1,Restaurant Manager,Active Listening,3.88,3.50,Food Service Managers
2,Restaurant Manager,Writing,3.00,3.12,Food Service Managers
3,Restaurant Manager,Speaking,3.88,4.00,Food Service Managers
4,Restaurant Manager,Mathematics,2.88,2.88,Food Service Managers
5,Restaurant Manager,Science,1.62,0.62,Food Service Managers
6,Restaurant Manager,Critical Thinking,3.62,3.75,Food Service Managers
7,Restaurant Manager,Active Learning,3.12,3.75,Food Service Managers
8,Restaurant Manager,Learning Strategies,3.12,3.12,Food Service Managers
9,Restaurant Manager,Monitoring,3.88,3.88,Food Service Managers


In [41]:
# ============================================
# SKILL OVERLAP VALIDATION
# ============================================

candidate_skill_names = set(
    candidate_skill_long["skill_normalized"].dropna()
)

occupation_skill_names = set(
    onet_required_skills["skill_normalized"].dropna()
)

overlapping_skills = sorted(
    candidate_skill_names.intersection(
        occupation_skill_names
    )
)

print("Candidate O*NET skills:", len(candidate_skill_names))
print("Occupation O*NET skills:", len(occupation_skill_names))
print("Overlapping skills:", len(overlapping_skills))

print("\nOverlapping skills:")
print(overlapping_skills)

Candidate O*NET skills: 10
Occupation O*NET skills: 10
Overlapping skills: 10

Overlapping skills:
['active learning', 'active listening', 'critical thinking', 'learning strategies', 'mathematics', 'monitoring', 'reading comprehension', 'science', 'speaking', 'writing']


In [43]:
# ============================================
# BUILD CANDIDATE × OCCUPATION SKILL GAPS
# ============================================

# All candidate-occupation combinations from Week 6
candidate_occupations = final_hybrid[
    ["candidate_id", "selected_occupation"]
].drop_duplicates()

print(
    "Candidate-occupation combinations:",
    len(candidate_occupations)
)

# Add every required O*NET skill to every candidate's occupation
skill_gap_base = candidate_occupations.merge(
    onet_required_skills[
        [
            "selected_occupation",
            "onet_soc_code",
            "skill",
            "skill_normalized",
            "importance",
            "level",
            "onet_title"
        ]
    ],
    on="selected_occupation",
    how="left"
)

print("Skill-gap base shape:", skill_gap_base.shape)

display(skill_gap_base.head(20))

Candidate-occupation combinations: 945
Skill-gap base shape: (12600, 8)


,candidate_id,selected_occupation,onet_soc_code,skill,skill_normalized,importance,level,onet_title
0,Candidate_001,Software Developer,15-1252.00,Reading Comprehension,reading comprehension,3.50,4.25,Software Developers
1,Candidate_001,Software Developer,15-1252.00,Active Listening,active listening,3.38,3.88,Software Developers
2,Candidate_001,Software Developer,15-1252.00,Writing,writing,3.25,3.62,Software Developers
3,Candidate_001,Software Developer,15-1252.00,Speaking,speaking,3.12,3.62,Software Developers
4,Candidate_001,Software Developer,15-1252.00,Mathematics,mathematics,2.75,3.25,Software Developers
5,Candidate_001,Software Developer,15-1252.00,Science,science,2.12,1.88,Software Developers
6,Candidate_001,Software Developer,15-1252.00,Critical Thinking,critical thinking,3.88,4.12,Software Developers
7,Candidate_001,Software Developer,15-1252.00,Active Learning,active learning,3.50,3.62,Software Developers
8,Candidate_001,Software Developer,15-1252.00,Learning Strategies,learning strategies,2.62,3.12,Software Developers
9,Candidate_001,Software Developer,15-1252.00,Monitoring,monitoring,3.00,3.50,Software Developers


In [45]:
# ============================================
# DIAGNOSE O*NET SKILL DUPLICATES
# ============================================

onet_skill_duplicates = (
    onet_skills_profile
    .groupby(
        ["selected_occupation", "skill"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "record_count"})
)

onet_skill_duplicates = onet_skill_duplicates[
    onet_skill_duplicates["record_count"] > 1
].sort_values(
    ["selected_occupation", "skill"]
)

print(
    "Number of duplicated occupation-skill combinations:",
    len(onet_skill_duplicates)
)

display(onet_skill_duplicates.head(50))

Number of duplicated occupation-skill combinations: 30


,selected_occupation,skill,record_count
20,Continuing Care Assistant,Active Learning,2
21,Continuing Care Assistant,Active Listening,2
22,Continuing Care Assistant,Critical Thinking,2
23,Continuing Care Assistant,Learning Strategies,2
24,Continuing Care Assistant,Mathematics,2
25,Continuing Care Assistant,Monitoring,2
26,Continuing Care Assistant,Reading Comprehension,2
27,Continuing Care Assistant,Science,2
28,Continuing Care Assistant,Speaking,2
29,Continuing Care Assistant,Writing,2


In [47]:
# ============================================
# CHECK NORMALIZED SKILL DUPLICATES
# ============================================

onet_required_check = onet_skills_profile.copy()

onet_required_check["skill_normalized"] = (
    onet_required_check["skill"]
    .astype(str)
    .str.lower()
    .str.strip()
)

normalized_duplicates = (
    onet_required_check
    .groupby(
        ["selected_occupation", "skill_normalized"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "record_count"})
)

normalized_duplicates = normalized_duplicates[
    normalized_duplicates["record_count"] > 1
].sort_values(
    ["selected_occupation", "skill_normalized"]
)

print(
    "Duplicated normalized occupation-skill combinations:",
    len(normalized_duplicates)
)

display(normalized_duplicates.head(50))

Duplicated normalized occupation-skill combinations: 30


,selected_occupation,skill_normalized,record_count
20,Continuing Care Assistant,active learning,2
21,Continuing Care Assistant,active listening,2
22,Continuing Care Assistant,critical thinking,2
23,Continuing Care Assistant,learning strategies,2
24,Continuing Care Assistant,mathematics,2
25,Continuing Care Assistant,monitoring,2
26,Continuing Care Assistant,reading comprehension,2
27,Continuing Care Assistant,science,2
28,Continuing Care Assistant,speaking,2
29,Continuing Care Assistant,writing,2


In [49]:
# ============================================
# DEDUPLICATE O*NET OCCUPATION-SKILL PROFILE
# ============================================

onet_required_skills_clean = (
    onet_skills_profile
    .copy()
)

onet_required_skills_clean["skill_normalized"] = (
    onet_required_skills_clean["skill"]
    .astype(str)
    .str.lower()
    .str.strip()
)

onet_required_skills_clean["importance"] = pd.to_numeric(
    onet_required_skills_clean["importance"],
    errors="coerce"
)

onet_required_skills_clean["level"] = pd.to_numeric(
    onet_required_skills_clean["level"],
    errors="coerce"
)

onet_required_skills_clean = (
    onet_required_skills_clean
    .groupby(
        [
            "selected_occupation",
            "skill_normalized"
        ],
        as_index=False
    )
    .agg(
        onet_soc_code=("onet_soc_code", "first"),
        skill=("skill", "first"),
        importance=("importance", "max"),
        level=("level", "max"),
        onet_title=("onet_title", "first")
    )
)

print(
    "Original O*NET skill records:",
    len(onet_skills_profile)
)

print(
    "Clean occupation-skill records:",
    len(onet_required_skills_clean)
)

print(
    "Unique occupations:",
    onet_required_skills_clean[
        "selected_occupation"
    ].nunique()
)

print(
    "Unique occupation-skill combinations:",
    onet_required_skills_clean[
        ["selected_occupation", "skill_normalized"]
    ].drop_duplicates().shape[0]
)

display(
    onet_required_skills_clean.head(20)
)

Original O*NET skill records: 200
Clean occupation-skill records: 150
Unique occupations: 15
Unique occupation-skill combinations: 150


,selected_occupation,skill_normalized,onet_soc_code,skill,importance,level,onet_title
0,Administrative Assistant,active learning,43-6014.00,Active Learning,2.88,3.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
1,Administrative Assistant,active listening,43-6014.00,Active Listening,4.00,3.75,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
2,Administrative Assistant,critical thinking,43-6014.00,Critical Thinking,3.00,3.62,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
3,Administrative Assistant,learning strategies,43-6014.00,Learning Strategies,2.12,1.88,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
4,Administrative Assistant,mathematics,43-6014.00,Mathematics,2.00,1.62,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
5,Administrative Assistant,monitoring,43-6014.00,Monitoring,3.12,3.25,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
6,Administrative Assistant,reading comprehension,43-6014.00,Reading Comprehension,3.88,3.88,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
7,Administrative Assistant,science,43-6014.00,Science,1.00,0.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
8,Administrative Assistant,speaking,43-6014.00,Speaking,4.00,3.62,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
9,Administrative Assistant,writing,43-6014.00,Writing,3.75,3.50,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"


In [51]:
# ============================================
# VALIDATE CANDIDATE SKILL EVIDENCE
# ============================================

candidate_skill_check = (
    candidate_skill_long
    .groupby(
        ["candidate_id", "skill_normalized"],
        as_index=False
    )
    .agg(
        candidate_evidence=("candidate_evidence", "max")
    )
)

print(
    "Original candidate skill evidence rows:",
    len(candidate_skill_long)
)

print(
    "Unique candidate-skill combinations:",
    len(candidate_skill_check)
)

candidate_duplicate_check = (
    candidate_skill_long
    .groupby(
        ["candidate_id", "skill_normalized"]
    )
    .size()
    .reset_index(name="record_count")
)

candidate_duplicate_check = candidate_duplicate_check[
    candidate_duplicate_check["record_count"] > 1
]

print(
    "Duplicated candidate-skill combinations:",
    len(candidate_duplicate_check)
)

display(candidate_duplicate_check.head(20))

Original candidate skill evidence rows: 456
Unique candidate-skill combinations: 456
Duplicated candidate-skill combinations: 0


,candidate_id,skill_normalized,record_count


In [53]:
# ============================================
# REBUILD SKILL-GAP BASE
# ============================================

candidate_occupations = (
    final_hybrid[
        ["candidate_id", "selected_occupation"]
    ]
    .drop_duplicates()
)

print(
    "Candidate-occupation combinations:",
    len(candidate_occupations)
)

skill_gap_base = candidate_occupations.merge(
    onet_required_skills_clean[
        [
            "selected_occupation",
            "onet_soc_code",
            "skill",
            "skill_normalized",
            "importance",
            "level",
            "onet_title"
        ]
    ],
    on="selected_occupation",
    how="left"
)

skill_gap_base = skill_gap_base.merge(
    candidate_skill_check[
        [
            "candidate_id",
            "skill_normalized",
            "candidate_evidence"
        ]
    ],
    on=[
        "candidate_id",
        "skill_normalized"
    ],
    how="left"
)

skill_gap_base["candidate_evidence"] = (
    skill_gap_base["candidate_evidence"]
    .fillna(0)
)

skill_gap_base["skill_present"] = (
    skill_gap_base["candidate_evidence"] > 0
).astype(int)

skill_gap_base["gap_status"] = np.where(
    skill_gap_base["skill_present"] == 1,
    "Existing",
    "Missing"
)

print(
    "\nSkill-gap base shape:",
    skill_gap_base.shape
)

display(
    skill_gap_base[
        [
            "candidate_id",
            "selected_occupation",
            "skill",
            "importance",
            "level",
            "candidate_evidence",
            "skill_present",
            "gap_status"
        ]
    ].head(30)
)

Candidate-occupation combinations: 945

Skill-gap base shape: (9450, 11)


,candidate_id,selected_occupation,skill,importance,level,candidate_evidence,skill_present,gap_status
0,Candidate_001,Software Developer,Active Learning,3.50,3.62,1.0,1,Existing
1,Candidate_001,Software Developer,Active Listening,3.38,3.88,0.0,0,Missing
2,Candidate_001,Software Developer,Critical Thinking,3.88,4.12,1.0,1,Existing
3,Candidate_001,Software Developer,Learning Strategies,2.62,3.12,1.0,1,Existing
4,Candidate_001,Software Developer,Mathematics,2.75,3.25,1.0,1,Existing
5,Candidate_001,Software Developer,Monitoring,3.00,3.50,0.0,0,Missing
6,Candidate_001,Software Developer,Reading Comprehension,3.50,4.25,1.0,1,Existing
7,Candidate_001,Software Developer,Science,2.12,1.88,1.0,1,Existing
8,Candidate_001,Software Developer,Speaking,3.12,3.62,0.0,0,Missing
9,Candidate_001,Software Developer,Writing,3.25,3.62,1.0,1,Existing


In [55]:
# ============================================================
# CELL 26 — Skill-Gap Priority Calculation
# ============================================================

# Normalize O*NET importance and required skill level
# O*NET importance scale is typically 0–5
# O*NET level scale is also treated on a 0–5 range here

skill_gap_base["importance_normalized"] = (
    skill_gap_base["importance"] / 5
).clip(0, 1)

skill_gap_base["level_normalized"] = (
    skill_gap_base["level"] / 5
).clip(0, 1)


# Calculate O*NET-based priority for missing skills
# Higher importance + higher required level = higher priority

skill_gap_base["onet_gap_priority"] = (
    0.5 * skill_gap_base["importance_normalized"]
    + 0.5 * skill_gap_base["level_normalized"]
)


# A skill that already exists does not represent a current gap
skill_gap_base["skill_gap_priority"] = np.where(
    skill_gap_base["gap_status"] == "Missing",
    skill_gap_base["onet_gap_priority"],
    0
)


print("Skill-gap priority calculation completed.")

display(
    skill_gap_base[
        [
            "candidate_id",
            "selected_occupation",
            "skill",
            "importance",
            "level",
            "candidate_evidence",
            "gap_status",
            "importance_normalized",
            "level_normalized",
            "skill_gap_priority"
        ]
    ].head(30)
)

Skill-gap priority calculation completed.


,candidate_id,selected_occupation,skill,importance,level,candidate_evidence,gap_status,importance_normalized,level_normalized,skill_gap_priority
0,Candidate_001,Software Developer,Active Learning,3.50,3.62,1.0,Existing,0.700,0.724,0.000
1,Candidate_001,Software Developer,Active Listening,3.38,3.88,0.0,Missing,0.676,0.776,0.726
2,Candidate_001,Software Developer,Critical Thinking,3.88,4.12,1.0,Existing,0.776,0.824,0.000
3,Candidate_001,Software Developer,Learning Strategies,2.62,3.12,1.0,Existing,0.524,0.624,0.000
4,Candidate_001,Software Developer,Mathematics,2.75,3.25,1.0,Existing,0.550,0.650,0.000
5,Candidate_001,Software Developer,Monitoring,3.00,3.50,0.0,Missing,0.600,0.700,0.650
6,Candidate_001,Software Developer,Reading Comprehension,3.50,4.25,1.0,Existing,0.700,0.850,0.000
7,Candidate_001,Software Developer,Science,2.12,1.88,1.0,Existing,0.424,0.376,0.000
8,Candidate_001,Software Developer,Speaking,3.12,3.62,0.0,Missing,0.624,0.724,0.674
9,Candidate_001,Software Developer,Writing,3.25,3.62,1.0,Existing,0.650,0.724,0.000


In [57]:
# ============================================================
# CELL 27 — Candidate-Occupation Skill-Gap Summary
# ============================================================

skill_gap_summary = (
    skill_gap_base
    .groupby(
        ["candidate_id", "selected_occupation"],
        as_index=False
    )
    .agg(
        required_skill_count=("skill_normalized", "nunique"),
        existing_skill_count=("skill_present", "sum"),
        missing_skill_count=("skill_present", lambda x: (x == 0).sum()),
        average_gap_priority=("skill_gap_priority", "mean"),
        maximum_gap_priority=("skill_gap_priority", "max")
    )
)

# Calculate skill-match percentage
skill_gap_summary["skill_match_percentage"] = (
    skill_gap_summary["existing_skill_count"]
    / skill_gap_summary["required_skill_count"]
    * 100
)

# Ensure percentage remains within valid range
skill_gap_summary["skill_match_percentage"] = (
    skill_gap_summary["skill_match_percentage"]
    .clip(0, 100)
    .round(2)
)

# Round priority measures
skill_gap_summary["average_gap_priority"] = (
    skill_gap_summary["average_gap_priority"].round(3)
)

skill_gap_summary["maximum_gap_priority"] = (
    skill_gap_summary["maximum_gap_priority"].round(3)
)


print(
    "Skill-gap summary shape:",
    skill_gap_summary.shape
)

display(
    skill_gap_summary.head(20)
)

Skill-gap summary shape: (945, 8)


,candidate_id,selected_occupation,required_skill_count,existing_skill_count,missing_skill_count,average_gap_priority,maximum_gap_priority,skill_match_percentage
0,Candidate_001,Administrative Assistant,10,7,3,0.217,0.775,70.0
1,Candidate_001,Bookkeeper,10,7,3,0.191,0.700,70.0
2,Candidate_001,Continuing Care Assistant,10,7,3,0.190,0.662,70.0
3,Candidate_001,Delivery Driver,10,7,3,0.179,0.600,70.0
4,Candidate_001,"Driver, Truck",10,7,3,0.179,0.612,70.0
5,Candidate_001,Food Service Supervisor,10,7,3,0.225,0.788,70.0
6,Candidate_001,Information Technology (IT) Analyst,10,7,3,0.237,0.812,70.0
7,Candidate_001,Inside Sales Representative,10,7,3,0.205,0.750,70.0
8,Candidate_001,Licensed Practical Nurse (L.P.N.),10,7,3,0.236,0.788,70.0
9,Candidate_001,Office Administrator,10,7,3,0.240,0.800,70.0


In [61]:
# ============================================================
# CELL 28 — Skill-Gap Validation
# ============================================================

print("Running skill-gap validation...\n")

# 1. Existing skills cannot exceed required skills
existing_exceeds_required = (
    skill_gap_summary["existing_skill_count"]
    > skill_gap_summary["required_skill_count"]
).sum()

print(
    "Existing > Required:",
    existing_exceeds_required
)


# 2. Skill-match percentage cannot exceed 100%
match_over_100 = (
    skill_gap_summary["skill_match_percentage"] > 100
).sum()

print(
    "Skill-match percentage > 100:",
    match_over_100
)


# 3. Existing + Missing must equal Required
skill_count_mismatch = (
    skill_gap_summary["existing_skill_count"]
    + skill_gap_summary["missing_skill_count"]
    != skill_gap_summary["required_skill_count"]
).sum()

print(
    "Existing + Missing != Required:",
    skill_count_mismatch
)


# 4. Verify expected candidate-occupation combinations
expected_combinations = 63 * 15
actual_combinations = len(skill_gap_summary)

print(
    "Expected candidate-occupation combinations:",
    expected_combinations
)

print(
    "Actual candidate-occupation combinations:",
    actual_combinations
)


# 5. Verify required skills per occupation
invalid_required_skill_counts = (
    skill_gap_summary["required_skill_count"] != 10
).sum()

print(
    "Candidate-occupation rows with required skill count != 10:",
    invalid_required_skill_counts
)


# ============================================================
# FINAL VALIDATION
# ============================================================

if (
    existing_exceeds_required == 0
    and match_over_100 == 0
    and skill_count_mismatch == 0
    and actual_combinations == expected_combinations
    and invalid_required_skill_counts == 0
):
    print("\n✅ PASS — Skill-gap validation successful.")
else:
    print("\n❌ FAIL — Review the validation results above.")

Running skill-gap validation...

Existing > Required: 0
Skill-match percentage > 100: 0
Existing + Missing != Required: 0
Expected candidate-occupation combinations: 945
Actual candidate-occupation combinations: 945
Candidate-occupation rows with required skill count != 10: 0

✅ PASS — Skill-gap validation successful.


In [63]:
# ============================================================
# CELL 29 — Detailed Missing-Skill Priority Table
# ============================================================

missing_skill_priorities = (
    skill_gap_base[
        skill_gap_base["gap_status"] == "Missing"
    ][
        [
            "candidate_id",
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "skill",
            "importance",
            "level",
            "candidate_evidence",
            "skill_gap_priority"
        ]
    ]
    .copy()
)

# Rank missing skills within each candidate-occupation pair
missing_skill_priorities["skill_priority_rank"] = (
    missing_skill_priorities
    .groupby(
        ["candidate_id", "selected_occupation"]
    )["skill_gap_priority"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)

# Sort so the highest-priority missing skills appear first
missing_skill_priorities = (
    missing_skill_priorities
    .sort_values(
        [
            "candidate_id",
            "selected_occupation",
            "skill_gap_priority"
        ],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

print(
    "Missing-skill records:",
    len(missing_skill_priorities)
)

print(
    "Unique candidates:",
    missing_skill_priorities["candidate_id"].nunique()
)

print(
    "Unique occupations:",
    missing_skill_priorities["selected_occupation"].nunique()
)

display(
    missing_skill_priorities.head(30)
)

Missing-skill records: 2610
Unique candidates: 56
Unique occupations: 15


,candidate_id,selected_occupation,onet_soc_code,onet_title,skill,importance,level,candidate_evidence,skill_gap_priority,skill_priority_rank
0,Candidate_001,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Active Listening,4.00,3.75,0.0,0.775,1
1,Candidate_001,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Speaking,4.00,3.62,0.0,0.762,2
2,Candidate_001,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Monitoring,3.12,3.25,0.0,0.637,3
3,Candidate_001,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",Active Listening,3.25,3.75,0.0,0.700,1
4,Candidate_001,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",Speaking,3.12,3.12,0.0,0.624,2
5,Candidate_001,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",Monitoring,3.00,2.88,0.0,0.588,3
6,Candidate_001,Continuing Care Assistant,31-1131.00,Nursing Assistants,Active Listening,3.62,3.00,0.0,0.662,1
7,Candidate_001,Continuing Care Assistant,31-1131.00,Nursing Assistants,Monitoring,3.25,3.00,0.0,0.625,2
8,Candidate_001,Continuing Care Assistant,31-1131.00,Nursing Assistants,Speaking,3.12,3.00,0.0,0.612,3
9,Candidate_001,Delivery Driver,53-3033.00,Light Truck Drivers,Active Listening,3.12,2.88,0.0,0.600,1


In [65]:
# ============================================================
# CELL 30 — Integrate Canadian Labour-Market Demand
# ============================================================

# Keep the occupation-level Canadian demand information
demand_for_skill_gap = (
    labour_demand[
        [
            "selected_occupation",
            "posting_count",
            "total_vacancies",
            "province_count",
            "demand_score",
            "demand_percentage"
        ]
    ]
    .drop_duplicates(
        subset=["selected_occupation"]
    )
    .copy()
)

print(
    "Unique occupations in labour-demand data:",
    demand_for_skill_gap["selected_occupation"].nunique()
)

# Merge Canadian demand into the missing-skill table
missing_skill_priorities = missing_skill_priorities.merge(
    demand_for_skill_gap,
    on="selected_occupation",
    how="left"
)

print(
    "Missing-skill table after demand integration:",
    missing_skill_priorities.shape
)

print(
    "Rows missing Canadian demand information:",
    missing_skill_priorities["demand_score"].isna().sum()
)

display(
    missing_skill_priorities[
        [
            "candidate_id",
            "selected_occupation",
            "skill",
            "importance",
            "level",
            "skill_gap_priority",
            "skill_priority_rank",
            "posting_count",
            "total_vacancies",
            "province_count",
            "demand_score",
            "demand_percentage"
        ]
    ].head(30)
)


Unique occupations in labour-demand data: 15
Missing-skill table after demand integration: (2610, 15)
Rows missing Canadian demand information: 0


,candidate_id,selected_occupation,skill,importance,level,skill_gap_priority,skill_priority_rank,posting_count,total_vacancies,province_count,demand_score,demand_percentage
0,Candidate_001,Administrative Assistant,Active Listening,4.00,3.75,0.775,1,812,890,11,0.438451,43.85
1,Candidate_001,Administrative Assistant,Speaking,4.00,3.62,0.762,2,812,890,11,0.438451,43.85
2,Candidate_001,Administrative Assistant,Monitoring,3.12,3.25,0.637,3,812,890,11,0.438451,43.85
3,Candidate_001,Bookkeeper,Active Listening,3.25,3.75,0.700,1,396,422,11,0.299226,29.92
4,Candidate_001,Bookkeeper,Speaking,3.12,3.12,0.624,2,396,422,11,0.299226,29.92
5,Candidate_001,Bookkeeper,Monitoring,3.00,2.88,0.588,3,396,422,11,0.299226,29.92
6,Candidate_001,Continuing Care Assistant,Active Listening,3.62,3.00,0.662,1,819,1172,9,0.439620,43.96
7,Candidate_001,Continuing Care Assistant,Monitoring,3.25,3.00,0.625,2,819,1172,9,0.439620,43.96
8,Candidate_001,Continuing Care Assistant,Speaking,3.12,3.00,0.612,3,819,1172,9,0.439620,43.96
9,Candidate_001,Delivery Driver,Active Listening,3.12,2.88,0.600,1,557,622,12,0.370531,37.05


In [69]:
# ============================================================
# CELL 31 — Combined Prescriptive Skill Priority
# ============================================================

# Normalize Canadian demand percentage to 0–1
missing_skill_priorities["demand_normalized"] = (
    missing_skill_priorities["demand_percentage"] / 100
).clip(0, 1)


# ------------------------------------------------------------
# Combined prescriptive priority
#
# 60% = O*NET skill-gap priority
# 40% = Canadian labour-market demand
# ------------------------------------------------------------

missing_skill_priorities["prescriptive_priority_score"] = (
    0.60 * missing_skill_priorities["skill_gap_priority"]
    + 0.40 * missing_skill_priorities["demand_normalized"]
)


# Convert to percentage for easier interpretation
missing_skill_priorities["prescriptive_priority_percentage"] = (
    missing_skill_priorities["prescriptive_priority_score"]
    * 100
).round(2)


# Round underlying score
missing_skill_priorities["prescriptive_priority_score"] = (
    missing_skill_priorities["prescriptive_priority_score"]
    .round(3)
)


# ------------------------------------------------------------
# Rank missing skills using the combined score
# ------------------------------------------------------------

missing_skill_priorities["prescriptive_rank"] = (
    missing_skill_priorities
    .groupby(
        ["candidate_id", "selected_occupation"]
    )["prescriptive_priority_score"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)


# Sort by candidate, occupation, and final priority
missing_skill_priorities = (
    missing_skill_priorities
    .sort_values(
        [
            "candidate_id",
            "selected_occupation",
            "prescriptive_priority_score"
        ],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)


print("Combined prescriptive priority calculated.")

display(
    missing_skill_priorities[
        [
            "candidate_id",
            "selected_occupation",
            "skill",
            "importance",
            "level",
            "skill_gap_priority",
            "demand_percentage",
            "prescriptive_priority_score",
            "prescriptive_priority_percentage",
            "prescriptive_rank"
        ]
    ].head(30)
)

Combined prescriptive priority calculated.


,candidate_id,selected_occupation,skill,importance,level,skill_gap_priority,demand_percentage,prescriptive_priority_score,prescriptive_priority_percentage,prescriptive_rank
0,Candidate_001,Administrative Assistant,Active Listening,4.00,3.75,0.775,43.85,0.640,64.04,1
1,Candidate_001,Administrative Assistant,Speaking,4.00,3.62,0.762,43.85,0.633,63.26,2
2,Candidate_001,Administrative Assistant,Monitoring,3.12,3.25,0.637,43.85,0.558,55.76,3
3,Candidate_001,Bookkeeper,Active Listening,3.25,3.75,0.700,29.92,0.540,53.97,1
4,Candidate_001,Bookkeeper,Speaking,3.12,3.12,0.624,29.92,0.494,49.41,2
5,Candidate_001,Bookkeeper,Monitoring,3.00,2.88,0.588,29.92,0.472,47.25,3
6,Candidate_001,Continuing Care Assistant,Active Listening,3.62,3.00,0.662,43.96,0.573,57.30,1
7,Candidate_001,Continuing Care Assistant,Monitoring,3.25,3.00,0.625,43.96,0.551,55.08,2
8,Candidate_001,Continuing Care Assistant,Speaking,3.12,3.00,0.612,43.96,0.543,54.30,3
9,Candidate_001,Delivery Driver,Active Listening,3.12,2.88,0.600,37.05,0.508,50.82,1


In [71]:
# ============================================================
# CELL 32 — Validate Prescriptive Priority
# ============================================================

print("Running prescriptive priority validation...\n")

# Check 1: Required columns exist
required_columns = [
    "candidate_id",
    "selected_occupation",
    "skill",
    "skill_gap_priority",
    "demand_percentage",
    "prescriptive_priority_score",
    "prescriptive_priority_percentage",
    "prescriptive_rank"
]

missing_columns = [
    col for col in required_columns
    if col not in missing_skill_priorities.columns
]

print("Missing required columns:", missing_columns)

# Check 2: Prescriptive score should be between 0 and 1
invalid_score_range = (
    (missing_skill_priorities["prescriptive_priority_score"] < 0)
    |
    (missing_skill_priorities["prescriptive_priority_score"] > 1)
).sum()

print("Prescriptive scores outside 0–1:", invalid_score_range)

# Check 3: Percentage should be between 0 and 100
invalid_percentage_range = (
    (missing_skill_priorities["prescriptive_priority_percentage"] < 0)
    |
    (missing_skill_priorities["prescriptive_priority_percentage"] > 100)
).sum()

print("Prescriptive percentages outside 0–100:", invalid_percentage_range)

# Check 4: No missing prescriptive scores
missing_scores = (
    missing_skill_priorities["prescriptive_priority_score"]
    .isna()
    .sum()
)

print("Missing prescriptive scores:", missing_scores)

# Check 5: No missing Canadian demand
missing_demand = (
    missing_skill_priorities["demand_percentage"]
    .isna()
    .sum()
)

print("Missing demand values:", missing_demand)

# Check 6: Rank should be positive
invalid_ranks = (
    missing_skill_priorities["prescriptive_rank"] < 1
).sum()

print("Invalid prescriptive ranks:", invalid_ranks)

# Check 7: Expected number of rows
expected_rows = 2610
actual_rows = len(missing_skill_priorities)

print("Expected missing-skill rows:", expected_rows)
print("Actual missing-skill rows:", actual_rows)

# Final validation
if (
    len(missing_columns) == 0
    and invalid_score_range == 0
    and invalid_percentage_range == 0
    and missing_scores == 0
    and missing_demand == 0
    and invalid_ranks == 0
    and actual_rows == expected_rows
):
    print("\n✅ PASS — Prescriptive priority validation successful.")
else:
    print("\n⚠️ REVIEW — One or more prescriptive validation checks failed.")

Running prescriptive priority validation...

Missing required columns: []
Prescriptive scores outside 0–1: 0
Prescriptive percentages outside 0–100: 0
Missing prescriptive scores: 0
Missing demand values: 0
Invalid prescriptive ranks: 0
Expected missing-skill rows: 2610
Actual missing-skill rows: 2610

✅ PASS — Prescriptive priority validation successful.


In [73]:
# ============================================================
# CELL 33 — Candidate-Level Prescriptive Recommendations
# ============================================================

# Select the top 3 missing skills for each candidate-occupation
candidate_prescriptive_recommendations = (
    missing_skill_priorities
    .sort_values(
        [
            "candidate_id",
            "selected_occupation",
            "prescriptive_priority_score"
        ],
        ascending=[True, True, False]
    )
    .groupby(
        ["candidate_id", "selected_occupation"],
        as_index=False
    )
    .head(3)
    .copy()
)

# Create priority order within each candidate-occupation
candidate_prescriptive_recommendations["skill_priority_rank"] = (
    candidate_prescriptive_recommendations
    .groupby(
        ["candidate_id", "selected_occupation"]
    )["prescriptive_priority_score"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)

# Keep the most useful prescriptive fields
candidate_prescriptive_recommendations = (
    candidate_prescriptive_recommendations[
        [
            "candidate_id",
            "selected_occupation",
            "skill",
            "importance",
            "level",
            "candidate_evidence",
            "skill_gap_priority",
            "demand_percentage",
            "prescriptive_priority_score",
            "prescriptive_priority_percentage",
            "skill_priority_rank"
        ]
    ]
    .sort_values(
        [
            "candidate_id",
            "selected_occupation",
            "skill_priority_rank"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Candidate-level prescriptive recommendations created."
)

print(
    "Shape:",
    candidate_prescriptive_recommendations.shape
)

print(
    "Unique candidate-occupation combinations:",
    candidate_prescriptive_recommendations[
        ["candidate_id", "selected_occupation"]
    ].drop_duplicates().shape[0]
)

display(
    candidate_prescriptive_recommendations.head(30)
)

Candidate-level prescriptive recommendations created.
Shape: (2070, 11)
Unique candidate-occupation combinations: 840


,candidate_id,selected_occupation,skill,importance,level,candidate_evidence,skill_gap_priority,demand_percentage,prescriptive_priority_score,prescriptive_priority_percentage,skill_priority_rank
0,Candidate_001,Administrative Assistant,Active Listening,4.00,3.75,0.0,0.775,43.85,0.640,64.04,1
1,Candidate_001,Administrative Assistant,Speaking,4.00,3.62,0.0,0.762,43.85,0.633,63.26,2
2,Candidate_001,Administrative Assistant,Monitoring,3.12,3.25,0.0,0.637,43.85,0.558,55.76,3
3,Candidate_001,Bookkeeper,Active Listening,3.25,3.75,0.0,0.700,29.92,0.540,53.97,1
4,Candidate_001,Bookkeeper,Speaking,3.12,3.12,0.0,0.624,29.92,0.494,49.41,2
5,Candidate_001,Bookkeeper,Monitoring,3.00,2.88,0.0,0.588,29.92,0.472,47.25,3
6,Candidate_001,Continuing Care Assistant,Active Listening,3.62,3.00,0.0,0.662,43.96,0.573,57.30,1
7,Candidate_001,Continuing Care Assistant,Monitoring,3.25,3.00,0.0,0.625,43.96,0.551,55.08,2
8,Candidate_001,Continuing Care Assistant,Speaking,3.12,3.00,0.0,0.612,43.96,0.543,54.30,3
9,Candidate_001,Delivery Driver,Active Listening,3.12,2.88,0.0,0.600,37.05,0.508,50.82,1


In [79]:
# ============================================================
# CELL 34 — Validate Candidate-Level Prescriptive Recommendations
# ============================================================

print("Running candidate-level prescriptive recommendation validation...\n")

# Check 1: Required columns
required_columns = [
    "candidate_id",
    "selected_occupation",
    "skill",
    "importance",
    "level",
    "candidate_evidence",
    "skill_gap_priority",
    "demand_percentage",
    "prescriptive_priority_score",
    "prescriptive_priority_percentage",
    "skill_priority_rank"
]

missing_columns = [
    col
    for col in required_columns
    if col not in candidate_prescriptive_recommendations.columns
]

print("Missing required columns:", missing_columns)

# Check 2: Maximum 3 recommendations per candidate-occupation
recommendation_counts = (
    candidate_prescriptive_recommendations
    .groupby(
        ["candidate_id", "selected_occupation"]
    )
    .size()
)

pairs_above_three = (
    recommendation_counts > 3
).sum()

print(
    "Candidate-occupation pairs with more than 3 skills:",
    pairs_above_three
)

# Check 3: Valid rank values
invalid_rank_values = (
    ~candidate_prescriptive_recommendations[
        "skill_priority_rank"
    ].isin([1, 2, 3])
).sum()

print(
    "Invalid skill priority ranks:",
    invalid_rank_values
)

# Check 4: No missing priority scores
missing_priority_scores = (
    candidate_prescriptive_recommendations[
        "prescriptive_priority_score"
    ].isna()
).sum()

print(
    "Missing prescriptive priority scores:",
    missing_priority_scores
)

# Check 5: All recommendations are missing skills
existing_skill_recommendations = (
    candidate_prescriptive_recommendations[
        "candidate_evidence"
    ] > 0
).sum()

print(
    "Recommendations with candidate evidence > 0:",
    existing_skill_recommendations
)

# Check 6: Verify ranking order within each candidate-occupation
ranking_errors = 0

for (
    candidate_id,
    occupation
), group in candidate_prescriptive_recommendations.groupby(
    ["candidate_id", "selected_occupation"]
):

    scores = group[
        "prescriptive_priority_score"
    ].tolist()

    if scores != sorted(scores, reverse=True):
        ranking_errors += 1

print(
    "Candidate-occupation groups with incorrect ranking order:",
    ranking_errors
)

# Check 7: Expected candidate-occupation pairs
expected_pairs = 840
actual_pairs = recommendation_counts.shape[0]

print(
    "Expected candidate-occupation pairs:",
    expected_pairs
)

print(
    "Actual candidate-occupation pairs:",
    actual_pairs
)

# Final validation
if (
    len(missing_columns) == 0
    and pairs_above_three == 0
    and invalid_rank_values == 0
    and missing_priority_scores == 0
    and existing_skill_recommendations == 0
    and ranking_errors == 0
    and actual_pairs == expected_pairs
):
    print(
        "\n✅ PASS — Candidate-level prescriptive "
        "recommendation validation successful."
    )
else:
    print(
        "\n⚠️ REVIEW — One or more validation checks failed."
    )

Running candidate-level prescriptive recommendation validation...

Missing required columns: []
Candidate-occupation pairs with more than 3 skills: 0
Invalid skill priority ranks: 0
Missing prescriptive priority scores: 0
Recommendations with candidate evidence > 0: 0
Candidate-occupation groups with incorrect ranking order: 0
Expected candidate-occupation pairs: 840
Actual candidate-occupation pairs: 840

✅ PASS — Candidate-level prescriptive recommendation validation successful.


In [81]:
# ============================================================
# CELL 35 — Integrate Education / CIP Pathway Information
# ============================================================

# Select education pathway information from the Week 5 mapping
education_pathways = (
    candidate_occupation_cip_mapping[
        [
            "candidate_id",
            "selected_occupation",
            "cip_program_count",
            "cip_pathway_available",
            "cip_status"
        ]
    ]
    .drop_duplicates(
        ["candidate_id", "selected_occupation"]
    )
    .copy()
)

# Merge education pathway information
candidate_prescriptive_recommendations = (
    candidate_prescriptive_recommendations
    .merge(
        education_pathways,
        on=[
            "candidate_id",
            "selected_occupation"
        ],
        how="left"
    )
)

print("Education pathway information integrated.")

print(
    "Shape:",
    candidate_prescriptive_recommendations.shape
)

print(
    "Rows missing CIP pathway information:",
    candidate_prescriptive_recommendations[
        [
            "cip_program_count",
            "cip_pathway_available",
            "cip_status"
        ]
    ]
    .isna()
    .any(axis=1)
    .sum()
)

display(
    candidate_prescriptive_recommendations[
        [
            "candidate_id",
            "selected_occupation",
            "skill",
            "prescriptive_priority_percentage",
            "skill_priority_rank",
            "cip_program_count",
            "cip_pathway_available",
            "cip_status"
        ]
    ].head(30)
)

Education pathway information integrated.
Shape: (2070, 14)
Rows missing CIP pathway information: 1380


,candidate_id,selected_occupation,skill,prescriptive_priority_percentage,skill_priority_rank,cip_program_count,cip_pathway_available,cip_status
0,Candidate_001,Administrative Assistant,Active Listening,64.04,1,NaN,NaN,NaN
1,Candidate_001,Administrative Assistant,Speaking,63.26,2,NaN,NaN,NaN
2,Candidate_001,Administrative Assistant,Monitoring,55.76,3,NaN,NaN,NaN
3,Candidate_001,Bookkeeper,Active Listening,53.97,1,1.0,True,CIP pathway available
4,Candidate_001,Bookkeeper,Speaking,49.41,2,1.0,True,CIP pathway available
5,Candidate_001,Bookkeeper,Monitoring,47.25,3,1.0,True,CIP pathway available
6,Candidate_001,Continuing Care Assistant,Active Listening,57.30,1,NaN,NaN,NaN
7,Candidate_001,Continuing Care Assistant,Monitoring,55.08,2,NaN,NaN,NaN
8,Candidate_001,Continuing Care Assistant,Speaking,54.30,3,NaN,NaN,NaN
9,Candidate_001,Delivery Driver,Active Listening,50.82,1,NaN,NaN,NaN


In [85]:
# ============================================================
# CELL 36 — Diagnose CIP Pathway Coverage
# ============================================================

print("CIP pathway coverage diagnosis\n")

# Total candidate-occupation combinations in the prescriptive table
prescriptive_pairs = (
    candidate_prescriptive_recommendations[
        ["candidate_id", "selected_occupation"]
    ]
    .drop_duplicates()
)

# Candidate-occupation combinations available in Week 5 CIP mapping
cip_pairs = (
    candidate_occupation_cip_mapping[
        ["candidate_id", "selected_occupation"]
    ]
    .drop_duplicates()
)

# Compare coverage
coverage_check = (
    prescriptive_pairs
    .merge(
        cip_pairs,
        on=[
            "candidate_id",
            "selected_occupation"
        ],
        how="left",
        indicator=True
    )
)

print(
    "Total prescriptive candidate-occupation pairs:",
    len(prescriptive_pairs)
)

print(
    "Candidate-occupation pairs with CIP mapping:",
    (
        coverage_check["_merge"] == "both"
    ).sum()
)

print(
    "Candidate-occupation pairs without CIP mapping:",
    (
        coverage_check["_merge"] == "left_only"
    ).sum()
)

# Show occupations with CIP coverage
cip_occupation_coverage = (
    coverage_check
    .assign(
        cip_available=(
            coverage_check["_merge"] == "both"
        )
    )
    .groupby("selected_occupation")
    .agg(
        candidate_occupation_pairs=(
            "cip_available",
            "size"
        ),
        pairs_with_cip=(
            "cip_available",
            "sum"
        )
    )
    .reset_index()
)

cip_occupation_coverage[
    "cip_coverage_percentage"
] = (
    cip_occupation_coverage["pairs_with_cip"]
    / cip_occupation_coverage[
        "candidate_occupation_pairs"
    ]
    * 100
).round(2)

display(
    cip_occupation_coverage
    .sort_values(
        "cip_coverage_percentage",
        ascending=False
    )
)

print("\nCIP mapping coverage diagnosis complete.")

CIP pathway coverage diagnosis

Total prescriptive candidate-occupation pairs: 840
Candidate-occupation pairs with CIP mapping: 280
Candidate-occupation pairs without CIP mapping: 560


,selected_occupation,candidate_occupation_pairs,pairs_with_cip,cip_coverage_percentage
14,Software Developer,56,39,69.64
10,Office Manager,56,34,60.71
9,Office Administrator,56,29,51.79
13,Secondary School Teacher,56,28,50.00
6,Information Technology (IT) Analyst,56,25,44.64
1,Bookkeeper,56,24,42.86
0,Administrative Assistant,56,22,39.29
7,Inside Sales Representative,56,21,37.50
8,Licensed Practical Nurse (L.P.N.),56,17,30.36
4,"Driver, Truck",56,16,28.57



CIP mapping coverage diagnosis complete.


In [87]:
# ============================================================
# CELL 37 — Finalize CIP Pathway Status
# ============================================================

# Preserve the original CIP information while creating
# explicit availability indicators.

candidate_prescriptive_recommendations[
    "cip_program_count"
] = (
    candidate_prescriptive_recommendations[
        "cip_program_count"
    ]
    .fillna(0)
    .astype(int)
)

candidate_prescriptive_recommendations[
    "cip_pathway_available"
] = (
    candidate_prescriptive_recommendations[
        "cip_pathway_available"
    ]
    .fillna(False)
    .astype(bool)
)

candidate_prescriptive_recommendations[
    "cip_status"
] = (
    candidate_prescriptive_recommendations[
        "cip_status"
    ]
    .fillna("No mapped CIP pathway available")
)

# Create a simple education action indicator
candidate_prescriptive_recommendations[
    "education_action"
] = np.where(
    candidate_prescriptive_recommendations[
        "cip_pathway_available"
    ],
    "Review mapped CIP education pathways",
    "Investigate education/training pathways separately"
)

print("CIP pathway status finalized.")

print(
    "\nCIP pathway availability:"
)

display(
    candidate_prescriptive_recommendations[
        "cip_pathway_available"
    ]
    .value_counts()
    .rename_axis("cip_pathway_available")
    .reset_index(name="row_count")
)

print(
    "\nCIP status distribution:"
)

display(
    candidate_prescriptive_recommendations[
        "cip_status"
    ]
    .value_counts()
    .rename_axis("cip_status")
    .reset_index(name="row_count")
)

print(
    "\nEducation action distribution:"
)

display(
    candidate_prescriptive_recommendations[
        "education_action"
    ]
    .value_counts()
    .rename_axis("education_action")
    .reset_index(name="row_count")
)

print("\nSample:")
display(
    candidate_prescriptive_recommendations[
        [
            "candidate_id",
            "selected_occupation",
            "skill",
            "cip_program_count",
            "cip_pathway_available",
            "cip_status",
            "education_action"
        ]
    ].head(20)
)

CIP pathway status finalized.

CIP pathway availability:


,cip_pathway_available,row_count
0,False,1439
1,True,631



CIP status distribution:


,cip_status,row_count
0,No mapped CIP pathway available,1380
1,CIP pathway available,631
2,No CIP pathway in crosswalk,59



Education action distribution:


,education_action,row_count
0,Investigate education/training pathways separately,1439
1,Review mapped CIP education pathways,631



Sample:


,candidate_id,selected_occupation,skill,cip_program_count,cip_pathway_available,cip_status,education_action
0,Candidate_001,Administrative Assistant,Active Listening,0,False,No mapped CIP pathway available,Investigate education/training pathways separately
1,Candidate_001,Administrative Assistant,Speaking,0,False,No mapped CIP pathway available,Investigate education/training pathways separately
2,Candidate_001,Administrative Assistant,Monitoring,0,False,No mapped CIP pathway available,Investigate education/training pathways separately
3,Candidate_001,Bookkeeper,Active Listening,1,True,CIP pathway available,Review mapped CIP education pathways
4,Candidate_001,Bookkeeper,Speaking,1,True,CIP pathway available,Review mapped CIP education pathways
5,Candidate_001,Bookkeeper,Monitoring,1,True,CIP pathway available,Review mapped CIP education pathways
6,Candidate_001,Continuing Care Assistant,Active Listening,0,False,No mapped CIP pathway available,Investigate education/training pathways separately
7,Candidate_001,Continuing Care Assistant,Monitoring,0,False,No mapped CIP pathway available,Investigate education/training pathways separately
8,Candidate_001,Continuing Care Assistant,Speaking,0,False,No mapped CIP pathway available,Investigate education/training pathways separately
9,Candidate_001,Delivery Driver,Active Listening,0,False,No mapped CIP pathway available,Investigate education/training pathways separately


In [66]:
# ============================================================
# CELL 38 — Validate CIP Integration
# ============================================================

print("Running CIP integration validation...\n")

# Check 1: Row count must remain unchanged
expected_rows = 2070
actual_rows = len(candidate_prescriptive_recommendations)

print("Expected rows:", expected_rows)
print("Actual rows:", actual_rows)

# Check 2: Candidate-occupation combinations
expected_pairs = 840

actual_pairs = (
    candidate_prescriptive_recommendations[
        ["candidate_id", "selected_occupation"]
    ]
    .drop_duplicates()
    .shape[0]
)

print("Expected candidate-occupation pairs:", expected_pairs)
print("Actual candidate-occupation pairs:", actual_pairs)

# Check 3: CIP program count must be non-negative
negative_cip_counts = (
    candidate_prescriptive_recommendations[
        "cip_program_count"
    ] < 0
).sum()

print(
    "Negative CIP program counts:",
    negative_cip_counts
)

# Check 4: CIP availability must agree with program count
availability_inconsistency = (
    (
        candidate_prescriptive_recommendations[
            "cip_pathway_available"
        ]
        &
        (
            candidate_prescriptive_recommendations[
                "cip_program_count"
            ] == 0
        )
    )
    |
    (
        ~candidate_prescriptive_recommendations[
            "cip_pathway_available"
        ]
        &
        (
            candidate_prescriptive_recommendations[
                "cip_program_count"
            ] > 0
        )
    )
).sum()

print(
    "CIP availability inconsistencies:",
    availability_inconsistency
)

# Check 5: Education action must agree with CIP availability
action_inconsistency = (
    (
        candidate_prescriptive_recommendations[
            "cip_pathway_available"
        ]
        &
        (
            candidate_prescriptive_recommendations[
                "education_action"
            ]
            != "Review mapped CIP education pathways"
        )
    )
    |
    (
        ~candidate_prescriptive_recommendations[
            "cip_pathway_available"
        ]
        &
        (
            candidate_prescriptive_recommendations[
                "education_action"
            ]
            != "Investigate education/training pathways separately"
        )
    )
).sum()

print(
    "Education action inconsistencies:",
    action_inconsistency
)

# Check 6: No missing values in finalized CIP fields
final_cip_columns = [
    "cip_program_count",
    "cip_pathway_available",
    "cip_status",
    "education_action"
]

missing_final_cip = (
    candidate_prescriptive_recommendations[
        final_cip_columns
    ]
    .isna()
    .sum()
    .sum()
)

print(
    "Missing values in finalized CIP fields:",
    missing_final_cip
)

# Final validation
if (
    actual_rows == expected_rows
    and actual_pairs == expected_pairs
    and negative_cip_counts == 0
    and availability_inconsistency == 0
    and action_inconsistency == 0
    and missing_final_cip == 0
):
    print(
        "\n✅ PASS — CIP integration validation successful."
    )
else:
    print(
        "\n⚠️ REVIEW — One or more CIP validation checks failed."
    )

Running CIP integration validation...

Expected rows: 2070
Actual rows: 2070
Expected candidate-occupation pairs: 840
Actual candidate-occupation pairs: 840
Negative CIP program counts: 0
CIP availability inconsistencies: 0
Education action inconsistencies: 0
Missing values in finalized CIP fields: 0

✅ PASS — CIP integration validation successful.


In [89]:
# ============================================================
# CELL 39 — Create Prescriptive Action Recommendations
# ============================================================

def generate_prescriptive_action(row):
    """
    Generate an interpretable action recommendation based on:
    1. Skill-gap priority
    2. Canadian labour demand
    3. CIP education pathway availability
    """

    skill = row["skill"]
    priority = row["prescriptive_priority_percentage"]
    demand = row["demand_percentage"]
    cip_available = row["cip_pathway_available"]

    # Priority classification
    if priority >= 70:
        priority_level = "High"
    elif priority >= 50:
        priority_level = "Medium"
    else:
        priority_level = "Lower"

    # Demand classification
    if demand >= 70:
        demand_level = "High"
    elif demand >= 40:
        demand_level = "Moderate"
    else:
        demand_level = "Lower"

    # Education action
    if cip_available:
        education_action = (
            "Review mapped CIP education pathways "
            "for this occupation."
        )
    else:
        education_action = (
            "Investigate relevant education or training "
            "pathways separately because no mapped CIP "
            "pathway is available."
        )

    # Main recommendation
    if priority_level == "High" and demand_level == "High":
        action = (
            f"Prioritize developing {skill}. "
            f"The skill gap has high prescriptive priority "
            f"({priority:.2f}%) and the occupation has high "
            f"Canadian labour demand ({demand:.2f}%)."
        )

    elif priority_level == "High":
        action = (
            f"Prioritize developing {skill}. "
            f"The skill gap has high prescriptive priority "
            f"({priority:.2f}%)."
        )

    elif demand_level == "High":
        action = (
            f"Consider developing {skill} because the "
            f"target occupation shows high Canadian labour "
            f"demand ({demand:.2f}%)."
        )

    elif priority_level == "Medium":
        action = (
            f"Consider developing {skill} as a medium-priority "
            f"skill gap ({priority:.2f}%)."
        )

    else:
        action = (
            f"Consider developing {skill} as an additional "
            f"skill for the target occupation."
        )

    return (
        action
        + " "
        + education_action
    )


candidate_prescriptive_recommendations[
    "priority_level"
] = (
    candidate_prescriptive_recommendations[
        "prescriptive_priority_percentage"
    ]
    .apply(
        lambda x:
        "High" if x >= 70
        else "Medium" if x >= 50
        else "Lower"
    )
)

candidate_prescriptive_recommendations[
    "demand_level"
] = (
    candidate_prescriptive_recommendations[
        "demand_percentage"
    ]
    .apply(
        lambda x:
        "High" if x >= 70
        else "Moderate" if x >= 40
        else "Lower"
    )
)

candidate_prescriptive_recommendations[
    "prescriptive_action"
] = (
    candidate_prescriptive_recommendations
    .apply(
        generate_prescriptive_action,
        axis=1
    )
)

print(
    "Prescriptive action recommendations created."
)

print(
    "Shape:",
    candidate_prescriptive_recommendations.shape
)

display(
    candidate_prescriptive_recommendations[
        [
            "candidate_id",
            "selected_occupation",
            "skill",
            "prescriptive_priority_percentage",
            "priority_level",
            "demand_percentage",
            "demand_level",
            "cip_status",
            "education_action",
            "prescriptive_action"
        ]
    ].head(30)
)

Prescriptive action recommendations created.
Shape: (2070, 18)


,candidate_id,selected_occupation,skill,prescriptive_priority_percentage,priority_level,demand_percentage,demand_level,cip_status,education_action,prescriptive_action
0,Candidate_001,Administrative Assistant,Active Listening,64.04,Medium,43.85,Moderate,No mapped CIP pathway available,Investigate education/training pathways separately,Consider developing Active Listening as a medium-priority skill gap (64.04%). Investigate relevant education or training pathways separately because no mapped CIP pathway is available.
1,Candidate_001,Administrative Assistant,Speaking,63.26,Medium,43.85,Moderate,No mapped CIP pathway available,Investigate education/training pathways separately,Consider developing Speaking as a medium-priority skill gap (63.26%). Investigate relevant education or training pathways separately because no mapped CIP pathway is available.
2,Candidate_001,Administrative Assistant,Monitoring,55.76,Medium,43.85,Moderate,No mapped CIP pathway available,Investigate education/training pathways separately,Consider developing Monitoring as a medium-priority skill gap (55.76%). Investigate relevant education or training pathways separately because no mapped CIP pathway is available.
3,Candidate_001,Bookkeeper,Active Listening,53.97,Medium,29.92,Lower,CIP pathway available,Review mapped CIP education pathways,Consider developing Active Listening as a medium-priority skill gap (53.97%). Review mapped CIP education pathways for this occupation.
4,Candidate_001,Bookkeeper,Speaking,49.41,Lower,29.92,Lower,CIP pathway available,Review mapped CIP education pathways,Consider developing Speaking as an additional skill for the target occupation. Review mapped CIP education pathways for this occupation.
5,Candidate_001,Bookkeeper,Monitoring,47.25,Lower,29.92,Lower,CIP pathway available,Review mapped CIP education pathways,Consider developing Monitoring as an additional skill for the target occupation. Review mapped CIP education pathways for this occupation.
6,Candidate_001,Continuing Care Assistant,Active Listening,57.30,Medium,43.96,Moderate,No mapped CIP pathway available,Investigate education/training pathways separately,Consider developing Active Listening as a medium-priority skill gap (57.30%). Investigate relevant education or training pathways separately because no mapped CIP pathway is available.
7,Candidate_001,Continuing Care Assistant,Monitoring,55.08,Medium,43.96,Moderate,No mapped CIP pathway available,Investigate education/training pathways separately,Consider developing Monitoring as a medium-priority skill gap (55.08%). Investigate relevant education or training pathways separately because no mapped CIP pathway is available.
8,Candidate_001,Continuing Care Assistant,Speaking,54.30,Medium,43.96,Moderate,No mapped CIP pathway available,Investigate education/training pathways separately,Consider developing Speaking as a medium-priority skill gap (54.30%). Investigate relevant education or training pathways separately because no mapped CIP pathway is available.
9,Candidate_001,Delivery Driver,Active Listening,50.82,Medium,37.05,Lower,No mapped CIP pathway available,Investigate education/training pathways separately,Consider developing Active Listening as a medium-priority skill gap (50.82%). Investigate relevant education or training pathways separately because no mapped CIP pathway is available.


In [93]:
# ============================================================
# CELL 40 — Integrate Week 6 Top-1 Recommended Occupation
# ============================================================

# Prepare the Week 6 top-1 recommendation table
top1_recommendation = (
    final_hybrid_top1[
        [
            "candidate_id",
            "selected_occupation",
            "hybrid_recommendation_score",
            "hybrid_occupation_rank",
            "recommendation_explanation"
        ]
    ]
    .drop_duplicates("candidate_id")
    .copy()
)

# Rename occupation column to clearly identify the
# Week 6 recommended occupation
top1_recommendation = (
    top1_recommendation
    .rename(
        columns={
            "selected_occupation":
                "recommended_occupation"
        }
    )
)

print(
    "Week 6 top-1 recommendation table prepared."
)

print(
    "Unique candidates:",
    top1_recommendation["candidate_id"].nunique()
)

print(
    "Rows:",
    len(top1_recommendation)
)

display(
    top1_recommendation.head(15)
)

Week 6 top-1 recommendation table prepared.
Unique candidates: 63
Rows: 63


,candidate_id,recommended_occupation,hybrid_recommendation_score,hybrid_occupation_rank,recommendation_explanation
0,Candidate_001,Software Developer,69.054471,1,Moderate recommendation based on combined candidate profile similarity and O*NET skill alignment.
1,Candidate_002,Office Manager,55.624220,1,Moderate recommendation based on combined candidate profile similarity and O*NET skill alignment.
2,Candidate_003,Administrative Assistant,93.244151,1,Strong match based on both baseline candidate features and weighted O*NET skill alignment.
3,Candidate_004,Software Developer,78.591423,1,Strong match based on both baseline candidate features and weighted O*NET skill alignment.
4,Candidate_005,Software Developer,53.350959,1,Moderate recommendation based on combined candidate profile similarity and O*NET skill alignment.
5,Candidate_006,Software Developer,90.463048,1,Strong match based on both baseline candidate features and weighted O*NET skill alignment.
6,Candidate_007,Food Service Supervisor,100.000000,1,Strong match based on both baseline candidate features and weighted O*NET skill alignment.
7,Candidate_008,Software Developer,68.084526,1,Moderate recommendation based on combined candidate profile similarity and O*NET skill alignment.
8,Candidate_009,Administrative Assistant,27.729322,1,Moderate recommendation based on combined candidate profile similarity and O*NET skill alignment.
9,Candidate_010,Office Manager,98.561151,1,Strong match based on both baseline candidate features and weighted O*NET skill alignment.


In [95]:
# ============================================================
# CELL 41 — Filter Prescriptive Recommendations to
#            Week 6 Recommended Occupation
# ============================================================

# Merge Week 6 top-1 occupation into the prescriptive table
recommended_occupation_prescriptive = (
    candidate_prescriptive_recommendations
    .merge(
        top1_recommendation[
            [
                "candidate_id",
                "recommended_occupation",
                "hybrid_recommendation_score",
                "recommendation_explanation"
            ]
        ],
        on="candidate_id",
        how="left"
    )
)

# Keep only the prescriptive recommendations that belong
# to the occupation recommended by the Week 6 hybrid model
recommended_occupation_prescriptive = (
    recommended_occupation_prescriptive[
        recommended_occupation_prescriptive[
            "selected_occupation"
        ]
        ==
        recommended_occupation_prescriptive[
            "recommended_occupation"
        ]
    ]
    .copy()
)

print(
    "Prescriptive recommendations aligned with "
    "Week 6 Top-1 occupations."
)

print(
    "Rows:",
    len(recommended_occupation_prescriptive)
)

print(
    "Unique candidates:",
    recommended_occupation_prescriptive[
        "candidate_id"
    ].nunique()
)

print(
    "Unique recommended occupations:",
    recommended_occupation_prescriptive[
        "recommended_occupation"
    ].nunique()
)

print(
    "\nRecommended occupation distribution:"
)

display(
    recommended_occupation_prescriptive[
        [
            "recommended_occupation",
            "candidate_id"
        ]
    ]
    .drop_duplicates()
    ["recommended_occupation"]
    .value_counts()
    .rename_axis("recommended_occupation")
    .reset_index(name="candidate_count")
)

print(
    "\nSample aligned prescriptive recommendations:"
)

display(
    recommended_occupation_prescriptive[
        [
            "candidate_id",
            "recommended_occupation",
            "hybrid_recommendation_score",
            "skill",
            "prescriptive_priority_percentage",
            "priority_level",
            "demand_percentage",
            "demand_level",
            "cip_status",
            "education_action",
            "prescriptive_action"
        ]
    ]
    .head(30)
)

Prescriptive recommendations aligned with Week 6 Top-1 occupations.
Rows: 138
Unique candidates: 56
Unique recommended occupations: 6

Recommended occupation distribution:


,recommended_occupation,candidate_count
0,Software Developer,26
1,Office Manager,15
2,Administrative Assistant,12
3,Delivery Driver,1
4,"Driver, Truck",1
5,Bookkeeper,1



Sample aligned prescriptive recommendations:


,candidate_id,recommended_occupation,hybrid_recommendation_score,skill,prescriptive_priority_percentage,priority_level,demand_percentage,demand_level,cip_status,education_action,prescriptive_action
42,Candidate_001,Software Developer,69.054471,Active Listening,52.72,Medium,22.91,Lower,CIP pathway available,Review mapped CIP education pathways,Consider developing Active Listening as a medium-priority skill gap (52.72%). Review mapped CIP education pathways for this occupation.
43,Candidate_001,Software Developer,69.054471,Speaking,49.60,Lower,22.91,Lower,CIP pathway available,Review mapped CIP education pathways,Consider developing Speaking as an additional skill for the target occupation. Review mapped CIP education pathways for this occupation.
44,Candidate_001,Software Developer,69.054471,Monitoring,48.16,Lower,22.91,Lower,CIP pathway available,Review mapped CIP education pathways,Consider developing Monitoring as an additional skill for the target occupation. Review mapped CIP education pathways for this occupation.
75,Candidate_002,Office Manager,55.624220,Monitoring,69.14,Medium,52.86,Moderate,CIP pathway available,Review mapped CIP education pathways,Consider developing Monitoring as a medium-priority skill gap (69.14%). Review mapped CIP education pathways for this occupation.
76,Candidate_002,Office Manager,55.624220,Speaking,69.14,Medium,52.86,Moderate,CIP pathway available,Review mapped CIP education pathways,Consider developing Speaking as a medium-priority skill gap (69.14%). Review mapped CIP education pathways for this occupation.
77,Candidate_002,Office Manager,55.624220,Active Listening,68.42,Medium,52.86,Moderate,CIP pathway available,Review mapped CIP education pathways,Consider developing Active Listening as a medium-priority skill gap (68.42%). Review mapped CIP education pathways for this occupation.
90,Candidate_003,Administrative Assistant,93.244151,Mathematics,39.26,Lower,43.85,Moderate,CIP pathway available,Review mapped CIP education pathways,Consider developing Mathematics as an additional skill for the target occupation. Review mapped CIP education pathways for this occupation.
91,Candidate_003,Administrative Assistant,93.244151,Science,23.54,Lower,43.85,Moderate,CIP pathway available,Review mapped CIP education pathways,Consider developing Science as an additional skill for the target occupation. Review mapped CIP education pathways for this occupation.
148,Candidate_004,Software Developer,78.591423,Active Listening,52.72,Medium,22.91,Lower,CIP pathway available,Review mapped CIP education pathways,Consider developing Active Listening as a medium-priority skill gap (52.72%). Review mapped CIP education pathways for this occupation.
149,Candidate_004,Software Developer,78.591423,Speaking,49.60,Lower,22.91,Lower,CIP pathway available,Review mapped CIP education pathways,Consider developing Speaking as an additional skill for the target occupation. Review mapped CIP education pathways for this occupation.


In [97]:
# ============================================================
# CELL 42 — Validate Top-1 Prescriptive Alignment
# ============================================================

print(
    "Running Top-1 prescriptive alignment validation..."
)

# ------------------------------------------------------------
# 1. Required columns
# ------------------------------------------------------------

required_columns = [
    "candidate_id",
    "recommended_occupation",
    "selected_occupation",
    "skill",
    "prescriptive_priority_percentage",
    "demand_percentage",
    "cip_status",
    "prescriptive_action"
]

missing_columns = [
    col
    for col in required_columns
    if col not in recommended_occupation_prescriptive.columns
]

print(
    "\nMissing required columns:",
    missing_columns
)

# ------------------------------------------------------------
# 2. Every selected occupation must equal the
#    Week 6 recommended occupation
# ------------------------------------------------------------

occupation_alignment_errors = (
    recommended_occupation_prescriptive[
        "selected_occupation"
    ]
    !=
    recommended_occupation_prescriptive[
        "recommended_occupation"
    ]
).sum()

print(
    "Occupation alignment errors:",
    occupation_alignment_errors
)

# ------------------------------------------------------------
# 3. All candidates must exist in Week 6 Top-1 table
# ------------------------------------------------------------

top1_candidate_ids = set(
    top1_recommendation["candidate_id"]
)

prescriptive_candidate_ids = set(
    recommended_occupation_prescriptive["candidate_id"]
)

unknown_candidates = (
    prescriptive_candidate_ids
    - top1_candidate_ids
)

print(
    "Candidates not found in Week 6 Top-1:",
    len(unknown_candidates)
)

# ------------------------------------------------------------
# 4. No candidate should have more than 3
#    prescriptive skills
# ------------------------------------------------------------

skills_per_candidate = (
    recommended_occupation_prescriptive
    .groupby(
        [
            "candidate_id",
            "recommended_occupation"
        ]
    )
    .size()
)

groups_over_three = (
    skills_per_candidate > 3
).sum()

print(
    "Candidate-occupation groups with "
    "more than 3 skills:",
    groups_over_three
)

# ------------------------------------------------------------
# 5. Validate priority range
# ------------------------------------------------------------

invalid_priority = (
    (
        recommended_occupation_prescriptive[
            "prescriptive_priority_percentage"
        ] < 0
    )
    |
    (
        recommended_occupation_prescriptive[
            "prescriptive_priority_percentage"
        ] > 100
    )
).sum()

print(
    "Prescriptive priority outside 0–100:",
    invalid_priority
)

# ------------------------------------------------------------
# 6. Validate Canadian demand range
# ------------------------------------------------------------

invalid_demand = (
    (
        recommended_occupation_prescriptive[
            "demand_percentage"
        ] < 0
    )
    |
    (
        recommended_occupation_prescriptive[
            "demand_percentage"
        ] > 100
    )
).sum()

print(
    "Canadian demand outside 0–100:",
    invalid_demand
)

# ------------------------------------------------------------
# 7. Check missing values in critical fields
# ------------------------------------------------------------

critical_fields = [
    "candidate_id",
    "recommended_occupation",
    "skill",
    "prescriptive_priority_percentage",
    "demand_percentage",
    "prescriptive_action"
]

missing_critical_values = (
    recommended_occupation_prescriptive[
        critical_fields
    ]
    .isna()
    .sum()
    .sum()
)

print(
    "Missing critical values:",
    missing_critical_values
)

# ------------------------------------------------------------
# 8. Confirm every candidate has only one
#    recommended occupation
# ------------------------------------------------------------

occupation_count_per_candidate = (
    recommended_occupation_prescriptive[
        [
            "candidate_id",
            "recommended_occupation"
        ]
    ]
    .drop_duplicates()
    .groupby("candidate_id")
    ["recommended_occupation"]
    .nunique()
)

multiple_recommended_occupations = (
    occupation_count_per_candidate > 1
).sum()

print(
    "Candidates with multiple Top-1 occupations:",
    multiple_recommended_occupations
)

# ------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------

if (
    len(missing_columns) == 0
    and occupation_alignment_errors == 0
    and len(unknown_candidates) == 0
    and groups_over_three == 0
    and invalid_priority == 0
    and invalid_demand == 0
    and missing_critical_values == 0
    and multiple_recommended_occupations == 0
):

    print(
        "\n✅ PASS — Top-1 prescriptive alignment "
        "validation successful."
    )

else:

    print(
        "\n❌ FAIL — Review the validation results "
        "before continuing."
    )

Running Top-1 prescriptive alignment validation...

Missing required columns: []
Occupation alignment errors: 0
Candidates not found in Week 6 Top-1: 0
Candidate-occupation groups with more than 3 skills: 0
Prescriptive priority outside 0–100: 0
Canadian demand outside 0–100: 0
Missing critical values: 0
Candidates with multiple Top-1 occupations: 0

✅ PASS — Top-1 prescriptive alignment validation successful.


In [101]:
# ============================================================
# CELL 43 — Final Candidate-Level Prescriptive Summary
# ============================================================

# ------------------------------------------------------------
# Create ordered skill recommendations for each candidate
# ------------------------------------------------------------

def combine_skill_recommendations(group):
    """
    Combine the top missing skills into a readable
    candidate-level recommendation.
    """

    ordered = (
        group
        .sort_values(
            "prescriptive_priority_score",
            ascending=False
        )
    )

    skills = ordered["skill"].tolist()

    priorities = ordered[
        "prescriptive_priority_percentage"
    ].tolist()

    skill_parts = [
        f"{skill} ({priority:.2f}%)"
        for skill, priority
        in zip(skills, priorities)
    ]

    return "; ".join(skill_parts)


# ------------------------------------------------------------
# Candidate-level aggregation
# ------------------------------------------------------------

prescriptive_candidate_summary = (
    recommended_occupation_prescriptive
    .groupby(
        [
            "candidate_id",
            "recommended_occupation"
        ],
        as_index=False
    )
    .agg(
        hybrid_recommendation_score=(
            "hybrid_recommendation_score",
            "first"
        ),

        recommendation_explanation=(
            "recommendation_explanation",
            "first"
        ),

        top_missing_skills=(
            "skill",
            lambda x: ", ".join(x)
        ),

        average_prescriptive_priority=(
            "prescriptive_priority_percentage",
            "mean"
        ),

        maximum_prescriptive_priority=(
            "prescriptive_priority_percentage",
            "max"
        ),

        demand_percentage=(
            "demand_percentage",
            "first"
        ),

        demand_level=(
            "demand_level",
            "first"
        ),

        cip_status=(
            "cip_status",
            "first"
        ),

        education_action=(
            "education_action",
            "first"
        )
    )
)

# ------------------------------------------------------------
# Add ordered skill-priority text
# ------------------------------------------------------------

ordered_skill_text = (
    recommended_occupation_prescriptive
    .groupby(
        [
            "candidate_id",
            "recommended_occupation"
        ]
    )
    .apply(
        combine_skill_recommendations
    )
    .reset_index(
        name="prioritized_skill_gaps"
    )
)

prescriptive_candidate_summary = (
    prescriptive_candidate_summary
    .drop(
        columns=["top_missing_skills"]
    )
    .merge(
        ordered_skill_text,
        on=[
            "candidate_id",
            "recommended_occupation"
        ],
        how="left"
    )
)

# ------------------------------------------------------------
# Round numerical values
# ------------------------------------------------------------

prescriptive_candidate_summary[
    "hybrid_recommendation_score"
] = (
    prescriptive_candidate_summary[
        "hybrid_recommendation_score"
    ]
    .round(2)
)

prescriptive_candidate_summary[
    "average_prescriptive_priority"
] = (
    prescriptive_candidate_summary[
        "average_prescriptive_priority"
    ]
    .round(2)
)

prescriptive_candidate_summary[
    "maximum_prescriptive_priority"
] = (
    prescriptive_candidate_summary[
        "maximum_prescriptive_priority"
    ]
    .round(2)
)

prescriptive_candidate_summary[
    "demand_percentage"
] = (
    prescriptive_candidate_summary[
        "demand_percentage"
    ]
    .round(2)
)

# ------------------------------------------------------------
# Final column ordering
# ------------------------------------------------------------

prescriptive_candidate_summary = (
    prescriptive_candidate_summary[
        [
            "candidate_id",
            "recommended_occupation",
            "hybrid_recommendation_score",
            "recommendation_explanation",
            "prioritized_skill_gaps",
            "average_prescriptive_priority",
            "maximum_prescriptive_priority",
            "demand_percentage",
            "demand_level",
            "cip_status",
            "education_action"
        ]
    ]
    .sort_values(
        [
            "candidate_id"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Final candidate-level prescriptive summary created."
)

print(
    "Shape:",
    prescriptive_candidate_summary.shape
)

print(
    "Unique candidates:",
    prescriptive_candidate_summary[
        "candidate_id"
    ].nunique()
)

display(
    prescriptive_candidate_summary.head(20)
)

Final candidate-level prescriptive summary created.
Shape: (56, 11)
Unique candidates: 56


,candidate_id,recommended_occupation,hybrid_recommendation_score,recommendation_explanation,prioritized_skill_gaps,average_prescriptive_priority,maximum_prescriptive_priority,demand_percentage,demand_level,cip_status,education_action
0,Candidate_001,Software Developer,69.05,Moderate recommendation based on combined candidate profile similarity and O*NET skill alignment.,Active Listening (52.72%); Speaking (49.60%); Monitoring (48.16%),50.16,52.72,22.91,Lower,CIP pathway available,Review mapped CIP education pathways
1,Candidate_002,Office Manager,55.62,Moderate recommendation based on combined candidate profile similarity and O*NET skill alignment.,Monitoring (69.14%); Speaking (69.14%); Active Listening (68.42%),68.90,69.14,52.86,Moderate,CIP pathway available,Review mapped CIP education pathways
2,Candidate_003,Administrative Assistant,93.24,Strong match based on both baseline candidate features and weighted O*NET skill alignment.,Mathematics (39.26%); Science (23.54%),31.40,39.26,43.85,Moderate,CIP pathway available,Review mapped CIP education pathways
3,Candidate_004,Software Developer,78.59,Strong match based on both baseline candidate features and weighted O*NET skill alignment.,Active Listening (52.72%); Speaking (49.60%),51.16,52.72,22.91,Lower,CIP pathway available,Review mapped CIP education pathways
4,Candidate_005,Software Developer,53.35,Moderate recommendation based on combined candidate profile similarity and O*NET skill alignment.,Active Listening (52.72%); Writing (50.38%); Speaking (49.60%),50.90,52.72,22.91,Lower,CIP pathway available,Review mapped CIP education pathways
5,Candidate_006,Software Developer,90.46,Strong match based on both baseline candidate features and weighted O*NET skill alignment.,Monitoring (48.16%),48.16,48.16,22.91,Lower,CIP pathway available,Review mapped CIP education pathways
6,Candidate_008,Software Developer,68.08,Moderate recommendation based on combined candidate profile similarity and O*NET skill alignment.,Active Listening (52.72%); Writing (50.38%); Speaking (49.60%),50.90,52.72,22.91,Lower,CIP pathway available,Review mapped CIP education pathways
7,Candidate_009,Administrative Assistant,27.73,Moderate recommendation based on combined candidate profile similarity and O*NET skill alignment.,Active Listening (64.04%); Speaking (63.26%); Critical Thinking (57.26%),61.52,64.04,43.85,Moderate,CIP pathway available,Review mapped CIP education pathways
8,Candidate_010,Office Manager,98.56,Strong match based on both baseline candidate features and weighted O*NET skill alignment.,Science (27.14%),27.14,27.14,52.86,Moderate,CIP pathway available,Review mapped CIP education pathways
9,Candidate_011,Office Manager,67.69,Moderate recommendation based on combined candidate profile similarity and O*NET skill alignment.,Speaking (69.14%); Active Listening (68.42%); Mathematics (54.14%),63.90,69.14,52.86,Moderate,CIP pathway available,Review mapped CIP education pathways


In [103]:
# ============================================================
# CELL 44 — Validate Final Candidate-Level Prescriptive Summary
# ============================================================

print(
    "Running final candidate-level prescriptive validation..."
)

# ------------------------------------------------------------
# 1. Required columns
# ------------------------------------------------------------

required_columns = [
    "candidate_id",
    "recommended_occupation",
    "hybrid_recommendation_score",
    "recommendation_explanation",
    "prioritized_skill_gaps",
    "average_prescriptive_priority",
    "maximum_prescriptive_priority",
    "demand_percentage",
    "demand_level",
    "cip_status",
    "education_action"
]

missing_columns = [
    col
    for col in required_columns
    if col not in prescriptive_candidate_summary.columns
]

print(
    "\nMissing required columns:",
    missing_columns
)

# ------------------------------------------------------------
# 2. One row per candidate
# ------------------------------------------------------------

duplicate_candidates = (
    prescriptive_candidate_summary[
        "candidate_id"
    ]
    .duplicated()
    .sum()
)

print(
    "Duplicate candidate rows:",
    duplicate_candidates
)

# ------------------------------------------------------------
# 3. Verify candidates belong to the Week 6 Top-1 set
# ------------------------------------------------------------

valid_top1_candidates = set(
    top1_recommendation["candidate_id"]
)

summary_candidates = set(
    prescriptive_candidate_summary[
        "candidate_id"
    ]
)

unknown_candidates = (
    summary_candidates
    - valid_top1_candidates
)

print(
    "Candidates not found in Week 6 Top-1:",
    len(unknown_candidates)
)

# ------------------------------------------------------------
# 4. Verify recommended occupations
# ------------------------------------------------------------

top1_lookup = (
    top1_recommendation
    .set_index("candidate_id")
    ["recommended_occupation"]
    .to_dict()
)

occupation_mismatches = 0

for _, row in prescriptive_candidate_summary.iterrows():

    candidate = row["candidate_id"]

    expected_occupation = (
        top1_lookup.get(candidate)
    )

    if (
        expected_occupation
        != row["recommended_occupation"]
    ):
        occupation_mismatches += 1

print(
    "Recommended occupation mismatches:",
    occupation_mismatches
)

# ------------------------------------------------------------
# 5. Validate priority values
# ------------------------------------------------------------

invalid_average_priority = (
    (
        prescriptive_candidate_summary[
            "average_prescriptive_priority"
        ] < 0
    )
    |
    (
        prescriptive_candidate_summary[
            "average_prescriptive_priority"
        ] > 100
    )
).sum()

invalid_max_priority = (
    (
        prescriptive_candidate_summary[
            "maximum_prescriptive_priority"
        ] < 0
    )
    |
    (
        prescriptive_candidate_summary[
            "maximum_prescriptive_priority"
        ] > 100
    )
).sum()

print(
    "Average priority outside 0–100:",
    invalid_average_priority
)

print(
    "Maximum priority outside 0–100:",
    invalid_max_priority
)

# ------------------------------------------------------------
# 6. Validate Canadian demand
# ------------------------------------------------------------

invalid_demand = (
    (
        prescriptive_candidate_summary[
            "demand_percentage"
        ] < 0
    )
    |
    (
        prescriptive_candidate_summary[
            "demand_percentage"
        ] > 100
    )
).sum()

print(
    "Canadian demand outside 0–100:",
    invalid_demand
)

# ------------------------------------------------------------
# 7. Validate critical missing values
# ------------------------------------------------------------

critical_fields = [
    "candidate_id",
    "recommended_occupation",
    "hybrid_recommendation_score",
    "prioritized_skill_gaps",
    "average_prescriptive_priority",
    "maximum_prescriptive_priority",
    "demand_percentage",
    "demand_level",
    "cip_status",
    "education_action"
]

missing_critical_values = (
    prescriptive_candidate_summary[
        critical_fields
    ]
    .isna()
    .sum()
    .sum()
)

print(
    "Missing critical values:",
    missing_critical_values
)

# ------------------------------------------------------------
# 8. Validate recommendation text
# ------------------------------------------------------------

empty_recommendations = (
    prescriptive_candidate_summary[
        "prioritized_skill_gaps"
    ]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Empty prioritized skill recommendations:",
    empty_recommendations
)

# ------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------

if (
    len(missing_columns) == 0
    and duplicate_candidates == 0
    and len(unknown_candidates) == 0
    and occupation_mismatches == 0
    and invalid_average_priority == 0
    and invalid_max_priority == 0
    and invalid_demand == 0
    and missing_critical_values == 0
    and empty_recommendations == 0
):

    print(
        "\n✅ PASS — Final candidate-level "
        "prescriptive validation successful."
    )

else:

    print(
        "\n❌ FAIL — Review the validation results "
        "before continuing."
    )

Running final candidate-level prescriptive validation...

Missing required columns: []
Duplicate candidate rows: 0
Candidates not found in Week 6 Top-1: 0
Recommended occupation mismatches: 0
Average priority outside 0–100: 0
Maximum priority outside 0–100: 0
Canadian demand outside 0–100: 0
Missing critical values: 0
Empty prioritized skill recommendations: 0

✅ PASS — Final candidate-level prescriptive validation successful.


In [109]:
# ============================================================
# CELL 45 — Export Week 7 Prescriptive Outputs
# ============================================================

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

week7_prescriptive_dir = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "Tables",
    "Week7"
)

os.makedirs(
    week7_prescriptive_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Detailed prescriptive skill-gap table
# ------------------------------------------------------------

missing_skill_priorities_path = os.path.join(
    week7_prescriptive_dir,
    "week7_missing_skill_priorities.csv"
)

missing_skill_priorities.to_csv(
    missing_skill_priorities_path,
    index=False
)

# ------------------------------------------------------------
# 2. Candidate-level prescriptive recommendations
# ------------------------------------------------------------

candidate_prescriptive_path = os.path.join(
    week7_prescriptive_dir,
    "week7_candidate_prescriptive_recommendations.csv"
)

candidate_prescriptive_recommendations.to_csv(
    candidate_prescriptive_path,
    index=False
)

# ------------------------------------------------------------
# 3. Top-1 occupation aligned prescriptive recommendations
# ------------------------------------------------------------

top1_prescriptive_path = os.path.join(
    week7_prescriptive_dir,
    "week7_top1_prescriptive_skill_gaps.csv"
)

recommended_occupation_prescriptive.to_csv(
    top1_prescriptive_path,
    index=False
)

# ------------------------------------------------------------
# 4. Final candidate-level prescriptive summary
# ------------------------------------------------------------

final_prescriptive_path = os.path.join(
    week7_prescriptive_dir,
    "week7_final_prescriptive_recommendations.csv"
)

prescriptive_candidate_summary.to_csv(
    final_prescriptive_path,
    index=False
)

# ------------------------------------------------------------
# 5. Prescriptive validation summary
# ------------------------------------------------------------

prescriptive_validation_summary = pd.DataFrame(
    {
        "validation_metric": [
            "Missing required columns",
            "Duplicate candidate rows",
            "Candidates not found in Week 6 Top-1",
            "Recommended occupation mismatches",
            "Average priority outside 0-100",
            "Maximum priority outside 0-100",
            "Canadian demand outside 0-100",
            "Missing critical values",
            "Empty prioritized skill recommendations"
        ],

        "result": [
            len(missing_columns),
            duplicate_candidates,
            len(unknown_candidates),
            occupation_mismatches,
            invalid_average_priority,
            invalid_max_priority,
            invalid_demand,
            missing_critical_values,
            empty_recommendations
        ],

        "status": [
            "PASS" if len(missing_columns) == 0 else "FAIL",
            "PASS" if duplicate_candidates == 0 else "FAIL",
            "PASS" if len(unknown_candidates) == 0 else "FAIL",
            "PASS" if occupation_mismatches == 0 else "FAIL",
            "PASS" if invalid_average_priority == 0 else "FAIL",
            "PASS" if invalid_max_priority == 0 else "FAIL",
            "PASS" if invalid_demand == 0 else "FAIL",
            "PASS" if missing_critical_values == 0 else "FAIL",
            "PASS" if empty_recommendations == 0 else "FAIL"
        ]
    }
)

validation_path = os.path.join(
    week7_prescriptive_dir,
    "week7_prescriptive_validation_summary.csv"
)

prescriptive_validation_summary.to_csv(
    validation_path,
    index=False
)

# ------------------------------------------------------------
# Confirm files
# ------------------------------------------------------------

print(
    "Week 7 prescriptive outputs exported successfully."
)

print(
    "\nOutput directory:"
)

print(
    week7_prescriptive_dir
)

print(
    "\nFiles created:"
)

for file_name in [
    "week7_missing_skill_priorities.csv",
    "week7_candidate_prescriptive_recommendations.csv",
    "week7_top1_prescriptive_skill_gaps.csv",
    "week7_final_prescriptive_recommendations.csv",
    "week7_prescriptive_validation_summary.csv"
]:
    file_path = os.path.join(
        week7_prescriptive_dir,
        file_name
    )

    print(
        "✓",
        file_name,
        "|",
        os.path.getsize(file_path),
        "bytes"
    )

Week 7 prescriptive outputs exported successfully.

Output directory:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7

Files created:
✓ week7_missing_skill_priorities.csv | 456045 bytes
✓ week7_candidate_prescriptive_recommendations.csv | 759048 bytes
✓ week7_top1_prescriptive_skill_gaps.csv | 60603 bytes
✓ week7_final_prescriptive_recommendations.csv | 15387 bytes
✓ week7_prescriptive_validation_summary.csv | 382 bytes


In [111]:
# ============================================================
# CELL 46 — Build RAG Knowledge Base
# ============================================================

print("Building RAG knowledge base...")

# ------------------------------------------------------------
# 1. O*NET occupation profiles
# ------------------------------------------------------------

rag_occupation = (
    onet_integrated_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "skill_count",
            "knowledge_count",
            "ability_count",
            "task_count",
            "education_record_count",
            "training_record_count",
            "job_zone"
        ]
    ]
    .copy()
)

rag_occupation["knowledge_type"] = "Occupation Profile"

# ------------------------------------------------------------
# 2. O*NET skills
# ------------------------------------------------------------

rag_skills = (
    onet_required_skills_clean[
        [
            "selected_occupation",
            "onet_soc_code",
            "skill",
            "importance",
            "level",
            "onet_title"
        ]
    ]
    .copy()
)

rag_skills["knowledge_type"] = "O*NET Skill"

# ------------------------------------------------------------
# 3. O*NET tasks
# ------------------------------------------------------------

rag_tasks = (
    onet_tasks_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "task",
            "task_type",
            "onet_title"
        ]
    ]
    .copy()
)

rag_tasks["knowledge_type"] = "O*NET Task"

# ------------------------------------------------------------
# 4. Canadian labour demand
# ------------------------------------------------------------

rag_demand = (
    labour_demand[
        [
            "selected_occupation",
            "noc21_code",
            "posting_count",
            "total_vacancies",
            "province_count",
            "demand_score",
            "demand_percentage"
        ]
    ]
    .copy()
)

rag_demand["knowledge_type"] = "Canadian Labour Demand"

# ------------------------------------------------------------
# 5. CIP education pathways
# ------------------------------------------------------------

rag_cip = (
    candidate_occupation_cip_mapping[
        [
            "selected_occupation",
            "cip_program_count",
            "cip_pathway_available",
            "cip_status"
        ]
    ]
    .drop_duplicates()
    .copy()
)

rag_cip["knowledge_type"] = "CIP Education Pathway"

# ------------------------------------------------------------
# 6. Integrated occupation profile
# ------------------------------------------------------------

rag_integrated = (
    onet_cip_integrated_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "education_record_count",
            "training_record_count",
            "job_zone",
            "cip_program_count",
            "cip_soc_mapping_count"
        ]
    ]
    .drop_duplicates()
    .copy()
)

rag_integrated["knowledge_type"] = (
    "Integrated Occupation Profile"
)

# ------------------------------------------------------------
# Display source sizes
# ------------------------------------------------------------

print("\nRAG source record counts:")

print(
    "O*NET occupation profiles:",
    len(rag_occupation)
)

print(
    "O*NET skills:",
    len(rag_skills)
)

print(
    "O*NET tasks:",
    len(rag_tasks)
)

print(
    "Canadian labour demand:",
    len(rag_demand)
)

print(
    "CIP pathways:",
    len(rag_cip)
)

print(
    "Integrated occupation profiles:",
    len(rag_integrated)
)

# ------------------------------------------------------------
# Display samples
# ------------------------------------------------------------

print("\nO*NET occupation sample:")
display(rag_occupation.head(5))

print("\nO*NET skill sample:")
display(rag_skills.head(5))

print("\nCanadian demand sample:")
display(rag_demand.head(5))

print("\nCIP pathway sample:")
display(rag_cip.head(5))

Building RAG knowledge base...

RAG source record counts:
O*NET occupation profiles: 20
O*NET skills: 150
O*NET tasks: 487
Canadian labour demand: 15
CIP pathways: 15
Integrated occupation profiles: 20

O*NET occupation sample:


,selected_occupation,onet_soc_code,onet_title,skill_count,knowledge_count,ability_count,task_count,education_record_count,training_record_count,job_zone,knowledge_type
0,Software Developer,15-1252.00,Software Developers,10,33,52,17,12,29,4,Occupation Profile
1,Information Technology (IT) Analyst,15-1211.00,Computer Systems Analysts,10,33,52,22,12,29,4,Occupation Profile
2,Information Technology (IT) Analyst,15-1211.01,Health Informatics Specialists,10,33,52,17,12,29,5,Occupation Profile
3,Information Technology (IT) Analyst,15-1212.00,Information Security Analysts,10,33,52,11,12,29,4,Occupation Profile
4,Information Technology (IT) Analyst,15-1253.00,Software Quality Assurance Analysts and Testers,10,33,52,30,12,29,4,Occupation Profile



O*NET skill sample:


,selected_occupation,onet_soc_code,skill,importance,level,onet_title,knowledge_type
0,Administrative Assistant,43-6014.00,Active Learning,2.88,3.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",O*NET Skill
1,Administrative Assistant,43-6014.00,Active Listening,4.00,3.75,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",O*NET Skill
2,Administrative Assistant,43-6014.00,Critical Thinking,3.00,3.62,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",O*NET Skill
3,Administrative Assistant,43-6014.00,Learning Strategies,2.12,1.88,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",O*NET Skill
4,Administrative Assistant,43-6014.00,Mathematics,2.00,1.62,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",O*NET Skill



Canadian demand sample:


,selected_occupation,noc21_code,posting_count,total_vacancies,province_count,demand_score,demand_percentage,knowledge_type
0,"Driver, Truck",73300,1423,3705,12,0.888027,88.80,Canadian Labour Demand
1,Retail Sales Associate,64100,1876,2449,13,0.864399,86.44,Canadian Labour Demand
2,Inside Sales Representative,64100,1876,2449,13,0.864399,86.44,Canadian Labour Demand
3,Food Service Supervisor,62020,1061,2524,12,0.683338,68.33,Canadian Labour Demand
4,Restaurant Manager,60030,1139,1473,12,0.586501,58.65,Canadian Labour Demand



CIP pathway sample:


,selected_occupation,cip_program_count,cip_pathway_available,cip_status,knowledge_type
0,Software Developer,20,True,CIP pathway available,CIP Education Pathway
1,Information Technology (IT) Analyst,22,True,CIP pathway available,CIP Education Pathway
2,Secondary School Teacher,92,True,CIP pathway available,CIP Education Pathway
3,Bookkeeper,1,True,CIP pathway available,CIP Education Pathway
4,Licensed Practical Nurse (L.P.N.),2,True,CIP pathway available,CIP Education Pathway


In [113]:
# ============================================================
# CELL 47 — Convert RAG Sources into Retrieval Documents
# ============================================================

print("Creating standardized RAG retrieval documents...")

rag_documents = []

# ------------------------------------------------------------
# 1. O*NET OCCUPATION PROFILE DOCUMENTS
# ------------------------------------------------------------

for _, row in rag_occupation.iterrows():

    document_text = (
        f"Occupation: {row['selected_occupation']}. "
        f"O*NET occupation: {row['onet_title']} "
        f"({row['onet_soc_code']}). "
        f"Job Zone: {row['job_zone']}. "
        f"The O*NET profile contains "
        f"{row['skill_count']} skill records, "
        f"{row['knowledge_count']} knowledge records, "
        f"{row['ability_count']} ability records, "
        f"{row['task_count']} task records, "
        f"{row['education_record_count']} education records, "
        f"and {row['training_record_count']} training records."
    )

    rag_documents.append(
        {
            "selected_occupation":
                row["selected_occupation"],

            "source_type":
                "Occupation Profile",

            "source_name":
                "O*NET",

            "source_reference":
                row["onet_soc_code"],

            "document_text":
                document_text
        }
    )


# ------------------------------------------------------------
# 2. O*NET SKILL DOCUMENTS
# ------------------------------------------------------------

for _, row in rag_skills.iterrows():

    document_text = (
        f"For {row['selected_occupation']}, "
        f"O*NET identifies {row['skill']} as a relevant skill. "
        f"The skill importance score is "
        f"{row['importance']:.2f} and the required level is "
        f"{row['level']:.2f}. "
        f"O*NET occupation: {row['onet_title']} "
        f"({row['onet_soc_code']})."
    )

    rag_documents.append(
        {
            "selected_occupation":
                row["selected_occupation"],

            "source_type":
                "O*NET Skill",

            "source_name":
                "O*NET",

            "source_reference":
                row["onet_soc_code"],

            "document_text":
                document_text
        }
    )


# ------------------------------------------------------------
# 3. O*NET TASK DOCUMENTS
# ------------------------------------------------------------

for _, row in rag_tasks.iterrows():

    task_text = str(row["task"]).strip()

    if task_text:

        document_text = (
            f"For {row['selected_occupation']}, "
            f"O*NET lists the following occupational task: "
            f"{task_text}. "
            f"O*NET occupation: {row['onet_title']} "
            f"({row['onet_soc_code']})."
        )

        rag_documents.append(
            {
                "selected_occupation":
                    row["selected_occupation"],

                "source_type":
                    "O*NET Task",

                "source_name":
                    "O*NET",

                "source_reference":
                    row["onet_soc_code"],

                "document_text":
                    document_text
            }
        )


# ------------------------------------------------------------
# 4. CANADIAN LABOUR-DEMAND DOCUMENTS
# ------------------------------------------------------------

for _, row in rag_demand.iterrows():

    document_text = (
        f"For {row['selected_occupation']} in the Canadian "
        f"labour-market dataset, there are "
        f"{int(row['posting_count'])} job postings and "
        f"{int(row['total_vacancies'])} total vacancies "
        f"across {int(row['province_count'])} provinces or "
        f"territories represented in the dataset. "
        f"The project-derived Canadian labour-demand score is "
        f"{row['demand_percentage']:.2f}%."
    )

    rag_documents.append(
        {
            "selected_occupation":
                row["selected_occupation"],

            "source_type":
                "Canadian Labour Demand",

            "source_name":
                "Government of Canada Job Bank",

            "source_reference":
                str(row["noc21_code"]),

            "document_text":
                document_text
        }
    )


# ------------------------------------------------------------
# 5. CIP EDUCATION-PATHWAY DOCUMENTS
# ------------------------------------------------------------

for _, row in rag_cip.iterrows():

    if bool(row["cip_pathway_available"]):

        document_text = (
            f"For {row['selected_occupation']}, "
            f"the project crosswalk identifies "
            f"{int(row['cip_program_count'])} mapped CIP "
            f"education program pathways. "
            f"Status: {row['cip_status']}."
        )

    else:

        document_text = (
            f"For {row['selected_occupation']}, "
            f"no mapped CIP education pathway is available "
            f"in the current project crosswalk. "
            f"Status: {row['cip_status']}."
        )

    rag_documents.append(
        {
            "selected_occupation":
                row["selected_occupation"],

            "source_type":
                "CIP Education Pathway",

            "source_name":
                "CIP / Project Crosswalk",

            "source_reference":
                "CIP2020",

            "document_text":
                document_text
        }
    )


# ------------------------------------------------------------
# 6. INTEGRATED OCCUPATION DOCUMENTS
# ------------------------------------------------------------

for _, row in rag_integrated.iterrows():

    document_text = (
        f"Integrated profile for "
        f"{row['selected_occupation']}: "
        f"O*NET occupation {row['onet_title']} "
        f"({row['onet_soc_code']}); "
        f"Job Zone {row['job_zone']}; "
        f"{row['education_record_count']} O*NET education "
        f"records; "
        f"{row['training_record_count']} training records; "
        f"{row['cip_program_count']} mapped CIP programs; "
        f"and {row['cip_soc_mapping_count']} CIP-SOC "
        f"mapping records."
    )

    rag_documents.append(
        {
            "selected_occupation":
                row["selected_occupation"],

            "source_type":
                "Integrated Occupation Profile",

            "source_name":
                "Integrated O*NET-CIP Profile",

            "source_reference":
                row["onet_soc_code"],

            "document_text":
                document_text
        }
    )


# ------------------------------------------------------------
# CONVERT TO DATAFRAME
# ------------------------------------------------------------

rag_documents = pd.DataFrame(rag_documents)

rag_documents.insert(
    0,
    "document_id",
    [
        f"RAG_DOC_{i:04d}"
        for i in range(1, len(rag_documents) + 1)
    ]
)


# ------------------------------------------------------------
# BASIC CLEANING
# ------------------------------------------------------------

rag_documents["document_text"] = (
    rag_documents["document_text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("\nRAG documents created:", len(rag_documents))

print(
    "Unique occupations:",
    rag_documents["selected_occupation"].nunique()
)

print(
    "Unique document IDs:",
    rag_documents["document_id"].nunique()
)

print("\nDocuments by source type:")

display(
    rag_documents[
        "source_type"
    ]
    .value_counts()
    .rename_axis("source_type")
    .reset_index(name="document_count")
)

print("\nSample RAG documents:")

display(
    rag_documents[
        [
            "document_id",
            "selected_occupation",
            "source_type",
            "source_name",
            "source_reference",
            "document_text"
        ]
    ].head(15)
)

Creating standardized RAG retrieval documents...

RAG documents created: 707
Unique occupations: 15
Unique document IDs: 707

Documents by source type:


,source_type,document_count
0,O*NET Task,487
1,O*NET Skill,150
2,Occupation Profile,20
3,Integrated Occupation Profile,20
4,Canadian Labour Demand,15
5,CIP Education Pathway,15



Sample RAG documents:


,document_id,selected_occupation,source_type,source_name,source_reference,document_text
0,RAG_DOC_0001,Software Developer,Occupation Profile,O*NET,15-1252.00,"Occupation: Software Developer. O*NET occupation: Software Developers (15-1252.00). Job Zone: 4. The O*NET profile contains 10 skill records, 33 knowledge records, 52 ability records, 17 task reco..."
1,RAG_DOC_0002,Information Technology (IT) Analyst,Occupation Profile,O*NET,15-1211.00,"Occupation: Information Technology (IT) Analyst. O*NET occupation: Computer Systems Analysts (15-1211.00). Job Zone: 4. The O*NET profile contains 10 skill records, 33 knowledge records, 52 abilit..."
2,RAG_DOC_0003,Information Technology (IT) Analyst,Occupation Profile,O*NET,15-1211.01,"Occupation: Information Technology (IT) Analyst. O*NET occupation: Health Informatics Specialists (15-1211.01). Job Zone: 5. The O*NET profile contains 10 skill records, 33 knowledge records, 52 a..."
3,RAG_DOC_0004,Information Technology (IT) Analyst,Occupation Profile,O*NET,15-1212.00,"Occupation: Information Technology (IT) Analyst. O*NET occupation: Information Security Analysts (15-1212.00). Job Zone: 4. The O*NET profile contains 10 skill records, 33 knowledge records, 52 ab..."
4,RAG_DOC_0005,Information Technology (IT) Analyst,Occupation Profile,O*NET,15-1253.00,"Occupation: Information Technology (IT) Analyst. O*NET occupation: Software Quality Assurance Analysts and Testers (15-1253.00). Job Zone: 4. The O*NET profile contains 10 skill records, 33 knowle..."
5,RAG_DOC_0006,Administrative Assistant,Occupation Profile,O*NET,43-6014.00,"Occupation: Administrative Assistant. O*NET occupation: Secretaries and Administrative Assistants, Except Legal, Medical, and Executive (43-6014.00). Job Zone: 2. The O*NET profile contains 10 ski..."
6,RAG_DOC_0007,Bookkeeper,Occupation Profile,O*NET,43-3031.00,"Occupation: Bookkeeper. O*NET occupation: Bookkeeping, Accounting, and Auditing Clerks (43-3031.00). Job Zone: 3. The O*NET profile contains 10 skill records, 33 knowledge records, 52 ability reco..."
7,RAG_DOC_0008,Office Administrator,Occupation Profile,O*NET,43-1011.00,"Occupation: Office Administrator. O*NET occupation: First-Line Supervisors of Office and Administrative Support Workers (43-1011.00). Job Zone: 3. The O*NET profile contains 10 skill records, 33 k..."
8,RAG_DOC_0009,Office Administrator,Occupation Profile,O*NET,43-6011.00,"Occupation: Office Administrator. O*NET occupation: Executive Secretaries and Executive Administrative Assistants (43-6011.00). Job Zone: 3. The O*NET profile contains 10 skill records, 33 knowled..."
9,RAG_DOC_0010,Office Manager,Occupation Profile,O*NET,43-1011.00,"Occupation: Office Manager. O*NET occupation: First-Line Supervisors of Office and Administrative Support Workers (43-1011.00). Job Zone: 3. The O*NET profile contains 10 skill records, 33 knowled..."


In [115]:
# ============================================================
# CELL 48 — Validate RAG Knowledge Base
# ============================================================

print("Running RAG knowledge-base validation...\n")

# ------------------------------------------------------------
# REQUIRED COLUMNS
# ------------------------------------------------------------

required_rag_columns = [
    "document_id",
    "selected_occupation",
    "source_type",
    "source_name",
    "source_reference",
    "document_text"
]

missing_rag_columns = [
    col
    for col in required_rag_columns
    if col not in rag_documents.columns
]


# ------------------------------------------------------------
# DUPLICATE DOCUMENT IDs
# ------------------------------------------------------------

duplicate_document_ids = (
    rag_documents["document_id"]
    .duplicated()
    .sum()
)


# ------------------------------------------------------------
# EMPTY DOCUMENT TEXT
# ------------------------------------------------------------

empty_document_text = (
    rag_documents["document_text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)


# ------------------------------------------------------------
# MISSING OCCUPATION
# ------------------------------------------------------------

missing_occupation = (
    rag_documents["selected_occupation"]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# MISSING SOURCE TYPE
# ------------------------------------------------------------

missing_source_type = (
    rag_documents["source_type"]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# EXACT DUPLICATE DOCUMENT TEXT
# ------------------------------------------------------------

duplicate_document_text = (
    rag_documents[
        [
            "selected_occupation",
            "source_type",
            "document_text"
        ]
    ]
    .duplicated()
    .sum()
)


# ------------------------------------------------------------
# OCCUPATION COVERAGE
# ------------------------------------------------------------

expected_occupations = set(
    labour_demand["selected_occupation"]
    .dropna()
    .unique()
)

rag_occupations = set(
    rag_documents["selected_occupation"]
    .dropna()
    .unique()
)

missing_rag_occupations = (
    expected_occupations
    - rag_occupations
)

unexpected_rag_occupations = (
    rag_occupations
    - expected_occupations
)


# ------------------------------------------------------------
# SOURCE-TYPE COVERAGE
# ------------------------------------------------------------

expected_source_types = {
    "Occupation Profile",
    "O*NET Skill",
    "O*NET Task",
    "Canadian Labour Demand",
    "CIP Education Pathway",
    "Integrated Occupation Profile"
}

actual_source_types = set(
    rag_documents["source_type"]
    .dropna()
    .unique()
)

missing_source_types = (
    expected_source_types
    - actual_source_types
)


# ------------------------------------------------------------
# DOCUMENT COUNT VALIDATION
# ------------------------------------------------------------

expected_document_count = (
    len(rag_occupation)
    + len(rag_skills)
    + len(rag_tasks)
    + len(rag_demand)
    + len(rag_cip)
    + len(rag_integrated)
)

actual_document_count = len(
    rag_documents
)


# ------------------------------------------------------------
# VALIDATION OUTPUT
# ------------------------------------------------------------

print(
    "Missing required columns:",
    missing_rag_columns
)

print(
    "Duplicate document IDs:",
    duplicate_document_ids
)

print(
    "Empty document text:",
    empty_document_text
)

print(
    "Missing occupation values:",
    missing_occupation
)

print(
    "Missing source-type values:",
    missing_source_type
)

print(
    "Exact duplicate retrieval documents:",
    duplicate_document_text
)

print(
    "Missing occupations:",
    missing_rag_occupations
)

print(
    "Unexpected occupations:",
    unexpected_rag_occupations
)

print(
    "Missing source types:",
    missing_source_types
)

print(
    "Expected document count:",
    expected_document_count
)

print(
    "Actual document count:",
    actual_document_count
)


# ------------------------------------------------------------
# FINAL PASS / FAIL
# ------------------------------------------------------------

rag_validation_pass = (
    len(missing_rag_columns) == 0
    and duplicate_document_ids == 0
    and empty_document_text == 0
    and missing_occupation == 0
    and missing_source_type == 0
    and duplicate_document_text == 0
    and len(missing_rag_occupations) == 0
    and len(unexpected_rag_occupations) == 0
    and len(missing_source_types) == 0
    and expected_document_count == actual_document_count
)

if rag_validation_pass:
    print(
        "\n✅ PASS — RAG knowledge-base validation successful."
    )
else:
    print(
        "\n❌ FAIL — Review the validation issues above."
    )


# ------------------------------------------------------------
# OCCUPATION-LEVEL COVERAGE TABLE
# ------------------------------------------------------------

rag_occupation_coverage = (
    rag_documents
    .groupby(
        [
            "selected_occupation",
            "source_type"
        ]
    )
    .size()
    .reset_index(
        name="document_count"
    )
    .pivot(
        index="selected_occupation",
        columns="source_type",
        values="document_count"
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

print(
    "\nRAG occupation/source coverage:"
)

display(
    rag_occupation_coverage
)

Running RAG knowledge-base validation...

Missing required columns: []
Duplicate document IDs: 0
Empty document text: 0
Missing occupation values: 0
Missing source-type values: 0
Exact duplicate retrieval documents: 0
Missing occupations: set()
Unexpected occupations: set()
Missing source types: set()
Expected document count: 707
Actual document count: 707

✅ PASS — RAG knowledge-base validation successful.

RAG occupation/source coverage:


source_type,selected_occupation,CIP Education Pathway,Canadian Labour Demand,Integrated Occupation Profile,O*NET Skill,O*NET Task,Occupation Profile
0,Administrative Assistant,1,1,1,10,31,1
1,Bookkeeper,1,1,1,10,28,1
2,Continuing Care Assistant,1,1,2,10,55,2
3,Delivery Driver,1,1,1,10,13,1
4,"Driver, Truck",1,1,1,10,29,1
5,Food Service Supervisor,1,1,1,10,26,1
6,Information Technology (IT) Analyst,1,1,4,10,80,4
7,Inside Sales Representative,1,1,1,10,24,1
8,Licensed Practical Nurse (L.P.N.),1,1,1,10,22,1
9,Office Administrator,1,1,2,10,50,2


In [117]:
# ============================================================
# CELL 49 — Create Semantic Embeddings for RAG Documents
# ============================================================

print("Preparing semantic embeddings for RAG documents...")

# ------------------------------------------------------------
# IMPORT SENTENCE TRANSFORMER
# ------------------------------------------------------------

try:
    from sentence_transformers import SentenceTransformer

    print("✓ sentence-transformers imported successfully.")

except ImportError:
    print(
        "❌ sentence-transformers is not installed.\n"
        "Run the following command in a new notebook cell:\n\n"
        "%pip install sentence-transformers"
    )
    raise


# ------------------------------------------------------------
# LOAD EMBEDDING MODEL
# ------------------------------------------------------------

embedding_model_name = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    "\nLoading embedding model:",
    embedding_model_name
)

embedding_model = SentenceTransformer(
    embedding_model_name
)

print(
    "✓ Embedding model loaded successfully."
)


# ------------------------------------------------------------
# PREPARE DOCUMENT TEXT
# ------------------------------------------------------------

rag_texts = (
    rag_documents["document_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

print(
    "\nDocuments to embed:",
    len(rag_texts)
)


# ------------------------------------------------------------
# CREATE EMBEDDINGS
# ------------------------------------------------------------

rag_embeddings = embedding_model.encode(
    rag_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


# ------------------------------------------------------------
# VALIDATE EMBEDDING MATRIX
# ------------------------------------------------------------

print(
    "\nEmbedding matrix created successfully."
)

print(
    "Embedding matrix shape:",
    rag_embeddings.shape
)

print(
    "Embedding data type:",
    rag_embeddings.dtype
)

print(
    "Number of RAG documents:",
    len(rag_documents)
)

print(
    "Number of embedding rows:",
    rag_embeddings.shape[0]
)

print(
    "Embedding dimensions:",
    rag_embeddings.shape[1]
)


# ------------------------------------------------------------
# BASIC EMBEDDING QUALITY CHECKS
# ------------------------------------------------------------

embedding_nan_count = int(
    np.isnan(rag_embeddings).sum()
)

embedding_inf_count = int(
    np.isinf(rag_embeddings).sum()
)

zero_embedding_rows = int(
    (
        np.linalg.norm(
            rag_embeddings,
            axis=1
        ) == 0
    ).sum()
)

document_embedding_match = (
    len(rag_documents)
    == rag_embeddings.shape[0]
)


print(
    "\nNaN embedding values:",
    embedding_nan_count
)

print(
    "Infinite embedding values:",
    embedding_inf_count
)

print(
    "Zero-vector embeddings:",
    zero_embedding_rows
)

print(
    "Document/embedding row match:",
    document_embedding_match
)


# ------------------------------------------------------------
# CHECK NORMALIZATION
# ------------------------------------------------------------

embedding_norms = np.linalg.norm(
    rag_embeddings,
    axis=1
)

print(
    "Minimum embedding norm:",
    round(float(embedding_norms.min()), 6)
)

print(
    "Maximum embedding norm:",
    round(float(embedding_norms.max()), 6)
)

print(
    "Average embedding norm:",
    round(float(embedding_norms.mean()), 6)
)


# ------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------

embedding_validation_pass = (
    document_embedding_match
    and embedding_nan_count == 0
    and embedding_inf_count == 0
    and zero_embedding_rows == 0
    and rag_embeddings.shape[1] > 0
)

if embedding_validation_pass:

    print(
        "\n✅ PASS — RAG semantic embeddings "
        "created successfully."
    )

else:

    print(
        "\n❌ FAIL — Review embedding validation "
        "results before continuing."
    )

Preparing semantic embeddings for RAG documents...


✓ sentence-transformers imported successfully.

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Embedding model loaded successfully.

Documents to embed: 707


Batches:   0%|          | 0/23 [00:00<?, ?it/s]


Embedding matrix created successfully.
Embedding matrix shape: (707, 384)
Embedding data type: float32
Number of RAG documents: 707
Number of embedding rows: 707
Embedding dimensions: 384

NaN embedding values: 0
Infinite embedding values: 0
Zero-vector embeddings: 0
Document/embedding row match: True
Minimum embedding norm: 1.0
Maximum embedding norm: 1.0
Average embedding norm: 1.0

✅ PASS — RAG semantic embeddings created successfully.


In [118]:
%pip install faiss-cpu

   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---------------- ----------------------- 6.6/16.3 MB 36.6 MB/s eta 0:00:01
   ----------------------------------- ---- 14.4/16.3 MB 37.7 MB/s eta 0:00:01
   ---------------------------------------- 16.3/16.3 MB 35.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [121]:
# ============================================================
# CELL 50 — Build and Validate FAISS Semantic Retrieval Index
# ============================================================

print("Building FAISS semantic retrieval index...")

# ------------------------------------------------------------
# IMPORT FAISS
# ------------------------------------------------------------

try:
    import faiss

    print("✓ FAISS imported successfully.")

except ImportError:
    print(
        "❌ FAISS is not installed.\n\n"
        "Run the following command in a new notebook cell:\n\n"
        "%pip install faiss-cpu"
    )
    raise


# ------------------------------------------------------------
# ENSURE FLOAT32 FORMAT
# ------------------------------------------------------------

rag_embeddings_faiss = np.asarray(
    rag_embeddings,
    dtype="float32"
)

embedding_dimension = (
    rag_embeddings_faiss.shape[1]
)

print(
    "\nEmbedding dimension:",
    embedding_dimension
)


# ------------------------------------------------------------
# CREATE FAISS INDEX
# ------------------------------------------------------------
# Because embeddings were normalized in Cell 49,
# inner-product similarity is equivalent to cosine similarity.

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

print(
    "FAISS index type:",
    type(faiss_index).__name__
)


# ------------------------------------------------------------
# ADD EMBEDDINGS TO INDEX
# ------------------------------------------------------------

faiss_index.add(
    rag_embeddings_faiss
)

print(
    "Documents added to FAISS index:",
    faiss_index.ntotal
)


# ------------------------------------------------------------
# INDEX VALIDATION
# ------------------------------------------------------------

expected_index_size = len(
    rag_documents
)

actual_index_size = (
    faiss_index.ntotal
)

index_size_match = (
    expected_index_size
    == actual_index_size
)

dimension_match = (
    faiss_index.d
    == rag_embeddings_faiss.shape[1]
)

print(
    "\nExpected index size:",
    expected_index_size
)

print(
    "Actual index size:",
    actual_index_size
)

print(
    "Index/document count match:",
    index_size_match
)

print(
    "FAISS dimension:",
    faiss_index.d
)

print(
    "Embedding dimension:",
    rag_embeddings_faiss.shape[1]
)

print(
    "Dimension match:",
    dimension_match
)


# ------------------------------------------------------------
# SIMPLE SELF-RETRIEVAL TEST
# ------------------------------------------------------------
# A document embedding should retrieve itself as the
# highest-similarity result.

test_document_position = 0

test_embedding = (
    rag_embeddings_faiss[
        test_document_position:
        test_document_position + 1
    ]
)

test_scores, test_indices = (
    faiss_index.search(
        test_embedding,
        5
    )
)

print(
    "\nSelf-retrieval test:"
)

print(
    "Query document ID:",
    rag_documents.iloc[
        test_document_position
    ]["document_id"]
)

print(
    "Top retrieved position:",
    int(test_indices[0][0])
)

print(
    "Expected position:",
    test_document_position
)

print(
    "Top similarity score:",
    round(
        float(test_scores[0][0]),
        6
    )
)

self_retrieval_correct = (
    int(test_indices[0][0])
    == test_document_position
)

print(
    "Self-retrieval correct:",
    self_retrieval_correct
)


# ------------------------------------------------------------
# DISPLAY TOP-5 SELF-RETRIEVAL RESULTS
# ------------------------------------------------------------

self_test_results = []

for rank, (idx, score) in enumerate(
    zip(
        test_indices[0],
        test_scores[0]
    ),
    start=1
):

    retrieved_row = (
        rag_documents.iloc[int(idx)]
    )

    self_test_results.append(
        {
            "rank": rank,
            "document_id":
                retrieved_row["document_id"],
            "selected_occupation":
                retrieved_row[
                    "selected_occupation"
                ],
            "source_type":
                retrieved_row["source_type"],
            "similarity_score":
                round(float(score), 6),
            "document_text":
                retrieved_row["document_text"]
        }
    )

self_test_results = pd.DataFrame(
    self_test_results
)

print(
    "\nTop-5 self-retrieval results:"
)

display(
    self_test_results
)


# ------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------

faiss_validation_pass = (
    index_size_match
    and dimension_match
    and self_retrieval_correct
    and actual_index_size > 0
)

if faiss_validation_pass:

    print(
        "\n✅ PASS — FAISS semantic retrieval "
        "index created successfully."
    )

else:

    print(
        "\n❌ FAIL — Review FAISS validation "
        "results before continuing."
    )

Building FAISS semantic retrieval index...
✓ FAISS imported successfully.

Embedding dimension: 384
FAISS index type: IndexFlatIP
Documents added to FAISS index: 707

Expected index size: 707
Actual index size: 707
Index/document count match: True
FAISS dimension: 384
Embedding dimension: 384
Dimension match: True

Self-retrieval test:
Query document ID: RAG_DOC_0001
Top retrieved position: 0
Expected position: 0
Top similarity score: 1.0
Self-retrieval correct: True

Top-5 self-retrieval results:


,rank,document_id,selected_occupation,source_type,similarity_score,document_text
0,1,RAG_DOC_0001,Software Developer,Occupation Profile,1.000000,"Occupation: Software Developer. O*NET occupation: Software Developers (15-1252.00). Job Zone: 4. The O*NET profile contains 10 skill records, 33 knowledge records, 52 ability records, 17 task reco..."
1,2,RAG_DOC_0005,Information Technology (IT) Analyst,Occupation Profile,0.888949,"Occupation: Information Technology (IT) Analyst. O*NET occupation: Software Quality Assurance Analysts and Testers (15-1253.00). Job Zone: 4. The O*NET profile contains 10 skill records, 33 knowle..."
2,3,RAG_DOC_0014,Inside Sales Representative,Occupation Profile,0.878213,"Occupation: Inside Sales Representative. O*NET occupation: Retail Salespersons (41-2031.00). Job Zone: 2. The O*NET profile contains 10 skill records, 33 knowledge records, 52 ability records, 24 ..."
3,4,RAG_DOC_0002,Information Technology (IT) Analyst,Occupation Profile,0.870833,"Occupation: Information Technology (IT) Analyst. O*NET occupation: Computer Systems Analysts (15-1211.00). Job Zone: 4. The O*NET profile contains 10 skill records, 33 knowledge records, 52 abilit..."
4,5,RAG_DOC_0008,Office Administrator,Occupation Profile,0.862018,"Occupation: Office Administrator. O*NET occupation: First-Line Supervisors of Office and Administrative Support Workers (43-1011.00). Job Zone: 3. The O*NET profile contains 10 skill records, 33 k..."



✅ PASS — FAISS semantic retrieval index created successfully.


In [125]:
# ============================================================
# CELL 51 — Create Semantic RAG Retrieval Function
# ============================================================

print("Creating semantic retrieval function...")


def retrieve_rag_documents(
    query,
    top_k=5,
    occupation_filter=None,
    source_type_filter=None
):
    """
    Retrieve semantically relevant RAG documents.

    Parameters
    ----------
    query : str
        Natural-language retrieval query.

    top_k : int
        Number of documents to return.

    occupation_filter : str or None
        Optional occupation restriction.

    source_type_filter : str or list or None
        Optional source-type restriction.

    Returns
    -------
    pandas.DataFrame
        Ranked retrieved documents.
    """

    # --------------------------------------------------------
    # Encode query
    # --------------------------------------------------------

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    # --------------------------------------------------------
    # Determine search depth
    # --------------------------------------------------------
    # Search broadly first, then apply metadata filters.

    search_k = min(
        len(rag_documents),
        max(top_k * 20, 100)
    )

    scores, indices = faiss_index.search(
        query_embedding,
        search_k
    )

    # --------------------------------------------------------
    # Build results
    # --------------------------------------------------------

    results = []

    for idx, score in zip(
        indices[0],
        scores[0]
    ):

        if idx < 0:
            continue

        row = rag_documents.iloc[int(idx)]

        # ----------------------------------------------------
        # Occupation filter
        # ----------------------------------------------------

        if occupation_filter is not None:

            if (
                row["selected_occupation"]
                != occupation_filter
            ):
                continue

        # ----------------------------------------------------
        # Source-type filter
        # ----------------------------------------------------

        if source_type_filter is not None:

            if isinstance(
                source_type_filter,
                str
            ):
                allowed_sources = {
                    source_type_filter
                }

            else:
                allowed_sources = set(
                    source_type_filter
                )

            if (
                row["source_type"]
                not in allowed_sources
            ):
                continue

        # ----------------------------------------------------
        # Append result
        # ----------------------------------------------------

        results.append(
            {
                "document_id":
                    row["document_id"],

                "selected_occupation":
                    row["selected_occupation"],

                "source_type":
                    row["source_type"],

                "source_name":
                    row["source_name"],

                "source_reference":
                    row["source_reference"],

                "similarity_score":
                    round(
                        float(score),
                        6
                    ),

                "document_text":
                    row["document_text"]
            }
        )

        if len(results) >= top_k:
            break

    results_df = pd.DataFrame(
        results
    )

    if not results_df.empty:

        results_df.insert(
            0,
            "retrieval_rank",
            range(
                1,
                len(results_df) + 1
            )
        )

    return results_df


print(
    "✓ Semantic retrieval function created successfully."
)


# ============================================================
# TEST 1 — SOFTWARE DEVELOPER SKILLS
# ============================================================

test_query_1 = (
    "What skills are important for a Software Developer?"
)

test_results_1 = retrieve_rag_documents(
    query=test_query_1,
    top_k=5,
    occupation_filter="Software Developer",
    source_type_filter="O*NET Skill"
)

print(
    "\nTEST 1 — Software Developer skills"
)

print(
    "Query:",
    test_query_1
)

display(
    test_results_1
)


# ============================================================
# TEST 2 — CANADIAN LABOUR DEMAND
# ============================================================

test_query_2 = (
    "What is the Canadian labour demand "
    "for Driver, Truck?"
)

test_results_2 = retrieve_rag_documents(
    query=test_query_2,
    top_k=3,
    occupation_filter="Driver, Truck",
    source_type_filter="Canadian Labour Demand"
)

print(
    "\nTEST 2 — Driver, Truck labour demand"
)

print(
    "Query:",
    test_query_2
)

display(
    test_results_2
)


# ============================================================
# TEST 3 — EDUCATION PATHWAY
# ============================================================

test_query_3 = (
    "What education pathway is available "
    "for Software Developer?"
)

test_results_3 = retrieve_rag_documents(
    query=test_query_3,
    top_k=3,
    occupation_filter="Software Developer",
    source_type_filter=[
        "CIP Education Pathway",
        "Integrated Occupation Profile"
    ]
)

print(
    "\nTEST 3 — Software Developer education pathway"
)

print(
    "Query:",
    test_query_3
)

display(
    test_results_3
)


# ============================================================
# BASIC RETRIEVAL VALIDATION
# ============================================================

retrieval_test_1_pass = (
    len(test_results_1) > 0
    and
    (
        test_results_1[
            "selected_occupation"
        ]
        == "Software Developer"
    ).all()
    and
    (
        test_results_1[
            "source_type"
        ]
        == "O*NET Skill"
    ).all()
)

retrieval_test_2_pass = (
    len(test_results_2) > 0
    and
    (
        test_results_2[
            "selected_occupation"
        ]
        == "Driver, Truck"
    ).all()
    and
    (
        test_results_2[
            "source_type"
        ]
        == "Canadian Labour Demand"
    ).all()
)

retrieval_test_3_pass = (
    len(test_results_3) > 0
    and
    (
        test_results_3[
            "selected_occupation"
        ]
        == "Software Developer"
    ).all()
    and
    test_results_3[
        "source_type"
    ].isin(
        [
            "CIP Education Pathway",
            "Integrated Occupation Profile"
        ]
    ).all()
)

print(
    "\nRetrieval validation results:"
)

print(
    "Software Developer skill retrieval:",
    retrieval_test_1_pass
)

print(
    "Driver, Truck demand retrieval:",
    retrieval_test_2_pass
)

print(
    "Software Developer education retrieval:",
    retrieval_test_3_pass
)

rag_retrieval_validation_pass = (
    retrieval_test_1_pass
    and retrieval_test_2_pass
    and retrieval_test_3_pass
)

if rag_retrieval_validation_pass:

    print(
        "\n✅ PASS — Semantic RAG retrieval "
        "function validated successfully."
    )

else:

    print(
        "\n❌ FAIL — Review retrieved documents "
        "before continuing."
    )

Creating semantic retrieval function...
✓ Semantic retrieval function created successfully.

TEST 1 — Software Developer skills
Query: What skills are important for a Software Developer?


,retrieval_rank,document_id,selected_occupation,source_type,source_name,source_reference,similarity_score,document_text
0,1,RAG_DOC_0164,Software Developer,O*NET Skill,O*NET,15-1252.00,0.587850,"For Software Developer, O*NET identifies Learning Strategies as a relevant skill. The skill importance score is 2.62 and the required level is 3.12. O*NET occupation: Software Developers (15-1252...."
1,2,RAG_DOC_0163,Software Developer,O*NET Skill,O*NET,15-1252.00,0.566569,"For Software Developer, O*NET identifies Critical Thinking as a relevant skill. The skill importance score is 3.88 and the required level is 4.12. O*NET occupation: Software Developers (15-1252.00)."
2,3,RAG_DOC_0168,Software Developer,O*NET Skill,O*NET,15-1252.00,0.564259,"For Software Developer, O*NET identifies Science as a relevant skill. The skill importance score is 2.12 and the required level is 1.88. O*NET occupation: Software Developers (15-1252.00)."
3,4,RAG_DOC_0169,Software Developer,O*NET Skill,O*NET,15-1252.00,0.563771,"For Software Developer, O*NET identifies Speaking as a relevant skill. The skill importance score is 3.12 and the required level is 3.62. O*NET occupation: Software Developers (15-1252.00)."
4,5,RAG_DOC_0165,Software Developer,O*NET Skill,O*NET,15-1252.00,0.559180,"For Software Developer, O*NET identifies Mathematics as a relevant skill. The skill importance score is 2.75 and the required level is 3.25. O*NET occupation: Software Developers (15-1252.00)."



TEST 2 — Driver, Truck labour demand
Query: What is the Canadian labour demand for Driver, Truck?


,retrieval_rank,document_id,selected_occupation,source_type,source_name,source_reference,similarity_score,document_text
0,1,RAG_DOC_0658,"Driver, Truck",Canadian Labour Demand,Government of Canada Job Bank,73300,0.756284,"For Driver, Truck in the Canadian labour-market dataset, there are 1423 job postings and 3705 total vacancies across 12 provinces or territories represented in the dataset. The project-derived Can..."



TEST 3 — Software Developer education pathway
Query: What education pathway is available for Software Developer?


,retrieval_rank,document_id,selected_occupation,source_type,source_name,source_reference,similarity_score,document_text
0,1,RAG_DOC_0673,Software Developer,CIP Education Pathway,CIP / Project Crosswalk,CIP2020,0.591327,"For Software Developer, the project crosswalk identifies 20 mapped CIP education program pathways. Status: CIP pathway available."
1,2,RAG_DOC_0707,Software Developer,Integrated Occupation Profile,Integrated O*NET-CIP Profile,15-1252.00,0.497098,Integrated profile for Software Developer: O*NET occupation Software Developers (15-1252.00); Job Zone 4; 12 O*NET education records; 29 training records; 20 mapped CIP programs; and 20 CIP-SOC ma...



Retrieval validation results:
Software Developer skill retrieval: True
Driver, Truck demand retrieval: True
Software Developer education retrieval: True

✅ PASS — Semantic RAG retrieval function validated successfully.


In [127]:
# ============================================================
# CELL 52 — Build Candidate-Level RAG Evidence Packages
# ============================================================

print("Building candidate-level RAG evidence packages...")


def build_candidate_rag_evidence(
    candidate_id,
    top_k_skills=5,
    top_k_tasks=3
):
    """
    Build a grounded evidence package for one candidate's
    Week 6 Top-1 recommended occupation.
    """

    # --------------------------------------------------------
    # 1. FIND CANDIDATE TOP-1 RECOMMENDATION
    # --------------------------------------------------------

    candidate_top1 = (
        top1_recommendation[
            top1_recommendation["candidate_id"]
            == candidate_id
        ]
        .copy()
    )

    if candidate_top1.empty:
        return None

    top1_row = candidate_top1.iloc[0]

    occupation = (
        top1_row["recommended_occupation"]
    )

    hybrid_score = float(
        top1_row["hybrid_recommendation_score"]
    )

    recommendation_explanation = (
        top1_row["recommendation_explanation"]
    )


    # --------------------------------------------------------
    # 2. FIND PRESCRIPTIVE SKILL GAPS
    # --------------------------------------------------------

    candidate_gaps = (
        recommended_occupation_prescriptive[
            (
                recommended_occupation_prescriptive[
                    "candidate_id"
                ]
                == candidate_id
            )
            &
            (
                recommended_occupation_prescriptive[
                    "selected_occupation"
                ]
                == occupation
            )
        ]
        .sort_values(
            "prescriptive_priority_score",
            ascending=False
        )
        .copy()
    )


    if not candidate_gaps.empty:

        missing_skills = (
            candidate_gaps[
                [
                    "skill",
                    "importance",
                    "level",
                    "prescriptive_priority_percentage",
                    "priority_level"
                ]
            ]
            .head(3)
            .to_dict(
                orient="records"
            )
        )

        demand_percentage = float(
            candidate_gaps.iloc[0][
                "demand_percentage"
            ]
        )

        demand_level = (
            candidate_gaps.iloc[0][
                "demand_level"
            ]
        )

        cip_status = (
            candidate_gaps.iloc[0][
                "cip_status"
            ]
        )

        education_action = (
            candidate_gaps.iloc[0][
                "education_action"
            ]
        )

    else:

        missing_skills = []

        demand_row = (
            labour_demand[
                labour_demand[
                    "selected_occupation"
                ]
                == occupation
            ]
        )

        if not demand_row.empty:

            demand_percentage = float(
                demand_row.iloc[0][
                    "demand_percentage"
                ]
            )

        else:
            demand_percentage = np.nan


        demand_level = (
            "High"
            if demand_percentage >= 70
            else "Moderate"
            if demand_percentage >= 40
            else "Lower"
        )

        cip_row = (
            rag_cip[
                rag_cip[
                    "selected_occupation"
                ]
                == occupation
            ]
        )

        if not cip_row.empty:

            cip_status = (
                cip_row.iloc[0][
                    "cip_status"
                ]
            )

            if bool(
                cip_row.iloc[0][
                    "cip_pathway_available"
                ]
            ):
                education_action = (
                    "Review mapped CIP "
                    "education pathways."
                )
            else:
                education_action = (
                    "Investigate relevant "
                    "education or training "
                    "pathways separately."
                )

        else:

            cip_status = (
                "No mapped CIP pathway available"
            )

            education_action = (
                "Investigate relevant education "
                "or training pathways separately."
            )


    # --------------------------------------------------------
    # 3. RETRIEVE O*NET SKILL EVIDENCE
    # --------------------------------------------------------

    skill_query = (
        f"What skills are important for "
        f"{occupation}?"
    )

    skill_evidence = retrieve_rag_documents(
        query=skill_query,
        top_k=top_k_skills,
        occupation_filter=occupation,
        source_type_filter="O*NET Skill"
    )


    # --------------------------------------------------------
    # 4. RETRIEVE TASK EVIDENCE
    # --------------------------------------------------------

    task_query = (
        f"What work tasks are performed by "
        f"{occupation}?"
    )

    task_evidence = retrieve_rag_documents(
        query=task_query,
        top_k=top_k_tasks,
        occupation_filter=occupation,
        source_type_filter="O*NET Task"
    )


    # --------------------------------------------------------
    # 5. RETRIEVE CANADIAN DEMAND EVIDENCE
    # --------------------------------------------------------

    demand_query = (
        f"What is the Canadian labour demand "
        f"for {occupation}?"
    )

    demand_evidence = retrieve_rag_documents(
        query=demand_query,
        top_k=1,
        occupation_filter=occupation,
        source_type_filter="Canadian Labour Demand"
    )


    # --------------------------------------------------------
    # 6. RETRIEVE EDUCATION EVIDENCE
    # --------------------------------------------------------

    education_query = (
        f"What education pathway is available "
        f"for {occupation}?"
    )

    education_evidence = retrieve_rag_documents(
        query=education_query,
        top_k=3,
        occupation_filter=occupation,
        source_type_filter=[
            "CIP Education Pathway",
            "Integrated Occupation Profile"
        ]
    )


    # --------------------------------------------------------
    # 7. CREATE EVIDENCE PACKAGE
    # --------------------------------------------------------

    evidence_package = {
        "candidate_id":
            candidate_id,

        "recommended_occupation":
            occupation,

        "hybrid_recommendation_score":
            round(hybrid_score, 4),

        "recommendation_explanation":
            recommendation_explanation,

        "missing_skills":
            missing_skills,

        "demand_percentage":
            round(
                demand_percentage,
                2
            )
            if not pd.isna(
                demand_percentage
            )
            else np.nan,

        "demand_level":
            demand_level,

        "cip_status":
            cip_status,

        "education_action":
            education_action,

        "skill_evidence":
            skill_evidence.to_dict(
                orient="records"
            ),

        "task_evidence":
            task_evidence.to_dict(
                orient="records"
            ),

        "demand_evidence":
            demand_evidence.to_dict(
                orient="records"
            ),

        "education_evidence":
            education_evidence.to_dict(
                orient="records"
            )
    }

    return evidence_package


# ============================================================
# TEST WITH CANDIDATE_001
# ============================================================

test_candidate_id = "Candidate_001"

candidate_001_rag_evidence = (
    build_candidate_rag_evidence(
        candidate_id=test_candidate_id
    )
)

print(
    "\nCandidate:",
    candidate_001_rag_evidence[
        "candidate_id"
    ]
)

print(
    "Recommended occupation:",
    candidate_001_rag_evidence[
        "recommended_occupation"
    ]
)

print(
    "Hybrid recommendation score:",
    candidate_001_rag_evidence[
        "hybrid_recommendation_score"
    ]
)

print(
    "Canadian demand:",
    candidate_001_rag_evidence[
        "demand_percentage"
    ],
    "%"
)

print(
    "Demand level:",
    candidate_001_rag_evidence[
        "demand_level"
    ]
)

print(
    "CIP status:",
    candidate_001_rag_evidence[
        "cip_status"
    ]
)

print(
    "\nTop missing skills:"
)

display(
    pd.DataFrame(
        candidate_001_rag_evidence[
            "missing_skills"
        ]
    )
)


print(
    "\nRetrieved O*NET skill evidence:"
)

display(
    pd.DataFrame(
        candidate_001_rag_evidence[
            "skill_evidence"
        ]
    )
)


print(
    "\nRetrieved O*NET task evidence:"
)

display(
    pd.DataFrame(
        candidate_001_rag_evidence[
            "task_evidence"
        ]
    )
)


print(
    "\nRetrieved Canadian demand evidence:"
)

display(
    pd.DataFrame(
        candidate_001_rag_evidence[
            "demand_evidence"
        ]
    )
)


print(
    "\nRetrieved education evidence:"
)

display(
    pd.DataFrame(
        candidate_001_rag_evidence[
            "education_evidence"
        ]
    )
)

Building candidate-level RAG evidence packages...

Candidate: Candidate_001
Recommended occupation: Software Developer
Hybrid recommendation score: 69.0545
Canadian demand: 22.91 %
Demand level: Lower
CIP status: CIP pathway available

Top missing skills:


,skill,importance,level,prescriptive_priority_percentage,priority_level
0,Active Listening,3.38,3.88,52.72,Medium
1,Speaking,3.12,3.62,49.60,Lower
2,Monitoring,3.00,3.50,48.16,Lower



Retrieved O*NET skill evidence:


,retrieval_rank,document_id,selected_occupation,source_type,source_name,source_reference,similarity_score,document_text
0,1,RAG_DOC_0164,Software Developer,O*NET Skill,O*NET,15-1252.00,0.591747,"For Software Developer, O*NET identifies Learning Strategies as a relevant skill. The skill importance score is 2.62 and the required level is 3.12. O*NET occupation: Software Developers (15-1252...."
1,2,RAG_DOC_0163,Software Developer,O*NET Skill,O*NET,15-1252.00,0.571745,"For Software Developer, O*NET identifies Critical Thinking as a relevant skill. The skill importance score is 3.88 and the required level is 4.12. O*NET occupation: Software Developers (15-1252.00)."
2,3,RAG_DOC_0168,Software Developer,O*NET Skill,O*NET,15-1252.00,0.570703,"For Software Developer, O*NET identifies Science as a relevant skill. The skill importance score is 2.12 and the required level is 1.88. O*NET occupation: Software Developers (15-1252.00)."
3,4,RAG_DOC_0169,Software Developer,O*NET Skill,O*NET,15-1252.00,0.569779,"For Software Developer, O*NET identifies Speaking as a relevant skill. The skill importance score is 3.12 and the required level is 3.62. O*NET occupation: Software Developers (15-1252.00)."
4,5,RAG_DOC_0165,Software Developer,O*NET Skill,O*NET,15-1252.00,0.567506,"For Software Developer, O*NET identifies Mathematics as a relevant skill. The skill importance score is 2.75 and the required level is 3.25. O*NET occupation: Software Developers (15-1252.00)."



Retrieved O*NET task evidence:


,retrieval_rank,document_id,selected_occupation,source_type,source_name,source_reference,similarity_score,document_text
0,1,RAG_DOC_0256,Software Developer,O*NET Task,O*NET,15-1252.00,0.559902,"For Software Developer, O*NET lists the following occupational task: Design, develop and modify software systems, using scientific analysis and mathematical models to predict and measure outcomes ..."
1,2,RAG_DOC_0253,Software Developer,O*NET Task,O*NET,15-1252.00,0.559402,"For Software Developer, O*NET lists the following occupational task: Prepare reports or correspondence concerning project specifications, activities, or status.. O*NET occupation: Software Develop..."
2,3,RAG_DOC_0250,Software Developer,O*NET Task,O*NET,15-1252.00,0.549482,"For Software Developer, O*NET lists the following occupational task: Develop or direct software system testing or validation procedures, programming, or documentation.. O*NET occupation: Software ..."



Retrieved Canadian demand evidence:


,retrieval_rank,document_id,selected_occupation,source_type,source_name,source_reference,similarity_score,document_text
0,1,RAG_DOC_0672,Software Developer,Canadian Labour Demand,Government of Canada Job Bank,21232,0.774537,"For Software Developer in the Canadian labour-market dataset, there are 323 job postings and 344 total vacancies across 8 provinces or territories represented in the dataset. The project-derived C..."



Retrieved education evidence:


,retrieval_rank,document_id,selected_occupation,source_type,source_name,source_reference,similarity_score,document_text
0,1,RAG_DOC_0673,Software Developer,CIP Education Pathway,CIP / Project Crosswalk,CIP2020,0.591327,"For Software Developer, the project crosswalk identifies 20 mapped CIP education program pathways. Status: CIP pathway available."
1,2,RAG_DOC_0707,Software Developer,Integrated Occupation Profile,Integrated O*NET-CIP Profile,15-1252.00,0.497098,Integrated profile for Software Developer: O*NET occupation Software Developers (15-1252.00); Job Zone 4; 12 O*NET education records; 29 training records; 20 mapped CIP programs; and 20 CIP-SOC ma...


In [129]:
# ============================================================
# CELL 53 — Generate Grounded RAG Career Explanation
# ============================================================

print("Creating grounded RAG career explanation function...")


def generate_grounded_rag_explanation(
    evidence_package
):
    """
    Generate a deterministic, evidence-grounded
    career recommendation explanation.

    The explanation uses only information contained
    in the candidate evidence package and retrieved
    project knowledge base.
    """

    if evidence_package is None:
        return None

    candidate_id = (
        evidence_package["candidate_id"]
    )

    occupation = (
        evidence_package[
            "recommended_occupation"
        ]
    )

    hybrid_score = (
        evidence_package[
            "hybrid_recommendation_score"
        ]
    )

    recommendation_explanation = (
        evidence_package[
            "recommendation_explanation"
        ]
    )

    missing_skills = (
        evidence_package[
            "missing_skills"
        ]
    )

    demand_percentage = (
        evidence_package[
            "demand_percentage"
        ]
    )

    demand_level = (
        evidence_package[
            "demand_level"
        ]
    )

    cip_status = (
        evidence_package[
            "cip_status"
        ]
    )

    education_action = (
        evidence_package[
            "education_action"
        ]
    )


    # --------------------------------------------------------
    # RECOMMENDATION SECTION
    # --------------------------------------------------------

    explanation_parts = []

    explanation_parts.append(
        f"For {candidate_id}, the recommended occupation is "
        f"{occupation}, with a hybrid recommendation score "
        f"of {hybrid_score:.2f}. "
        f"The Week 6 recommendation assessment classified "
        f"this match as: {recommendation_explanation}."
    )


    # --------------------------------------------------------
    # SKILL-GAP SECTION
    # --------------------------------------------------------

    if len(missing_skills) > 0:

        skill_parts = []

        for skill_info in missing_skills:

            skill_parts.append(
                f"{skill_info['skill']} "
                f"({skill_info['prescriptive_priority_percentage']:.2f}% "
                f"priority; O*NET importance "
                f"{skill_info['importance']:.2f}; "
                f"required level "
                f"{skill_info['level']:.2f})"
            )

        explanation_parts.append(
            "The highest-priority identified skill gaps are "
            + "; ".join(skill_parts)
            + "."
        )

    else:

        explanation_parts.append(
            "No missing O*NET skills were identified for the "
            "candidate's Top-1 recommended occupation using "
            "the current project skill-matching method."
        )


    # --------------------------------------------------------
    # O*NET TASK EVIDENCE
    # --------------------------------------------------------

    task_evidence = (
        evidence_package[
            "task_evidence"
        ]
    )

    if len(task_evidence) > 0:

        task_texts = []

        for record in task_evidence[:3]:

            task_texts.append(
                record["document_text"]
            )

        explanation_parts.append(
            "Retrieved O*NET task evidence indicates that "
            + " ".join(task_texts)
        )


    # --------------------------------------------------------
    # CANADIAN LABOUR-DEMAND SECTION
    # --------------------------------------------------------

    demand_evidence = (
        evidence_package[
            "demand_evidence"
        ]
    )

    if len(demand_evidence) > 0:

        explanation_parts.append(
            f"Canadian labour-market evidence indicates a "
            f"project-derived demand score of "
            f"{demand_percentage:.2f}% for {occupation}, "
            f"which is categorized as {demand_level}. "
            f"{demand_evidence[0]['document_text']}"
        )

    else:

        explanation_parts.append(
            "No Canadian labour-demand evidence was retrieved "
            "for this occupation."
        )


    # --------------------------------------------------------
    # EDUCATION PATHWAY SECTION
    # --------------------------------------------------------

    education_evidence = (
        evidence_package[
            "education_evidence"
        ]
    )

    explanation_parts.append(
        f"Education pathway status: {cip_status}. "
        f"{education_action}"
    )

    if len(education_evidence) > 0:

        education_texts = []

        for record in education_evidence:

            education_texts.append(
                record["document_text"]
            )

        explanation_parts.append(
            "Retrieved education evidence: "
            + " ".join(education_texts)
        )


    # --------------------------------------------------------
    # SOURCE CITATIONS
    # --------------------------------------------------------

    source_records = []

    for evidence_key in [
        "skill_evidence",
        "task_evidence",
        "demand_evidence",
        "education_evidence"
    ]:

        for record in evidence_package[
            evidence_key
        ]:

            source_records.append(
                (
                    record["document_id"],
                    record["source_name"],
                    record["source_reference"]
                )
            )

    # Remove duplicate source records while preserving order
    source_records = list(
        dict.fromkeys(
            source_records
        )
    )

    source_text = "; ".join(
        [
            f"{doc_id} — {source_name} "
            f"[{source_reference}]"
            for (
                doc_id,
                source_name,
                source_reference
            ) in source_records
        ]
    )

    explanation_parts.append(
        "Supporting retrieved sources: "
        + source_text
        + "."
    )


    # --------------------------------------------------------
    # LIMITATION / RESPONSIBLE USE
    # --------------------------------------------------------

    explanation_parts.append(
        "This recommendation is decision-support only. "
        "The scores are project-derived analytical measures "
        "and should be interpreted together with candidate "
        "preferences, experience, current labour-market "
        "conditions, and human career guidance."
    )


    # --------------------------------------------------------
    # FINAL TEXT
    # --------------------------------------------------------

    final_explanation = "\n\n".join(
        explanation_parts
    )

    return final_explanation


# ============================================================
# TEST — CANDIDATE_001
# ============================================================

candidate_001_rag_explanation = (
    generate_grounded_rag_explanation(
        candidate_001_rag_evidence
    )
)

print(
    "\nGROUNDed RAG EXPLANATION — Candidate_001\n"
)

print(
    candidate_001_rag_explanation
)

Creating grounded RAG career explanation function...

GROUNDed RAG EXPLANATION — Candidate_001

For Candidate_001, the recommended occupation is Software Developer, with a hybrid recommendation score of 69.05. The Week 6 recommendation assessment classified this match as: Moderate recommendation based on combined candidate profile similarity and O*NET skill alignment..

The highest-priority identified skill gaps are Active Listening (52.72% priority; O*NET importance 3.38; required level 3.88); Speaking (49.60% priority; O*NET importance 3.12; required level 3.62); Monitoring (48.16% priority; O*NET importance 3.00; required level 3.50).

Retrieved O*NET task evidence indicates that For Software Developer, O*NET lists the following occupational task: Design, develop and modify software systems, using scientific analysis and mathematical models to predict and measure outcomes and consequences of design.. O*NET occupation: Software Developers (15-1252.00). For Software Developer, O*NET l

In [131]:
# ============================================================
# CELL 54 — Retrieve Exact O*NET Evidence for Skill Gaps
# ============================================================

print("Creating exact skill-gap evidence retrieval...")


def retrieve_exact_skill_gap_evidence(
    occupation,
    missing_skills
):
    """
    Retrieve the exact O*NET RAG document corresponding
    to each identified missing skill for an occupation.

    This strengthens grounding by linking every prescriptive
    skill gap directly to its supporting O*NET evidence.
    """

    exact_skill_evidence = []

    for skill_info in missing_skills:

        target_skill = (
            skill_info["skill"]
        )

        # ----------------------------------------------------
        # Match standardized RAG skill document
        # ----------------------------------------------------

        matches = rag_documents[
            (
                rag_documents[
                    "selected_occupation"
                ]
                == occupation
            )
            &
            (
                rag_documents[
                    "source_type"
                ]
                == "O*NET Skill"
            )
            &
            (
                rag_documents[
                    "document_text"
                ]
                .str.contains(
                    f"identifies {target_skill} as",
                    regex=False,
                    na=False
                )
            )
        ].copy()


        # ----------------------------------------------------
        # STORE MATCH
        # ----------------------------------------------------

        if not matches.empty:

            row = matches.iloc[0]

            exact_skill_evidence.append(
                {
                    "skill":
                        target_skill,

                    "importance":
                        skill_info[
                            "importance"
                        ],

                    "level":
                        skill_info[
                            "level"
                        ],

                    "prescriptive_priority_percentage":
                        skill_info[
                            "prescriptive_priority_percentage"
                        ],

                    "priority_level":
                        skill_info[
                            "priority_level"
                        ],

                    "document_id":
                        row[
                            "document_id"
                        ],

                    "source_name":
                        row[
                            "source_name"
                        ],

                    "source_reference":
                        row[
                            "source_reference"
                        ],

                    "document_text":
                        row[
                            "document_text"
                        ]
                }
            )

        else:

            exact_skill_evidence.append(
                {
                    "skill":
                        target_skill,

                    "importance":
                        skill_info[
                            "importance"
                        ],

                    "level":
                        skill_info[
                            "level"
                        ],

                    "prescriptive_priority_percentage":
                        skill_info[
                            "prescriptive_priority_percentage"
                        ],

                    "priority_level":
                        skill_info[
                            "priority_level"
                        ],

                    "document_id":
                        None,

                    "source_name":
                        None,

                    "source_reference":
                        None,

                    "document_text":
                        None
                }
            )

    return exact_skill_evidence


# ============================================================
# ADD EXACT EVIDENCE TO CANDIDATE_001 PACKAGE
# ============================================================

candidate_001_rag_evidence[
    "exact_skill_gap_evidence"
] = retrieve_exact_skill_gap_evidence(
    occupation=
        candidate_001_rag_evidence[
            "recommended_occupation"
        ],

    missing_skills=
        candidate_001_rag_evidence[
            "missing_skills"
        ]
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

exact_skill_gap_df = pd.DataFrame(
    candidate_001_rag_evidence[
        "exact_skill_gap_evidence"
    ]
)

print(
    "\nExact O*NET evidence for "
    "Candidate_001 skill gaps:"
)

display(
    exact_skill_gap_df
)


# ============================================================
# VALIDATION
# ============================================================

expected_gap_count = len(
    candidate_001_rag_evidence[
        "missing_skills"
    ]
)

actual_gap_evidence_count = len(
    exact_skill_gap_df
)

missing_exact_documents = int(
    exact_skill_gap_df[
        "document_id"
    ]
    .isna()
    .sum()
)

skill_names_match = (
    set(
        exact_skill_gap_df[
            "skill"
        ]
    )
    ==
    set(
        [
            x["skill"]
            for x in
            candidate_001_rag_evidence[
                "missing_skills"
            ]
        ]
    )
)

occupation_evidence_match = (
    exact_skill_gap_df[
        "document_text"
    ]
    .fillna("")
    .str.contains(
        candidate_001_rag_evidence[
            "recommended_occupation"
        ],
        regex=False
    )
    .all()
)


print(
    "\nExact skill-gap evidence validation:"
)

print(
    "Expected skill gaps:",
    expected_gap_count
)

print(
    "Retrieved exact skill documents:",
    actual_gap_evidence_count
)

print(
    "Missing exact O*NET documents:",
    missing_exact_documents
)

print(
    "Skill names match:",
    skill_names_match
)

print(
    "Occupation evidence match:",
    occupation_evidence_match
)


exact_skill_evidence_pass = (
    expected_gap_count
    == actual_gap_evidence_count
    and missing_exact_documents == 0
    and skill_names_match
    and occupation_evidence_match
)


if exact_skill_evidence_pass:

    print(
        "\n✅ PASS — Exact O*NET evidence "
        "retrieved for every skill gap."
    )

else:

    print(
        "\n❌ FAIL — Review missing or "
        "misaligned skill evidence."
    )

Creating exact skill-gap evidence retrieval...

Exact O*NET evidence for Candidate_001 skill gaps:


,skill,importance,level,prescriptive_priority_percentage,priority_level,document_id,source_name,source_reference,document_text
0,Active Listening,3.38,3.88,52.72,Medium,RAG_DOC_0162,O*NET,15-1252.00,"For Software Developer, O*NET identifies Active Listening as a relevant skill. The skill importance score is 3.38 and the required level is 3.88. O*NET occupation: Software Developers (15-1252.00)."
1,Speaking,3.12,3.62,49.60,Lower,RAG_DOC_0169,O*NET,15-1252.00,"For Software Developer, O*NET identifies Speaking as a relevant skill. The skill importance score is 3.12 and the required level is 3.62. O*NET occupation: Software Developers (15-1252.00)."
2,Monitoring,3.00,3.50,48.16,Lower,RAG_DOC_0166,O*NET,15-1252.00,"For Software Developer, O*NET identifies Monitoring as a relevant skill. The skill importance score is 3.00 and the required level is 3.50. O*NET occupation: Software Developers (15-1252.00)."



Exact skill-gap evidence validation:
Expected skill gaps: 3
Retrieved exact skill documents: 3
Missing exact O*NET documents: 0
Skill names match: True
Occupation evidence match: True

✅ PASS — Exact O*NET evidence retrieved for every skill gap.


In [133]:
# ============================================================
# CELL 55 — Create Final Grounded RAG Explanation
# ============================================================

print("Creating final grounded RAG explanation function...")


def clean_sentence_end(text):
    """
    Remove duplicate sentence-ending punctuation.
    """
    if text is None:
        return ""

    text = str(text).strip()

    while text.endswith(".."):
        text = text[:-1]

    return text


def generate_final_grounded_rag_explanation(
    evidence_package
):
    """
    Generate a clean, deterministic, evidence-grounded
    career recommendation explanation with traceable sources.
    """

    if evidence_package is None:
        return None

    candidate_id = (
        evidence_package["candidate_id"]
    )

    occupation = (
        evidence_package[
            "recommended_occupation"
        ]
    )

    hybrid_score = float(
        evidence_package[
            "hybrid_recommendation_score"
        ]
    )

    recommendation_explanation = clean_sentence_end(
        evidence_package[
            "recommendation_explanation"
        ]
    )

    missing_skills = (
        evidence_package[
            "missing_skills"
        ]
    )

    exact_skill_evidence = (
        evidence_package.get(
            "exact_skill_gap_evidence",
            []
        )
    )

    demand_percentage = (
        evidence_package[
            "demand_percentage"
        ]
    )

    demand_level = (
        evidence_package[
            "demand_level"
        ]
    )

    cip_status = (
        evidence_package[
            "cip_status"
        ]
    )

    education_action = clean_sentence_end(
        evidence_package[
            "education_action"
        ]
    )

    task_evidence = (
        evidence_package[
            "task_evidence"
        ]
    )

    demand_evidence = (
        evidence_package[
            "demand_evidence"
        ]
    )

    education_evidence = (
        evidence_package[
            "education_evidence"
        ]
    )


    # --------------------------------------------------------
    # 1. RECOMMENDATION
    # --------------------------------------------------------

    sections = []

    sections.append(
        f"Recommendation: {occupation} is the Top-1 career "
        f"recommendation for {candidate_id}, with a hybrid "
        f"recommendation score of {hybrid_score:.2f}. "
        f"{recommendation_explanation}"
    )


    # --------------------------------------------------------
    # 2. SKILL GAPS WITH EXACT SOURCES
    # --------------------------------------------------------

    if exact_skill_evidence:

        skill_sentences = []

        for record in exact_skill_evidence:

            skill_sentences.append(
                f"{record['skill']} "
                f"({record['prescriptive_priority_percentage']:.2f}% "
                f"prescriptive priority; O*NET importance "
                f"{record['importance']:.2f}; required level "
                f"{record['level']:.2f}) "
                f"[{record['document_id']}]"
            )

        sections.append(
            "Priority skill gaps: "
            + "; ".join(skill_sentences)
            + "."
        )

    elif missing_skills:

        skill_sentences = []

        for record in missing_skills:

            skill_sentences.append(
                f"{record['skill']} "
                f"({record['prescriptive_priority_percentage']:.2f}% "
                f"priority)"
            )

        sections.append(
            "Priority skill gaps: "
            + "; ".join(skill_sentences)
            + "."
        )

    else:

        sections.append(
            "Skill-gap analysis: No missing O*NET skills "
            "were identified for the candidate's Top-1 "
            "occupation using the current matching method."
        )


    # --------------------------------------------------------
    # 3. OCCUPATIONAL TASK EVIDENCE
    # --------------------------------------------------------

    if task_evidence:

        task_sentences = []

        for record in task_evidence[:3]:

            task_text = clean_sentence_end(
                record["document_text"]
            )

            task_sentences.append(
                f"{task_text} "
                f"[{record['document_id']}]"
            )

        sections.append(
            "Relevant occupational evidence: "
            + " ".join(task_sentences)
        )


    # --------------------------------------------------------
    # 4. CANADIAN LABOUR DEMAND
    # --------------------------------------------------------

    if demand_evidence:

        demand_text = clean_sentence_end(
            demand_evidence[0][
                "document_text"
            ]
        )

        sections.append(
            f"Canadian labour demand: The project-derived "
            f"demand score for {occupation} is "
            f"{demand_percentage:.2f}%, categorized as "
            f"{demand_level}. "
            f"{demand_text} "
            f"[{demand_evidence[0]['document_id']}]"
        )

    else:

        sections.append(
            "Canadian labour demand: No supporting labour-"
            "demand document was retrieved."
        )


    # --------------------------------------------------------
    # 5. EDUCATION PATHWAY
    # --------------------------------------------------------

    education_parts = [
        f"Education pathway status: {cip_status}.",
        f"{education_action}."
    ]

    for record in education_evidence:

        education_text = clean_sentence_end(
            record["document_text"]
        )

        education_parts.append(
            f"{education_text} "
            f"[{record['document_id']}]"
        )

    sections.append(
        " ".join(
            education_parts
        )
    )


    # --------------------------------------------------------
    # 6. SOURCE SUMMARY
    # --------------------------------------------------------

    citation_records = []

    for record in exact_skill_evidence:

        if record["document_id"] is not None:

            citation_records.append(
                (
                    record["document_id"],
                    record["source_name"],
                    record["source_reference"]
                )
            )

    for evidence_key in [
        "task_evidence",
        "demand_evidence",
        "education_evidence"
    ]:

        for record in evidence_package[
            evidence_key
        ]:

            citation_records.append(
                (
                    record["document_id"],
                    record["source_name"],
                    record["source_reference"]
                )
            )

    citation_records = list(
        dict.fromkeys(
            citation_records
        )
    )

    citation_text = "; ".join(
        [
            f"{document_id} — "
            f"{source_name} "
            f"[{source_reference}]"
            for (
                document_id,
                source_name,
                source_reference
            ) in citation_records
        ]
    )

    sections.append(
        "Supporting sources: "
        + citation_text
        + "."
    )


    # --------------------------------------------------------
    # 7. RESPONSIBLE-USE STATEMENT
    # --------------------------------------------------------

    sections.append(
        "Limitation: This recommendation is a decision-support "
        "output rather than a guaranteed career outcome. "
        "The recommendation and demand scores are "
        "project-derived analytical measures and should be "
        "considered together with candidate preferences, "
        "experience, current labour-market conditions, and "
        "human career guidance."
    )


    return "\n\n".join(
        sections
    )


# ============================================================
# TEST WITH CANDIDATE_001
# ============================================================

candidate_001_final_rag_explanation = (
    generate_final_grounded_rag_explanation(
        candidate_001_rag_evidence
    )
)

print(
    "\nFINAL GROUNDED RAG EXPLANATION — Candidate_001\n"
)

print(
    candidate_001_final_rag_explanation
)

Creating final grounded RAG explanation function...

FINAL GROUNDED RAG EXPLANATION — Candidate_001

Recommendation: Software Developer is the Top-1 career recommendation for Candidate_001, with a hybrid recommendation score of 69.05. Moderate recommendation based on combined candidate profile similarity and O*NET skill alignment.

Priority skill gaps: Active Listening (52.72% prescriptive priority; O*NET importance 3.38; required level 3.88) [RAG_DOC_0162]; Speaking (49.60% prescriptive priority; O*NET importance 3.12; required level 3.62) [RAG_DOC_0169]; Monitoring (48.16% prescriptive priority; O*NET importance 3.00; required level 3.50) [RAG_DOC_0166].

Relevant occupational evidence: For Software Developer, O*NET lists the following occupational task: Design, develop and modify software systems, using scientific analysis and mathematical models to predict and measure outcomes and consequences of design.. O*NET occupation: Software Developers (15-1252.00). [RAG_DOC_0256] For Softwa

In [135]:
# ============================================================
# CELL 56 — Generate RAG Evidence and Explanations
#           for All Candidates
# ============================================================

print("Generating grounded RAG outputs for all candidates...")


# ------------------------------------------------------------
# HELPER — CLEAN FINAL TEXT
# ------------------------------------------------------------

def clean_rag_output_text(text):
    """
    Clean minor punctuation/spacing artifacts in final
    generated RAG explanations without changing content.
    """

    if text is None:
        return None

    text = str(text)

    # Remove duplicate full stops
    text = text.replace("..", ".")

    # Remove accidental repeated spaces
    text = " ".join(text.split())

    return text.strip()


# ------------------------------------------------------------
# GENERATE EVIDENCE + EXPLANATION FOR EVERY CANDIDATE
# ------------------------------------------------------------

all_candidate_rag_records = []

candidate_ids = (
    top1_recommendation[
        "candidate_id"
    ]
    .drop_duplicates()
    .tolist()
)

for candidate_id in candidate_ids:

    # --------------------------------------------------------
    # Build evidence package
    # --------------------------------------------------------

    evidence_package = (
        build_candidate_rag_evidence(
            candidate_id=candidate_id
        )
    )

    if evidence_package is None:
        continue


    # --------------------------------------------------------
    # Retrieve exact evidence for identified skill gaps
    # --------------------------------------------------------

    exact_skill_evidence = (
        retrieve_exact_skill_gap_evidence(
            occupation=
                evidence_package[
                    "recommended_occupation"
                ],

            missing_skills=
                evidence_package[
                    "missing_skills"
                ]
        )
    )

    evidence_package[
        "exact_skill_gap_evidence"
    ] = exact_skill_evidence


    # --------------------------------------------------------
    # Generate grounded explanation
    # --------------------------------------------------------

    rag_explanation = (
        generate_final_grounded_rag_explanation(
            evidence_package
        )
    )

    rag_explanation = (
        clean_rag_output_text(
            rag_explanation
        )
    )


    # --------------------------------------------------------
    # PREPARE SKILL-GAP TEXT
    # --------------------------------------------------------

    if exact_skill_evidence:

        missing_skill_names = ", ".join(
            [
                record["skill"]
                for record in exact_skill_evidence
            ]
        )

        missing_skill_count = len(
            exact_skill_evidence
        )

        skill_source_ids = "; ".join(
            [
                record["document_id"]
                for record in exact_skill_evidence
                if record["document_id"] is not None
            ]
        )

    else:

        missing_skill_names = (
            "No missing O*NET skills identified"
        )

        missing_skill_count = 0

        skill_source_ids = ""


    # --------------------------------------------------------
    # RETRIEVED SOURCE IDS
    # --------------------------------------------------------

    task_source_ids = "; ".join(
        [
            record["document_id"]
            for record in evidence_package[
                "task_evidence"
            ]
        ]
    )

    demand_source_ids = "; ".join(
        [
            record["document_id"]
            for record in evidence_package[
                "demand_evidence"
            ]
        ]
    )

    education_source_ids = "; ".join(
        [
            record["document_id"]
            for record in evidence_package[
                "education_evidence"
            ]
        ]
    )


    # --------------------------------------------------------
    # SAVE CANDIDATE RECORD
    # --------------------------------------------------------

    all_candidate_rag_records.append(
        {
            "candidate_id":
                candidate_id,

            "recommended_occupation":
                evidence_package[
                    "recommended_occupation"
                ],

            "hybrid_recommendation_score":
                evidence_package[
                    "hybrid_recommendation_score"
                ],

            "missing_skill_count":
                missing_skill_count,

            "priority_missing_skills":
                missing_skill_names,

            "demand_percentage":
                evidence_package[
                    "demand_percentage"
                ],

            "demand_level":
                evidence_package[
                    "demand_level"
                ],

            "cip_status":
                evidence_package[
                    "cip_status"
                ],

            "education_action":
                evidence_package[
                    "education_action"
                ],

            "skill_source_ids":
                skill_source_ids,

            "task_source_ids":
                task_source_ids,

            "demand_source_ids":
                demand_source_ids,

            "education_source_ids":
                education_source_ids,

            "rag_explanation":
                rag_explanation
        }
    )


# ------------------------------------------------------------
# CONVERT TO DATAFRAME
# ------------------------------------------------------------

all_candidate_rag_explanations = (
    pd.DataFrame(
        all_candidate_rag_records
    )
)


# ------------------------------------------------------------
# DISPLAY SUMMARY
# ------------------------------------------------------------

print(
    "\nRAG explanations generated:",
    len(all_candidate_rag_explanations)
)

print(
    "Unique candidates:",
    all_candidate_rag_explanations[
        "candidate_id"
    ].nunique()
)

print(
    "Candidates with missing skills:",
    (
        all_candidate_rag_explanations[
            "missing_skill_count"
        ]
        > 0
    ).sum()
)

print(
    "Candidates with no identified missing skills:",
    (
        all_candidate_rag_explanations[
            "missing_skill_count"
        ]
        == 0
    ).sum()
)

print(
    "\nRecommended occupation distribution:"
)

display(
    all_candidate_rag_explanations[
        "recommended_occupation"
    ]
    .value_counts()
    .rename_axis(
        "recommended_occupation"
    )
    .reset_index(
        name="candidate_count"
    )
)


print(
    "\nSample candidate RAG outputs:"
)

display(
    all_candidate_rag_explanations[
        [
            "candidate_id",
            "recommended_occupation",
            "hybrid_recommendation_score",
            "missing_skill_count",
            "priority_missing_skills",
            "demand_percentage",
            "demand_level",
            "cip_status"
        ]
    ]
    .head(15)
)

Generating grounded RAG outputs for all candidates...

RAG explanations generated: 63
Unique candidates: 63
Candidates with missing skills: 56
Candidates with no identified missing skills: 7

Recommended occupation distribution:


,recommended_occupation,candidate_count
0,Software Developer,26
1,Office Manager,15
2,Administrative Assistant,12
3,Food Service Supervisor,7
4,Delivery Driver,1
5,"Driver, Truck",1
6,Bookkeeper,1



Sample candidate RAG outputs:


,candidate_id,recommended_occupation,hybrid_recommendation_score,missing_skill_count,priority_missing_skills,demand_percentage,demand_level,cip_status
0,Candidate_001,Software Developer,69.0545,3,"Active Listening, Speaking, Monitoring",22.91,Lower,CIP pathway available
1,Candidate_002,Office Manager,55.6242,3,"Monitoring, Speaking, Active Listening",52.86,Moderate,CIP pathway available
2,Candidate_003,Administrative Assistant,93.2442,2,"Mathematics, Science",43.85,Moderate,CIP pathway available
3,Candidate_004,Software Developer,78.5914,2,"Active Listening, Speaking",22.91,Lower,CIP pathway available
4,Candidate_005,Software Developer,53.3510,3,"Active Listening, Writing, Speaking",22.91,Lower,CIP pathway available
5,Candidate_006,Software Developer,90.4630,1,Monitoring,22.91,Lower,CIP pathway available
6,Candidate_007,Food Service Supervisor,100.0000,0,No missing O*NET skills identified,68.33,Moderate,CIP pathway available
7,Candidate_008,Software Developer,68.0845,3,"Active Listening, Writing, Speaking",22.91,Lower,CIP pathway available
8,Candidate_009,Administrative Assistant,27.7293,3,"Active Listening, Speaking, Critical Thinking",43.85,Moderate,CIP pathway available
9,Candidate_010,Office Manager,98.5612,1,Science,52.86,Moderate,CIP pathway available


In [137]:
# ============================================================
# CELL 57 — Validate All Candidate RAG Outputs
# ============================================================

print("Running final candidate RAG output validation...\n")


# ------------------------------------------------------------
# 1. REQUIRED COLUMNS
# ------------------------------------------------------------

required_rag_output_columns = [
    "candidate_id",
    "recommended_occupation",
    "hybrid_recommendation_score",
    "missing_skill_count",
    "priority_missing_skills",
    "demand_percentage",
    "demand_level",
    "cip_status",
    "education_action",
    "skill_source_ids",
    "task_source_ids",
    "demand_source_ids",
    "education_source_ids",
    "rag_explanation"
]

missing_output_columns = [
    col
    for col in required_rag_output_columns
    if col not in all_candidate_rag_explanations.columns
]


# ------------------------------------------------------------
# 2. CANDIDATE COVERAGE
# ------------------------------------------------------------

expected_candidate_count = (
    top1_recommendation[
        "candidate_id"
    ]
    .nunique()
)

actual_candidate_count = (
    all_candidate_rag_explanations[
        "candidate_id"
    ]
    .nunique()
)

duplicate_candidate_rows = int(
    all_candidate_rag_explanations[
        "candidate_id"
    ]
    .duplicated()
    .sum()
)


# ------------------------------------------------------------
# 3. TOP-1 OCCUPATION ALIGNMENT
# ------------------------------------------------------------

rag_alignment_check = (
    all_candidate_rag_explanations[
        [
            "candidate_id",
            "recommended_occupation"
        ]
    ]
    .merge(
        top1_recommendation[
            [
                "candidate_id",
                "recommended_occupation"
            ]
        ],
        on="candidate_id",
        how="left",
        suffixes=(
            "_rag",
            "_week6"
        )
    )
)

occupation_alignment_errors = int(
    (
        rag_alignment_check[
            "recommended_occupation_rag"
        ]
        !=
        rag_alignment_check[
            "recommended_occupation_week6"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 4. SCORE VALIDATION
# ------------------------------------------------------------

invalid_hybrid_scores = int(
    (
        (
            all_candidate_rag_explanations[
                "hybrid_recommendation_score"
            ]
            < 0
        )
        |
        (
            all_candidate_rag_explanations[
                "hybrid_recommendation_score"
            ]
            > 100
        )
    ).sum()
)

invalid_demand_scores = int(
    (
        (
            all_candidate_rag_explanations[
                "demand_percentage"
            ]
            < 0
        )
        |
        (
            all_candidate_rag_explanations[
                "demand_percentage"
            ]
            > 100
        )
    ).sum()
)

# ------------------------------------------------------------
# 5. EXPLANATION COMPLETENESS — CORRECTED
# ------------------------------------------------------------

explanation_text = (
    all_candidate_rag_explanations[
        "rag_explanation"
    ]
    .fillna("")
    .astype(str)
)


empty_explanations = int(
    explanation_text
    .str.strip()
    .eq("")
    .sum()
)


missing_recommendation_section = int(
    (
        ~explanation_text.str.contains(
            "Recommendation:",
            regex=False
        )
    ).sum()
)


missing_demand_section = int(
    (
        ~explanation_text.str.contains(
            "Canadian labour demand:",
            regex=False
        )
    ).sum()
)


missing_education_section = int(
    (
        ~explanation_text.str.contains(
            "Education pathway status:",
            regex=False
        )
    ).sum()
)


missing_source_section = int(
    (
        ~explanation_text.str.contains(
            "Supporting sources:",
            regex=False
        )
    ).sum()
)


missing_limitation_section = int(
    (
        ~explanation_text.str.contains(
            "Limitation:",
            regex=False
        )
    ).sum()
)

 



# ------------------------------------------------------------
# 6. SOURCE-ID VALIDATION
# ------------------------------------------------------------

valid_document_ids = set(
    rag_documents[
        "document_id"
    ]
    .astype(str)
)

invalid_source_references = []

source_columns = [
    "skill_source_ids",
    "task_source_ids",
    "demand_source_ids",
    "education_source_ids"
]

for _, row in (
    all_candidate_rag_explanations.iterrows()
):

    for source_column in source_columns:

        source_text = row[
            source_column
        ]

        if pd.isna(source_text):
            continue

        source_text = str(
            source_text
        ).strip()

        if source_text == "":
            continue

        ids = [
            x.strip()
            for x in source_text.split(";")
            if x.strip()
        ]

        for document_id in ids:

            if (
                document_id
                not in valid_document_ids
            ):

                invalid_source_references.append(
                    {
                        "candidate_id":
                            row[
                                "candidate_id"
                            ],

                        "source_column":
                            source_column,

                        "document_id":
                            document_id
                    }
                )

invalid_source_reference_count = len(
    invalid_source_references
)


# ------------------------------------------------------------
# 7. SKILL-GAP / SOURCE CONSISTENCY
# ------------------------------------------------------------

candidates_with_gaps = (
    all_candidate_rag_explanations[
        "missing_skill_count"
    ]
    > 0
)

gap_candidates_missing_skill_sources = int(
    (
        candidates_with_gaps
        &
        all_candidate_rag_explanations[
            "skill_source_ids"
        ]
        .fillna("")
        .str.strip()
        .eq("")
    ).sum()
)

no_gap_candidates_with_skill_sources = int(
    (
        ~candidates_with_gaps
        &
        all_candidate_rag_explanations[
            "skill_source_ids"
        ]
        .fillna("")
        .str.strip()
        .ne("")
    ).sum()
)


# ------------------------------------------------------------
# 8. DEMAND / EDUCATION SOURCE COVERAGE
# ------------------------------------------------------------

missing_demand_sources = int(
    all_candidate_rag_explanations[
        "demand_source_ids"
    ]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

missing_education_sources = int(
    all_candidate_rag_explanations[
        "education_source_ids"
    ]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

missing_task_sources = int(
    all_candidate_rag_explanations[
        "task_source_ids"
    ]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)


# ------------------------------------------------------------
# 9. PRINT VALIDATION RESULTS
# ------------------------------------------------------------

print(
    "Missing required columns:",
    missing_output_columns
)

print(
    "Expected candidates:",
    expected_candidate_count
)

print(
    "Actual candidates:",
    actual_candidate_count
)

print(
    "Duplicate candidate rows:",
    duplicate_candidate_rows
)

print(
    "Top-1 occupation alignment errors:",
    occupation_alignment_errors
)

print(
    "Invalid hybrid scores:",
    invalid_hybrid_scores
)

print(
    "Invalid demand scores:",
    invalid_demand_scores
)

print(
    "Empty RAG explanations:",
    empty_explanations
)

print(
    "Missing recommendation sections:",
    missing_recommendation_section
)

print(
    "Missing demand sections:",
    missing_demand_section
)

print(
    "Missing education sections:",
    missing_education_section
)

print(
    "Missing source sections:",
    missing_source_section
)

print(
    "Missing limitation sections:",
    missing_limitation_section
)

print(
    "Invalid source references:",
    invalid_source_reference_count
)

print(
    "Gap candidates missing skill sources:",
    gap_candidates_missing_skill_sources
)

print(
    "No-gap candidates incorrectly having skill sources:",
    no_gap_candidates_with_skill_sources
)

print(
    "Candidates missing task sources:",
    missing_task_sources
)

print(
    "Candidates missing demand sources:",
    missing_demand_sources
)

print(
    "Candidates missing education sources:",
    missing_education_sources
)


# ------------------------------------------------------------
# 10. FINAL PASS / FAIL
# ------------------------------------------------------------

final_rag_validation_pass = (
    len(missing_output_columns) == 0
    and expected_candidate_count == actual_candidate_count
    and duplicate_candidate_rows == 0
    and occupation_alignment_errors == 0
    and invalid_hybrid_scores == 0
    and invalid_demand_scores == 0
    and empty_explanations == 0
    and missing_recommendation_section == 0
    and missing_demand_section == 0
    and missing_education_section == 0
    and missing_source_section == 0
    and missing_limitation_section == 0
    and invalid_source_reference_count == 0
    and gap_candidates_missing_skill_sources == 0
    and no_gap_candidates_with_skill_sources == 0
    and missing_task_sources == 0
    and missing_demand_sources == 0
    and missing_education_sources == 0
)

if final_rag_validation_pass:

    print(
        "\n✅ PASS — Final candidate RAG "
        "output validation successful."
    )

else:

    print(
        "\n❌ FAIL — Review RAG output "
        "validation issues before exporting."
    )


# ------------------------------------------------------------
# 11. OPTIONAL INVALID-SOURCE DETAIL
# ------------------------------------------------------------

if invalid_source_reference_count > 0:

    print(
        "\nInvalid source-reference details:"
    )

    display(
        pd.DataFrame(
            invalid_source_references
        )
    )

Running final candidate RAG output validation...

Missing required columns: []
Expected candidates: 63
Actual candidates: 63
Duplicate candidate rows: 0
Top-1 occupation alignment errors: 0
Invalid hybrid scores: 0
Invalid demand scores: 0
Empty RAG explanations: 0
Missing recommendation sections: 0
Missing demand sections: 0
Missing education sections: 0
Missing source sections: 0
Missing limitation sections: 0
Invalid source references: 0
Gap candidates missing skill sources: 0
No-gap candidates incorrectly having skill sources: 0
Candidates missing task sources: 0
Candidates missing demand sources: 0
Candidates missing education sources: 0

✅ PASS — Final candidate RAG output validation successful.


In [143]:
# ============================================================
# CELL 58A — Define Week 7 Output Directory
# ============================================================

from pathlib import Path

WEEK7_OUTPUT_DIR = Path(
    r"C:\Users\Admin\Capstone_Project"
    r"\Outputs\Tables\Week7"
)

WEEK7_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Week 7 output directory:")
print(WEEK7_OUTPUT_DIR)

print(
    "Directory exists:",
    WEEK7_OUTPUT_DIR.exists()
)

print(
    "Directory is valid:",
    WEEK7_OUTPUT_DIR.is_dir()
)

Week 7 output directory:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7
Directory exists: True
Directory is valid: True


In [145]:
# ============================================================
# CELL 58 — Export Week 7 RAG Outputs
# ============================================================

print("Exporting Week 7 RAG outputs...")


# ------------------------------------------------------------
# 1. EXPORT RAG KNOWLEDGE BASE
# ------------------------------------------------------------

rag_documents_path = (
    WEEK7_OUTPUT_DIR
    / "week7_rag_knowledge_base.csv"
)

rag_documents.to_csv(
    rag_documents_path,
    index=False
)


# ------------------------------------------------------------
# 2. EXPORT RAG EMBEDDINGS
# ------------------------------------------------------------

rag_embeddings_path = (
    WEEK7_OUTPUT_DIR
    / "week7_rag_embeddings.npy"
)

np.save(
    rag_embeddings_path,
    rag_embeddings
)


# ------------------------------------------------------------
# 3. EXPORT FAISS INDEX
# ------------------------------------------------------------

faiss_index_path = (
    WEEK7_OUTPUT_DIR
    / "week7_rag_faiss.index"
)

faiss.write_index(
    faiss_index,
    str(faiss_index_path)
)


# ------------------------------------------------------------
# 4. EXPORT ALL CANDIDATE RAG EXPLANATIONS
# ------------------------------------------------------------

rag_explanations_path = (
    WEEK7_OUTPUT_DIR
    / "week7_candidate_rag_explanations.csv"
)

all_candidate_rag_explanations.to_csv(
    rag_explanations_path,
    index=False
)


# ------------------------------------------------------------
# 5. EXPORT RAG VALIDATION SUMMARY
# ------------------------------------------------------------

rag_validation_summary = pd.DataFrame(
    [
        {
            "validation_metric":
                "Expected candidates",
            "value":
                expected_candidate_count
        },
        {
            "validation_metric":
                "Actual candidates",
            "value":
                actual_candidate_count
        },
        {
            "validation_metric":
                "Duplicate candidate rows",
            "value":
                duplicate_candidate_rows
        },
        {
            "validation_metric":
                "Top-1 occupation alignment errors",
            "value":
                occupation_alignment_errors
        },
        {
            "validation_metric":
                "Invalid hybrid scores",
            "value":
                invalid_hybrid_scores
        },
        {
            "validation_metric":
                "Invalid demand scores",
            "value":
                invalid_demand_scores
        },
        {
            "validation_metric":
                "Empty RAG explanations",
            "value":
                empty_explanations
        },
        {
            "validation_metric":
                "Invalid source references",
            "value":
                invalid_source_reference_count
        },
        {
            "validation_metric":
                "Gap candidates missing skill sources",
            "value":
                gap_candidates_missing_skill_sources
        },
        {
            "validation_metric":
                "Candidates missing task sources",
            "value":
                missing_task_sources
        },
        {
            "validation_metric":
                "Candidates missing demand sources",
            "value":
                missing_demand_sources
        },
        {
            "validation_metric":
                "Candidates missing education sources",
            "value":
                missing_education_sources
        },
        {
            "validation_metric":
                "Final RAG validation pass",
            "value":
                final_rag_validation_pass
        }
    ]
)

rag_validation_path = (
    WEEK7_OUTPUT_DIR
    / "week7_rag_validation_summary.csv"
)

rag_validation_summary.to_csv(
    rag_validation_path,
    index=False
)


# ------------------------------------------------------------
# 6. VALIDATE EXPORTED FILES
# ------------------------------------------------------------

exported_rag_files = [
    rag_documents_path,
    rag_embeddings_path,
    faiss_index_path,
    rag_explanations_path,
    rag_validation_path
]

print(
    "\nWeek 7 RAG outputs exported successfully."
)

print(
    "\nOutput directory:"
)

print(
    WEEK7_OUTPUT_DIR
)

print(
    "\nFiles created:"
)

for file_path in exported_rag_files:

    if file_path.exists():

        print(
            "✓",
            file_path.name,
            "|",
            file_path.stat().st_size,
            "bytes"
        )

    else:

        print(
            "✗ MISSING:",
            file_path.name
        )


# ------------------------------------------------------------
# 7. FINAL EXPORT VALIDATION
# ------------------------------------------------------------

all_rag_exports_exist = all(
    file_path.exists()
    for file_path in exported_rag_files
)

if all_rag_exports_exist:

    print(
        "\n✅ PASS — All Week 7 RAG artifacts "
        "exported successfully."
    )

else:

    print(
        "\n❌ FAIL — One or more RAG export "
        "files are missing."
    )

Exporting Week 7 RAG outputs...

Week 7 RAG outputs exported successfully.

Output directory:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7

Files created:
✓ week7_rag_knowledge_base.csv | 219274 bytes
✓ week7_rag_embeddings.npy | 1086080 bytes
✓ week7_rag_faiss.index | 1085997 bytes
✓ week7_candidate_rag_explanations.csv | 202789 bytes
✓ week7_rag_validation_summary.csv | 425 bytes

✅ PASS — All Week 7 RAG artifacts exported successfully.


In [147]:
# ============================================================
# CELL 59 — Create Skill-Extraction Human Review Template
# ============================================================

print("Creating skill-extraction human-review template...")


# ------------------------------------------------------------
# 1. SELECT CANDIDATES FOR HUMAN REVIEW
# ------------------------------------------------------------

review_sample_size = min(
    20,
    candidate_features[
        "candidate_id"
    ].nunique()
)

review_candidates = (
    candidate_features[
        "candidate_id"
    ]
    .drop_duplicates()
    .sort_values()
    .head(
        review_sample_size
    )
    .tolist()
)

print(
    "Candidates selected for review:",
    len(review_candidates)
)


# ------------------------------------------------------------
# 2. PREPARE CANDIDATE O*NET SKILL EVIDENCE
# ------------------------------------------------------------

skill_columns = [
    col
    for col in candidate_onet_skill_matrix.columns
    if not col.endswith(
        "_evidence_count"
    )
    and col not in [
        "candidate_id",
        "onet_skill_match_count",
        "total_onet_evidence_count"
    ]
]

print(
    "O*NET skill dimensions:",
    len(skill_columns)
)

print(
    "Skills:",
    skill_columns
)


# ------------------------------------------------------------
# 3. CONVERT MATRIX TO HUMAN-REVIEW FORMAT
# ------------------------------------------------------------

skill_review_records = []

for candidate_id in review_candidates:

    candidate_row = (
        candidate_onet_skill_matrix[
            candidate_onet_skill_matrix[
                "candidate_id"
            ]
            == candidate_id
        ]
    )

    if candidate_row.empty:
        continue

    candidate_row = (
        candidate_row.iloc[0]
    )

    candidate_profile = (
        candidate_features[
            candidate_features[
                "candidate_id"
            ]
            == candidate_id
        ]
    )

    if not candidate_profile.empty:

        candidate_profile = (
            candidate_profile.iloc[0]
        )

        cv_text = candidate_profile.get(
            "clean_text",
            ""
        )

    else:

        cv_text = ""

    for skill in skill_columns:

        evidence_column = (
            skill
            + "_evidence_count"
        )

        extracted_value = (
            candidate_row[
                skill
            ]
        )

        if evidence_column in (
            candidate_onet_skill_matrix.columns
        ):

            evidence_count = (
                candidate_row[
                    evidence_column
                ]
            )

        else:

            evidence_count = 0

        # Include only skills identified by the system
        if extracted_value > 0:

            skill_review_records.append(
                {
                    "candidate_id":
                        candidate_id,

                    "skill":
                        skill,

                    "system_extracted":
                        1,

                    "system_skill_value":
                        extracted_value,

                    "evidence_count":
                        evidence_count,

                    "cv_text":
                        cv_text,

                    # ----------------------------------------
                    # HUMAN REVIEW FIELDS
                    # ----------------------------------------

                    "human_relevant":
                        "",

                    "human_missed_skill":
                        "",

                    "reviewer_notes":
                        ""
                }
            )


skill_extraction_review = (
    pd.DataFrame(
        skill_review_records
    )
)


# ------------------------------------------------------------
# 4. VALIDATE TEMPLATE
# ------------------------------------------------------------

print(
    "\nHuman-review rows:",
    len(skill_extraction_review)
)

print(
    "Unique candidates:",
    skill_extraction_review[
        "candidate_id"
    ].nunique()
)

print(
    "Unique extracted skills:",
    skill_extraction_review[
        "skill"
    ].nunique()
)

print(
    "Duplicate candidate-skill rows:",
    skill_extraction_review[
        [
            "candidate_id",
            "skill"
        ]
    ]
    .duplicated()
    .sum()
)


# ------------------------------------------------------------
# 5. DISPLAY REVIEW SAMPLE
# ------------------------------------------------------------

print(
    "\nSkill-extraction review template sample:"
)

display(
    skill_extraction_review[
        [
            "candidate_id",
            "skill",
            "system_extracted",
            "system_skill_value",
            "evidence_count",
            "human_relevant",
            "human_missed_skill",
            "reviewer_notes"
        ]
    ]
    .head(20)
)


# ------------------------------------------------------------
# 6. EXPORT HUMAN-REVIEW TEMPLATE
# ------------------------------------------------------------

skill_review_path = (
    WEEK7_OUTPUT_DIR
    / "week7_skill_extraction_human_review.csv"
)

skill_extraction_review.to_csv(
    skill_review_path,
    index=False
)


print(
    "\nHuman-review template exported:"
)

print(
    skill_review_path
)

print(
    "File exists:",
    skill_review_path.exists()
)

print(
    "File size:",
    skill_review_path.stat().st_size,
    "bytes"
)


# ------------------------------------------------------------
# 7. REVIEW INSTRUCTIONS
# ------------------------------------------------------------

print(
    "\nHUMAN REVIEW INSTRUCTIONS:"
)

print(
    "1. human_relevant: enter 1 if the extracted skill "
    "is supported by the CV; otherwise enter 0."
)

print(
    "2. human_missed_skill: record any important O*NET "
    "skill present in the CV but missed by the system."
)

print(
    "3. reviewer_notes: optionally record why the "
    "decision was made."
)

print(
    "4. Do NOT calculate Precision, Recall, or F1 until "
    "human review has been completed."
)

Creating skill-extraction human-review template...
Candidates selected for review: 20
O*NET skill dimensions: 10
Skills: ['Active Learning', 'Active Listening', 'Critical Thinking', 'Learning Strategies', 'Mathematics', 'Monitoring', 'Reading Comprehension', 'Science', 'Speaking', 'Writing']

Human-review rows: 143
Unique candidates: 20
Unique extracted skills: 10
Duplicate candidate-skill rows: 0

Skill-extraction review template sample:


,candidate_id,skill,system_extracted,system_skill_value,evidence_count,human_relevant,human_missed_skill,reviewer_notes
0,Candidate_001,Active Learning,1,1,2,,,
1,Candidate_001,Critical Thinking,1,1,4,,,
2,Candidate_001,Learning Strategies,1,1,2,,,
3,Candidate_001,Mathematics,1,1,2,,,
4,Candidate_001,Reading Comprehension,1,1,5,,,
5,Candidate_001,Science,1,1,1,,,
6,Candidate_001,Writing,1,1,3,,,
7,Candidate_002,Active Learning,1,1,3,,,
8,Candidate_002,Critical Thinking,1,1,3,,,
9,Candidate_002,Learning Strategies,1,1,2,,,



Human-review template exported:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7\week7_skill_extraction_human_review.csv
File exists: True
File size: 343404 bytes

HUMAN REVIEW INSTRUCTIONS:
1. human_relevant: enter 1 if the extracted skill is supported by the CV; otherwise enter 0.
2. human_missed_skill: record any important O*NET skill present in the CV but missed by the system.
3. reviewer_notes: optionally record why the decision was made.
4. Do NOT calculate Precision, Recall, or F1 until human review has been completed.


In [149]:
# ============================================================
# CELL 60 — Create Complete Skill-Extraction Review Matrix
# ============================================================

print("Creating complete skill-extraction evaluation matrix...")


# ------------------------------------------------------------
# 1. BUILD FULL CANDIDATE × SKILL REVIEW GRID
# ------------------------------------------------------------

full_skill_review_records = []

for candidate_id in review_candidates:

    candidate_matrix_row = (
        candidate_onet_skill_matrix[
            candidate_onet_skill_matrix[
                "candidate_id"
            ] == candidate_id
        ]
    )

    if candidate_matrix_row.empty:
        continue

    candidate_matrix_row = (
        candidate_matrix_row.iloc[0]
    )


    candidate_profile_row = (
        candidate_features[
            candidate_features[
                "candidate_id"
            ] == candidate_id
        ]
    )

    if not candidate_profile_row.empty:

        candidate_profile_row = (
            candidate_profile_row.iloc[0]
        )

        cv_text = candidate_profile_row.get(
            "clean_text",
            ""
        )

    else:

        cv_text = ""


    # --------------------------------------------------------
    # CHECK ALL 10 O*NET SKILLS
    # --------------------------------------------------------

    for skill in skill_columns:

        evidence_column = (
            skill + "_evidence_count"
        )

        system_value = float(
            candidate_matrix_row[
                skill
            ]
        )

        system_extracted = int(
            system_value > 0
        )

        if (
            evidence_column
            in candidate_onet_skill_matrix.columns
        ):

            evidence_count = int(
                candidate_matrix_row[
                    evidence_column
                ]
            )

        else:

            evidence_count = 0


        full_skill_review_records.append(
            {
                "candidate_id":
                    candidate_id,

                "skill":
                    skill,

                "system_extracted":
                    system_extracted,

                "system_skill_value":
                    system_value,

                "evidence_count":
                    evidence_count,

                "cv_text":
                    cv_text,

                # --------------------------------------------
                # HUMAN GROUND-TRUTH FIELD
                # --------------------------------------------

                "human_skill_present":
                    "",

                "reviewer_notes":
                    ""
            }
        )


# ------------------------------------------------------------
# 2. CONVERT TO DATAFRAME
# ------------------------------------------------------------

skill_extraction_ground_truth = (
    pd.DataFrame(
        full_skill_review_records
    )
)


# ------------------------------------------------------------
# 3. VALIDATE COMPLETE MATRIX
# ------------------------------------------------------------

expected_rows = (
    len(review_candidates)
    * len(skill_columns)
)

actual_rows = len(
    skill_extraction_ground_truth
)

duplicate_candidate_skill_rows = int(
    skill_extraction_ground_truth[
        [
            "candidate_id",
            "skill"
        ]
    ]
    .duplicated()
    .sum()
)

system_positive_count = int(
    skill_extraction_ground_truth[
        "system_extracted"
    ]
    .sum()
)

system_negative_count = int(
    (
        skill_extraction_ground_truth[
            "system_extracted"
        ] == 0
    ).sum()
)


print(
    "\nExpected review rows:",
    expected_rows
)

print(
    "Actual review rows:",
    actual_rows
)

print(
    "Unique candidates:",
    skill_extraction_ground_truth[
        "candidate_id"
    ].nunique()
)

print(
    "Unique skills:",
    skill_extraction_ground_truth[
        "skill"
    ].nunique()
)

print(
    "Duplicate candidate-skill rows:",
    duplicate_candidate_skill_rows
)

print(
    "System-extracted skill rows:",
    system_positive_count
)

print(
    "System-not-extracted skill rows:",
    system_negative_count
)


# ------------------------------------------------------------
# 4. DISPLAY SAMPLE
# ------------------------------------------------------------

print(
    "\nComplete skill-extraction review sample:"
)

display(
    skill_extraction_ground_truth[
        [
            "candidate_id",
            "skill",
            "system_extracted",
            "system_skill_value",
            "evidence_count",
            "human_skill_present",
            "reviewer_notes"
        ]
    ]
    .head(25)
)


# ------------------------------------------------------------
# 5. EXPORT REVIEW TEMPLATE
# ------------------------------------------------------------

skill_ground_truth_path = (
    WEEK7_OUTPUT_DIR
    / "week7_skill_extraction_ground_truth_review.csv"
)

skill_extraction_ground_truth.to_csv(
    skill_ground_truth_path,
    index=False
)


print(
    "\nComplete ground-truth review template exported:"
)

print(
    skill_ground_truth_path
)

print(
    "File exists:",
    skill_ground_truth_path.exists()
)

print(
    "File size:",
    skill_ground_truth_path.stat().st_size,
    "bytes"
)


# ------------------------------------------------------------
# 6. FINAL VALIDATION
# ------------------------------------------------------------

skill_review_template_pass = (
    expected_rows == actual_rows
    and duplicate_candidate_skill_rows == 0
    and skill_extraction_ground_truth[
        "candidate_id"
    ].nunique() == len(review_candidates)
    and skill_extraction_ground_truth[
        "skill"
    ].nunique() == len(skill_columns)
)

if skill_review_template_pass:

    print(
        "\n✅ PASS — Complete skill-extraction "
        "ground-truth review matrix created successfully."
    )

else:

    print(
        "\n❌ FAIL — Review the skill-extraction "
        "evaluation matrix."
    )


# ------------------------------------------------------------
# 7. HUMAN REVIEW INSTRUCTIONS
# ------------------------------------------------------------

print(
    "\nHUMAN REVIEW INSTRUCTIONS:"
)

print(
    "For every row, read the candidate CV text and enter:"
)

print(
    "human_skill_present = 1 if the skill is genuinely "
    "supported by the CV."
)

print(
    "human_skill_present = 0 if the skill is not supported "
    "by the CV."
)

print(
    "Do not change system_extracted, system_skill_value, "
    "or evidence_count."
)

print(
    "Precision, Recall, F1, false positives, and false "
    "negatives will be calculated only after this human "
    "ground-truth column is completed."
)

Creating complete skill-extraction evaluation matrix...

Expected review rows: 200
Actual review rows: 200
Unique candidates: 20
Unique skills: 10
Duplicate candidate-skill rows: 0
System-extracted skill rows: 143
System-not-extracted skill rows: 57

Complete skill-extraction review sample:


,candidate_id,skill,system_extracted,system_skill_value,evidence_count,human_skill_present,reviewer_notes
0,Candidate_001,Active Learning,1,1.0,2,,
1,Candidate_001,Active Listening,0,0.0,0,,
2,Candidate_001,Critical Thinking,1,1.0,4,,
3,Candidate_001,Learning Strategies,1,1.0,2,,
4,Candidate_001,Mathematics,1,1.0,2,,
5,Candidate_001,Monitoring,0,0.0,0,,
6,Candidate_001,Reading Comprehension,1,1.0,5,,
7,Candidate_001,Science,1,1.0,1,,
8,Candidate_001,Speaking,0,0.0,0,,
9,Candidate_001,Writing,1,1.0,3,,



Complete ground-truth review template exported:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7\week7_skill_extraction_ground_truth_review.csv
File exists: True
File size: 486874 bytes

✅ PASS — Complete skill-extraction ground-truth review matrix created successfully.

HUMAN REVIEW INSTRUCTIONS:
For every row, read the candidate CV text and enter:
human_skill_present = 1 if the skill is genuinely supported by the CV.
human_skill_present = 0 if the skill is not supported by the CV.
Do not change system_extracted, system_skill_value, or evidence_count.
Precision, Recall, F1, false positives, and false negatives will be calculated only after this human ground-truth column is completed.


In [151]:
# ============================================================
# CELL 61 — Skill Extraction Evaluation Function
# ============================================================

print("Creating skill-extraction evaluation function...")


def evaluate_skill_extraction(
    review_df
):
    """
    Calculate skill-extraction evaluation metrics using
    completed human ground-truth labels.

    Metrics:
    - Precision
    - Recall
    - F1
    - Accuracy
    - True Positives
    - False Positives
    - False Negatives
    - True Negatives

    Metrics are calculated ONLY when human review is complete.
    """

    evaluation_df = (
        review_df.copy()
    )


    # --------------------------------------------------------
    # 1. VALIDATE REQUIRED COLUMNS
    # --------------------------------------------------------

    required_columns = [
        "candidate_id",
        "skill",
        "system_extracted",
        "human_skill_present"
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in evaluation_df.columns
    ]

    if missing_columns:

        print(
            "❌ Missing required columns:",
            missing_columns
        )

        return None, None, None


    # --------------------------------------------------------
    # 2. CLEAN HUMAN LABELS
    # --------------------------------------------------------

    evaluation_df[
        "human_skill_present"
    ] = pd.to_numeric(
        evaluation_df[
            "human_skill_present"
        ],
        errors="coerce"
    )


    total_rows = len(
        evaluation_df
    )

    reviewed_rows = int(
        evaluation_df[
            "human_skill_present"
        ]
        .notna()
        .sum()
    )

    unreviewed_rows = int(
        evaluation_df[
            "human_skill_present"
        ]
        .isna()
        .sum()
    )


    print(
        "\nTotal evaluation rows:",
        total_rows
    )

    print(
        "Human-reviewed rows:",
        reviewed_rows
    )

    print(
        "Unreviewed rows:",
        unreviewed_rows
    )


    # --------------------------------------------------------
    # 3. CHECK LABEL VALIDITY
    # --------------------------------------------------------

    reviewed_values = (
        evaluation_df.loc[
            evaluation_df[
                "human_skill_present"
            ].notna(),
            "human_skill_present"
        ]
    )

    invalid_human_labels = int(
        (
            ~reviewed_values.isin(
                [0, 1]
            )
        ).sum()
    )

    print(
        "Invalid human labels:",
        invalid_human_labels
    )


    if invalid_human_labels > 0:

        print(
            "\n❌ Human labels must contain "
            "only 0 or 1."
        )

        return None, None, evaluation_df


    # --------------------------------------------------------
    # 4. DO NOT CALCULATE FINAL METRICS
    #    UNTIL REVIEW IS COMPLETE
    # --------------------------------------------------------

    if unreviewed_rows > 0:

        completion_percentage = (
            reviewed_rows
            / total_rows
            * 100
        )

        print(
            "\nHuman-review completion:",
            f"{completion_percentage:.2f}%"
        )

        print(
            "\n⚠️ Skill-extraction evaluation "
            "is not yet complete."
        )

        print(
            "Precision, Recall, F1, and final "
            "confusion-matrix metrics will not "
            "be calculated until all rows have "
            "human ground-truth labels."
        )

        return None, None, evaluation_df


    # --------------------------------------------------------
    # 5. CONVERT LABELS TO INTEGER
    # --------------------------------------------------------

    evaluation_df[
        "human_skill_present"
    ] = (
        evaluation_df[
            "human_skill_present"
        ]
        .astype(int)
    )

    evaluation_df[
        "system_extracted"
    ] = (
        evaluation_df[
            "system_extracted"
        ]
        .astype(int)
    )


    # --------------------------------------------------------
    # 6. CLASSIFY EACH RESULT
    # --------------------------------------------------------

    conditions = [
        (
            (
                evaluation_df[
                    "system_extracted"
                ] == 1
            )
            &
            (
                evaluation_df[
                    "human_skill_present"
                ] == 1
            )
        ),
        (
            (
                evaluation_df[
                    "system_extracted"
                ] == 1
            )
            &
            (
                evaluation_df[
                    "human_skill_present"
                ] == 0
            )
        ),
        (
            (
                evaluation_df[
                    "system_extracted"
                ] == 0
            )
            &
            (
                evaluation_df[
                    "human_skill_present"
                ] == 1
            )
        ),
        (
            (
                evaluation_df[
                    "system_extracted"
                ] == 0
            )
            &
            (
                evaluation_df[
                    "human_skill_present"
                ] == 0
            )
        )
    ]

    labels = [
        "True Positive",
        "False Positive",
        "False Negative",
        "True Negative"
    ]

    evaluation_df[
        "evaluation_result"
    ] = np.select(
        conditions,
        labels,
        default="Unknown"
    )


    # --------------------------------------------------------
    # 7. CONFUSION MATRIX COUNTS
    # --------------------------------------------------------

    tp = int(
        (
            evaluation_df[
                "evaluation_result"
            ] == "True Positive"
        ).sum()
    )

    fp = int(
        (
            evaluation_df[
                "evaluation_result"
            ] == "False Positive"
        ).sum()
    )

    fn = int(
        (
            evaluation_df[
                "evaluation_result"
            ] == "False Negative"
        ).sum()
    )

    tn = int(
        (
            evaluation_df[
                "evaluation_result"
            ] == "True Negative"
        ).sum()
    )


    # --------------------------------------------------------
    # 8. CALCULATE METRICS
    # --------------------------------------------------------

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2
        * precision
        * recall
        / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    accuracy = (
        (tp + tn)
        / (tp + fp + fn + tn)
        if (tp + fp + fn + tn) > 0
        else 0
    )


    # --------------------------------------------------------
    # 9. SUMMARY TABLE
    # --------------------------------------------------------

    metric_summary = pd.DataFrame(
        [
            {
                "metric":
                    "Precision",
                "value":
                    round(
                        precision,
                        4
                    ),
                "percentage":
                    round(
                        precision * 100,
                        2
                    )
            },
            {
                "metric":
                    "Recall",
                "value":
                    round(
                        recall,
                        4
                    ),
                "percentage":
                    round(
                        recall * 100,
                        2
                    )
            },
            {
                "metric":
                    "F1 Score",
                "value":
                    round(
                        f1,
                        4
                    ),
                "percentage":
                    round(
                        f1 * 100,
                        2
                    )
            },
            {
                "metric":
                    "Accuracy",
                "value":
                    round(
                        accuracy,
                        4
                    ),
                "percentage":
                    round(
                        accuracy * 100,
                        2
                    )
            }
        ]
    )


    confusion_summary = pd.DataFrame(
        [
            {
                "category":
                    "True Positive",
                "count":
                    tp
            },
            {
                "category":
                    "False Positive",
                "count":
                    fp
            },
            {
                "category":
                    "False Negative",
                "count":
                    fn
            },
            {
                "category":
                    "True Negative",
                "count":
                    tn
            }
        ]
    )


    # --------------------------------------------------------
    # 10. DISPLAY
    # --------------------------------------------------------

    print(
        "\n✅ Human review complete."
    )

    print(
        "\nSkill-extraction metrics:"
    )

    display(
        metric_summary
    )

    print(
        "\nConfusion matrix counts:"
    )

    display(
        confusion_summary
    )


    return (
        metric_summary,
        confusion_summary,
        evaluation_df
    )


print(
    "✓ Skill-extraction evaluation "
    "function created successfully."
)


# ============================================================
# TEST WITH CURRENT UNREVIEWED TEMPLATE
# ============================================================

(
    skill_extraction_metrics,
    skill_extraction_confusion,
    skill_extraction_evaluated
) = evaluate_skill_extraction(
    skill_extraction_ground_truth
)

Creating skill-extraction evaluation function...
✓ Skill-extraction evaluation function created successfully.

Total evaluation rows: 200
Human-reviewed rows: 0
Unreviewed rows: 200
Invalid human labels: 0

Human-review completion: 0.00%

⚠️ Skill-extraction evaluation is not yet complete.
Precision, Recall, F1, and final confusion-matrix metrics will not be calculated until all rows have human ground-truth labels.


In [153]:
# ============================================================
# CELL 62 — Create Recommendation Ranking Human-Review Template
# ============================================================

print("Creating recommendation-ranking human-review template...")


# ------------------------------------------------------------
# 1. SELECT SAME REVIEW CANDIDATES
# ------------------------------------------------------------

ranking_review_candidates = (
    review_candidates.copy()
)

print(
    "Candidates selected for ranking review:",
    len(ranking_review_candidates)
)


# ------------------------------------------------------------
# 2. GET WEEK 6 TOP-5 RECOMMENDATIONS
# ------------------------------------------------------------

ranking_review_base = (
    final_hybrid[
        final_hybrid[
            "candidate_id"
        ].isin(
            ranking_review_candidates
        )
    ]
    .sort_values(
        [
            "candidate_id",
            "final_hybrid_rank"
        ]
    )
    .groupby(
        "candidate_id",
        group_keys=False
    )
    .head(5)
    .copy()
)


# ------------------------------------------------------------
# 3. CREATE REVIEW TABLE
# ------------------------------------------------------------

recommendation_review = (
    ranking_review_base[
        [
            "candidate_id",
            "selected_occupation",
            "final_hybrid_rank",
            "final_hybrid_score",
            "final_hybrid_percentage",
            "skill_percentage",
            "tfidf_percentage",
            "semantic_percentage",
            "demand_percentage",
            "education_percentage"
        ]
    ]
    .rename(
        columns={
            "selected_occupation":
                "recommended_occupation",
            "final_hybrid_rank":
                "system_rank"
        }
    )
    .copy()
)


# ------------------------------------------------------------
# 4. ADD HUMAN GROUND-TRUTH FIELDS
# ------------------------------------------------------------

recommendation_review[
    "human_relevant"
] = ""

recommendation_review[
    "human_relevance_grade"
] = ""

recommendation_review[
    "reviewer_notes"
] = ""


# ------------------------------------------------------------
# 5. VALIDATE TEMPLATE
# ------------------------------------------------------------

expected_ranking_rows = (
    len(ranking_review_candidates)
    * 5
)

actual_ranking_rows = len(
    recommendation_review
)

duplicate_candidate_rank_rows = int(
    recommendation_review[
        [
            "candidate_id",
            "system_rank"
        ]
    ]
    .duplicated()
    .sum()
)

invalid_system_ranks = int(
    (
        ~recommendation_review[
            "system_rank"
        ]
        .isin(
            [1, 2, 3, 4, 5]
        )
    ).sum()
)


print(
    "\nExpected review rows:",
    expected_ranking_rows
)

print(
    "Actual review rows:",
    actual_ranking_rows
)

print(
    "Unique candidates:",
    recommendation_review[
        "candidate_id"
    ].nunique()
)

print(
    "Duplicate candidate-rank rows:",
    duplicate_candidate_rank_rows
)

print(
    "Invalid system ranks:",
    invalid_system_ranks
)


# ------------------------------------------------------------
# 6. DISPLAY SAMPLE
# ------------------------------------------------------------

print(
    "\nRecommendation-ranking review sample:"
)

display(
    recommendation_review[
        [
            "candidate_id",
            "recommended_occupation",
            "system_rank",
            "final_hybrid_percentage",
            "skill_percentage",
            "semantic_percentage",
            "demand_percentage",
            "education_percentage",
            "human_relevant",
            "human_relevance_grade",
            "reviewer_notes"
        ]
    ]
    .head(20)
)


# ------------------------------------------------------------
# 7. EXPORT TEMPLATE
# ------------------------------------------------------------

recommendation_review_path = (
    WEEK7_OUTPUT_DIR
    / "week7_recommendation_ranking_human_review.csv"
)

recommendation_review.to_csv(
    recommendation_review_path,
    index=False
)


print(
    "\nRecommendation-ranking review template exported:"
)

print(
    recommendation_review_path
)

print(
    "File exists:",
    recommendation_review_path.exists()
)

print(
    "File size:",
    recommendation_review_path.stat().st_size,
    "bytes"
)


# ------------------------------------------------------------
# 8. FINAL VALIDATION
# ------------------------------------------------------------

recommendation_review_template_pass = (
    expected_ranking_rows
    == actual_ranking_rows
    and duplicate_candidate_rank_rows == 0
    and invalid_system_ranks == 0
)

if recommendation_review_template_pass:

    print(
        "\n✅ PASS — Recommendation-ranking "
        "human-review template created successfully."
    )

else:

    print(
        "\n❌ FAIL — Review recommendation-ranking "
        "template construction."
    )


# ------------------------------------------------------------
# 9. HUMAN REVIEW INSTRUCTIONS
# ------------------------------------------------------------

print(
    "\nHUMAN REVIEW INSTRUCTIONS:"
)

print(
    "human_relevant = 1 if the occupation is a reasonable "
    "career recommendation for the candidate; otherwise 0."
)

print(
    "human_relevance_grade should use:"
)

print(
    "0 = Not relevant"
)

print(
    "1 = Somewhat relevant"
)

print(
    "2 = Relevant"
)

print(
    "3 = Highly relevant"
)

print(
    "Do not change system_rank or model scores."
)

print(
    "Top-K accuracy, Precision@K, Recall@K, MRR, and NDCG "
    "will be calculated only after human review is completed."
)

Creating recommendation-ranking human-review template...
Candidates selected for ranking review: 20

Expected review rows: 100
Actual review rows: 100
Unique candidates: 20
Duplicate candidate-rank rows: 0
Invalid system ranks: 0

Recommendation-ranking review sample:


,candidate_id,recommended_occupation,system_rank,final_hybrid_percentage,skill_percentage,semantic_percentage,demand_percentage,education_percentage,human_relevant,human_relevance_grade,reviewer_notes
0,Candidate_001,Software Developer,1,84.58,100.00,100.00,22.91,100.0,,,
1,Candidate_001,Information Technology (IT) Analyst,2,65.30,69.68,73.10,26.64,100.0,,,
2,Candidate_001,Bookkeeper,3,60.34,61.28,65.46,29.92,100.0,,,
3,Candidate_001,Secondary School Teacher,4,59.20,62.29,57.50,36.36,100.0,,,
4,Candidate_001,Office Administrator,5,46.58,43.62,59.26,52.86,0.0,,,
15,Candidate_002,Software Developer,1,78.25,81.90,100.00,22.91,100.0,,,
16,Candidate_002,Office Manager,2,67.13,100.00,33.01,52.86,100.0,,,
17,Candidate_002,Information Technology (IT) Analyst,3,63.42,61.55,75.84,26.64,100.0,,,
18,Candidate_002,Office Administrator,4,62.21,81.98,36.99,52.86,100.0,,,
19,Candidate_002,Secondary School Teacher,5,61.01,78.23,46.73,36.36,100.0,,,



Recommendation-ranking review template exported:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7\week7_recommendation_ranking_human_review.csv
File exists: True
File size: 9859 bytes

✅ PASS — Recommendation-ranking human-review template created successfully.

HUMAN REVIEW INSTRUCTIONS:
human_relevant = 1 if the occupation is a reasonable career recommendation for the candidate; otherwise 0.
human_relevance_grade should use:
0 = Not relevant
1 = Somewhat relevant
2 = Relevant
3 = Highly relevant
Do not change system_rank or model scores.
Top-K accuracy, Precision@K, Recall@K, MRR, and NDCG will be calculated only after human review is completed.


In [155]:
# ============================================================
# CELL 63 — Create Full Recommendation Ranking Ground-Truth Matrix
# ============================================================

print("Creating complete recommendation-ranking ground-truth matrix...")


# ------------------------------------------------------------
# 1. USE SAME 20 REVIEW CANDIDATES
# ------------------------------------------------------------

full_ranking_review_candidates = (
    review_candidates.copy()
)

print(
    "Candidates selected for full ranking review:",
    len(full_ranking_review_candidates)
)


# ------------------------------------------------------------
# 2. GET ALL 15 OCCUPATION RECOMMENDATIONS
# ------------------------------------------------------------

full_ranking_review = (
    final_hybrid[
        final_hybrid[
            "candidate_id"
        ].isin(
            full_ranking_review_candidates
        )
    ]
    .sort_values(
        [
            "candidate_id",
            "final_hybrid_rank"
        ]
    )
    .copy()
)


# ------------------------------------------------------------
# 3. SELECT EVALUATION COLUMNS
# ------------------------------------------------------------

recommendation_ground_truth = (
    full_ranking_review[
        [
            "candidate_id",
            "selected_occupation",
            "final_hybrid_rank",
            "final_hybrid_score",
            "final_hybrid_percentage",
            "skill_percentage",
            "tfidf_percentage",
            "semantic_percentage",
            "demand_percentage",
            "education_percentage"
        ]
    ]
    .rename(
        columns={
            "selected_occupation":
                "recommended_occupation",

            "final_hybrid_rank":
                "system_rank"
        }
    )
    .copy()
)


# ------------------------------------------------------------
# 4. ADD HUMAN GROUND-TRUTH FIELDS
# ------------------------------------------------------------

recommendation_ground_truth[
    "human_relevant"
] = ""

recommendation_ground_truth[
    "human_relevance_grade"
] = ""

recommendation_ground_truth[
    "reviewer_notes"
] = ""


# ------------------------------------------------------------
# 5. VALIDATE COMPLETE MATRIX
# ------------------------------------------------------------

occupation_count = (
    final_hybrid[
        "selected_occupation"
    ]
    .nunique()
)

expected_full_ranking_rows = (
    len(full_ranking_review_candidates)
    * occupation_count
)

actual_full_ranking_rows = len(
    recommendation_ground_truth
)

unique_review_candidates = (
    recommendation_ground_truth[
        "candidate_id"
    ]
    .nunique()
)

unique_review_occupations = (
    recommendation_ground_truth[
        "recommended_occupation"
    ]
    .nunique()
)

duplicate_candidate_occupation_rows = int(
    recommendation_ground_truth[
        [
            "candidate_id",
            "recommended_occupation"
        ]
    ]
    .duplicated()
    .sum()
)

duplicate_candidate_rank_rows = int(
    recommendation_ground_truth[
        [
            "candidate_id",
            "system_rank"
        ]
    ]
    .duplicated()
    .sum()
)

rank_min = (
    recommendation_ground_truth[
        "system_rank"
    ]
    .min()
)

rank_max = (
    recommendation_ground_truth[
        "system_rank"
    ]
    .max()
)


print(
    "\nOccupation count:",
    occupation_count
)

print(
    "Expected review rows:",
    expected_full_ranking_rows
)

print(
    "Actual review rows:",
    actual_full_ranking_rows
)

print(
    "Unique candidates:",
    unique_review_candidates
)

print(
    "Unique occupations:",
    unique_review_occupations
)

print(
    "Duplicate candidate-occupation rows:",
    duplicate_candidate_occupation_rows
)

print(
    "Duplicate candidate-rank rows:",
    duplicate_candidate_rank_rows
)

print(
    "Rank range:",
    rank_min,
    "to",
    rank_max
)


# ------------------------------------------------------------
# 6. VERIFY EACH CANDIDATE HAS ALL OCCUPATIONS
# ------------------------------------------------------------

candidate_occupation_counts = (
    recommendation_ground_truth
    .groupby(
        "candidate_id"
    )
    .size()
)

invalid_candidate_counts = int(
    (
        candidate_occupation_counts
        != occupation_count
    ).sum()
)

print(
    "Candidates without complete occupation set:",
    invalid_candidate_counts
)


# ------------------------------------------------------------
# 7. DISPLAY FIRST CANDIDATE'S FULL RANKING
# ------------------------------------------------------------

first_review_candidate = (
    full_ranking_review_candidates[0]
)

print(
    "\nFull ranking sample for:",
    first_review_candidate
)

display(
    recommendation_ground_truth[
        recommendation_ground_truth[
            "candidate_id"
        ] == first_review_candidate
    ][
        [
            "candidate_id",
            "recommended_occupation",
            "system_rank",
            "final_hybrid_percentage",
            "skill_percentage",
            "semantic_percentage",
            "demand_percentage",
            "education_percentage",
            "human_relevant",
            "human_relevance_grade"
        ]
    ]
)


# ------------------------------------------------------------
# 8. EXPORT COMPLETE REVIEW TEMPLATE
# ------------------------------------------------------------

recommendation_ground_truth_path = (
    WEEK7_OUTPUT_DIR
    / "week7_recommendation_ranking_ground_truth_review.csv"
)

recommendation_ground_truth.to_csv(
    recommendation_ground_truth_path,
    index=False
)


print(
    "\nComplete ranking ground-truth template exported:"
)

print(
    recommendation_ground_truth_path
)

print(
    "File exists:",
    recommendation_ground_truth_path.exists()
)

print(
    "File size:",
    recommendation_ground_truth_path.stat().st_size,
    "bytes"
)


# ------------------------------------------------------------
# 9. FINAL VALIDATION
# ------------------------------------------------------------

full_ranking_template_pass = (
    expected_full_ranking_rows
    == actual_full_ranking_rows

    and unique_review_candidates
    == len(full_ranking_review_candidates)

    and unique_review_occupations
    == occupation_count

    and duplicate_candidate_occupation_rows
    == 0

    and duplicate_candidate_rank_rows
    == 0

    and invalid_candidate_counts
    == 0

    and rank_min
    == 1

    and rank_max
    == occupation_count
)


if full_ranking_template_pass:

    print(
        "\n✅ PASS — Complete recommendation-ranking "
        "ground-truth matrix created successfully."
    )

else:

    print(
        "\n❌ FAIL — Review complete recommendation-ranking "
        "matrix construction."
    )


# ------------------------------------------------------------
# 10. HUMAN REVIEW INSTRUCTIONS
# ------------------------------------------------------------

print(
    "\nHUMAN REVIEW INSTRUCTIONS:"
)

print(
    "Review all occupations for each candidate."
)

print(
    "human_relevant = 1 if the occupation is a reasonable "
    "career match; otherwise 0."
)

print(
    "human_relevance_grade:"
)

print(
    "0 = Not relevant"
)

print(
    "1 = Somewhat relevant"
)

print(
    "2 = Relevant"
)

print(
    "3 = Highly relevant"
)

print(
    "Do not modify system_rank or model-generated scores."
)

print(
    "This complete matrix will support Top-1, Top-3, Top-5, "
    "Precision@K, Recall@K, MRR, and NDCG evaluation."
)

Creating complete recommendation-ranking ground-truth matrix...
Candidates selected for full ranking review: 20

Occupation count: 15
Expected review rows: 300
Actual review rows: 300
Unique candidates: 20
Unique occupations: 15
Duplicate candidate-occupation rows: 0
Duplicate candidate-rank rows: 0
Rank range: 1 to 15
Candidates without complete occupation set: 0

Full ranking sample for: Candidate_001


,candidate_id,recommended_occupation,system_rank,final_hybrid_percentage,skill_percentage,semantic_percentage,demand_percentage,education_percentage,human_relevant,human_relevance_grade
0,Candidate_001,Software Developer,1,84.58,100.00,100.00,22.91,100.0,,
1,Candidate_001,Information Technology (IT) Analyst,2,65.30,69.68,73.10,26.64,100.0,,
2,Candidate_001,Bookkeeper,3,60.34,61.28,65.46,29.92,100.0,,
3,Candidate_001,Secondary School Teacher,4,59.20,62.29,57.50,36.36,100.0,,
4,Candidate_001,Office Administrator,5,46.58,43.62,59.26,52.86,0.0,,
5,Candidate_001,Office Manager,6,43.95,41.54,53.82,52.86,0.0,,
6,Candidate_001,Licensed Practical Nurse (L.P.N.),7,41.04,51.54,19.77,30.39,100.0,,
7,Candidate_001,Restaurant Manager,8,37.48,26.80,46.77,58.65,0.0,,
8,Candidate_001,"Driver, Truck",9,34.68,25.32,23.01,88.80,0.0,,
9,Candidate_001,Food Service Supervisor,10,34.48,3.64,55.82,68.33,0.0,,



Complete ranking ground-truth template exported:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7\week7_recommendation_ranking_ground_truth_review.csv
File exists: True
File size: 28761 bytes

✅ PASS — Complete recommendation-ranking ground-truth matrix created successfully.

HUMAN REVIEW INSTRUCTIONS:
Review all occupations for each candidate.
human_relevant = 1 if the occupation is a reasonable career match; otherwise 0.
human_relevance_grade:
0 = Not relevant
1 = Somewhat relevant
2 = Relevant
3 = Highly relevant
Do not modify system_rank or model-generated scores.
This complete matrix will support Top-1, Top-3, Top-5, Precision@K, Recall@K, MRR, and NDCG evaluation.


In [157]:
# ============================================================
# CELL 64 — Recommendation Ranking Evaluation Function
# ============================================================

print("Creating recommendation-ranking evaluation function...")


def evaluate_recommendation_ranking(
    review_df,
    k_values=(1, 3, 5)
):
    """
    Evaluate recommendation ranking using human ground truth.

    Metrics:
    - Top-1 hit rate
    - Top-3 hit rate
    - Top-5 hit rate
    - Precision@K
    - Recall@K
    - MRR
    - NDCG@K

    Metrics are calculated only after all human labels are complete.
    """

    evaluation_df = review_df.copy()


    # --------------------------------------------------------
    # 1. VALIDATE REQUIRED COLUMNS
    # --------------------------------------------------------

    required_columns = [
        "candidate_id",
        "recommended_occupation",
        "system_rank",
        "human_relevant",
        "human_relevance_grade"
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in evaluation_df.columns
    ]

    if missing_columns:

        print(
            "❌ Missing required columns:",
            missing_columns
        )

        return None, None, None


    # --------------------------------------------------------
    # 2. CLEAN HUMAN LABELS
    # --------------------------------------------------------

    evaluation_df[
        "human_relevant"
    ] = pd.to_numeric(
        evaluation_df[
            "human_relevant"
        ],
        errors="coerce"
    )

    evaluation_df[
        "human_relevance_grade"
    ] = pd.to_numeric(
        evaluation_df[
            "human_relevance_grade"
        ],
        errors="coerce"
    )


    total_rows = len(
        evaluation_df
    )

    reviewed_binary_rows = int(
        evaluation_df[
            "human_relevant"
        ]
        .notna()
        .sum()
    )

    reviewed_grade_rows = int(
        evaluation_df[
            "human_relevance_grade"
        ]
        .notna()
        .sum()
    )

    unreviewed_binary_rows = (
        total_rows
        - reviewed_binary_rows
    )

    unreviewed_grade_rows = (
        total_rows
        - reviewed_grade_rows
    )


    print(
        "\nTotal ranking rows:",
        total_rows
    )

    print(
        "Human relevance labels completed:",
        reviewed_binary_rows
    )

    print(
        "Human relevance labels missing:",
        unreviewed_binary_rows
    )

    print(
        "Human relevance grades completed:",
        reviewed_grade_rows
    )

    print(
        "Human relevance grades missing:",
        unreviewed_grade_rows
    )


    # --------------------------------------------------------
    # 3. VALIDATE HUMAN LABEL VALUES
    # --------------------------------------------------------

    binary_values = (
        evaluation_df.loc[
            evaluation_df[
                "human_relevant"
            ].notna(),
            "human_relevant"
        ]
    )

    grade_values = (
        evaluation_df.loc[
            evaluation_df[
                "human_relevance_grade"
            ].notna(),
            "human_relevance_grade"
        ]
    )


    invalid_binary_labels = int(
        (
            ~binary_values.isin(
                [0, 1]
            )
        ).sum()
    )

    invalid_grade_labels = int(
        (
            ~grade_values.isin(
                [0, 1, 2, 3]
            )
        ).sum()
    )


    print(
        "Invalid binary relevance labels:",
        invalid_binary_labels
    )

    print(
        "Invalid graded relevance labels:",
        invalid_grade_labels
    )


    if (
        invalid_binary_labels > 0
        or invalid_grade_labels > 0
    ):

        print(
            "\n❌ Invalid human-review labels detected."
        )

        print(
            "human_relevant must contain only 0 or 1."
        )

        print(
            "human_relevance_grade must contain only "
            "0, 1, 2, or 3."
        )

        return None, None, evaluation_df


    # --------------------------------------------------------
    # 4. STOP IF HUMAN REVIEW IS INCOMPLETE
    # --------------------------------------------------------

    if (
        unreviewed_binary_rows > 0
        or unreviewed_grade_rows > 0
    ):

        binary_completion = (
            reviewed_binary_rows
            / total_rows
            * 100
        )

        grade_completion = (
            reviewed_grade_rows
            / total_rows
            * 100
        )

        print(
            "\nBinary-label review completion:",
            f"{binary_completion:.2f}%"
        )

        print(
            "Graded-label review completion:",
            f"{grade_completion:.2f}%"
        )

        print(
            "\n⚠️ Recommendation-ranking evaluation "
            "is not yet complete."
        )

        print(
            "Top-K, Precision@K, Recall@K, MRR, and "
            "NDCG will not be calculated until all "
            "human-review labels are completed."
        )

        return None, None, evaluation_df


    # --------------------------------------------------------
    # 5. CONVERT LABELS TO INTEGER
    # --------------------------------------------------------

    evaluation_df[
        "human_relevant"
    ] = (
        evaluation_df[
            "human_relevant"
        ]
        .astype(int)
    )

    evaluation_df[
        "human_relevance_grade"
    ] = (
        evaluation_df[
            "human_relevance_grade"
        ]
        .astype(int)
    )

    evaluation_df[
        "system_rank"
    ] = (
        evaluation_df[
            "system_rank"
        ]
        .astype(int)
    )


    # --------------------------------------------------------
    # 6. PER-CANDIDATE METRICS
    # --------------------------------------------------------

    candidate_metric_records = []

    for candidate_id, group in (
        evaluation_df
        .groupby(
            "candidate_id"
        )
    ):

        group = (
            group
            .sort_values(
                "system_rank"
            )
            .copy()
        )

        total_relevant = int(
            group[
                "human_relevant"
            ].sum()
        )


        candidate_result = {
            "candidate_id":
                candidate_id,

            "total_relevant_occupations":
                total_relevant
        }


        # ----------------------------------------------------
        # TOP-K, PRECISION@K, RECALL@K
        # ----------------------------------------------------

        for k in k_values:

            top_k = (
                group[
                    group[
                        "system_rank"
                    ] <= k
                ]
            )

            relevant_in_top_k = int(
                top_k[
                    "human_relevant"
                ].sum()
            )

            candidate_result[
                f"top_{k}_hit"
            ] = int(
                relevant_in_top_k > 0
            )

            candidate_result[
                f"precision_at_{k}"
            ] = (
                relevant_in_top_k
                / k
            )

            candidate_result[
                f"recall_at_{k}"
            ] = (
                relevant_in_top_k
                / total_relevant
                if total_relevant > 0
                else 0
            )


        # ----------------------------------------------------
        # MRR
        # ----------------------------------------------------

        relevant_rows = (
            group[
                group[
                    "human_relevant"
                ] == 1
            ]
        )

        if not relevant_rows.empty:

            first_relevant_rank = int(
                relevant_rows[
                    "system_rank"
                ].min()
            )

            reciprocal_rank = (
                1
                / first_relevant_rank
            )

        else:

            first_relevant_rank = np.nan
            reciprocal_rank = 0


        candidate_result[
            "first_relevant_rank"
        ] = first_relevant_rank

        candidate_result[
            "reciprocal_rank"
        ] = reciprocal_rank


        # ----------------------------------------------------
        # NDCG@K USING GRADED RELEVANCE
        # ----------------------------------------------------

        for k in k_values:

            top_k = (
                group[
                    group[
                        "system_rank"
                    ] <= k
                ]
                .sort_values(
                    "system_rank"
                )
            )

            relevance_scores = (
                top_k[
                    "human_relevance_grade"
                ]
                .tolist()
            )

            dcg = 0

            for position, relevance in enumerate(
                relevance_scores,
                start=1
            ):

                gain = (
                    (2 ** relevance)
                    - 1
                )

                discount = np.log2(
                    position + 1
                )

                dcg += (
                    gain
                    / discount
                )


            ideal_scores = (
                group[
                    "human_relevance_grade"
                ]
                .sort_values(
                    ascending=False
                )
                .head(k)
                .tolist()
            )

            idcg = 0

            for position, relevance in enumerate(
                ideal_scores,
                start=1
            ):

                gain = (
                    (2 ** relevance)
                    - 1
                )

                discount = np.log2(
                    position + 1
                )

                idcg += (
                    gain
                    / discount
                )


            ndcg = (
                dcg / idcg
                if idcg > 0
                else 0
            )

            candidate_result[
                f"ndcg_at_{k}"
            ] = ndcg


        candidate_metric_records.append(
            candidate_result
        )


    candidate_ranking_metrics = (
        pd.DataFrame(
            candidate_metric_records
        )
    )


    # --------------------------------------------------------
    # 7. AGGREGATE METRICS
    # --------------------------------------------------------

    summary_records = []


    for k in k_values:

        summary_records.append(
            {
                "metric":
                    f"Top-{k} Hit Rate",

                "value":
                    candidate_ranking_metrics[
                        f"top_{k}_hit"
                    ].mean(),

                "percentage":
                    candidate_ranking_metrics[
                        f"top_{k}_hit"
                    ].mean()
                    * 100
            }
        )

        summary_records.append(
            {
                "metric":
                    f"Precision@{k}",

                "value":
                    candidate_ranking_metrics[
                        f"precision_at_{k}"
                    ].mean(),

                "percentage":
                    candidate_ranking_metrics[
                        f"precision_at_{k}"
                    ].mean()
                    * 100
            }
        )

        summary_records.append(
            {
                "metric":
                    f"Recall@{k}",

                "value":
                    candidate_ranking_metrics[
                        f"recall_at_{k}"
                    ].mean(),

                "percentage":
                    candidate_ranking_metrics[
                        f"recall_at_{k}"
                    ].mean()
                    * 100
            }
        )

        summary_records.append(
            {
                "metric":
                    f"NDCG@{k}",

                "value":
                    candidate_ranking_metrics[
                        f"ndcg_at_{k}"
                    ].mean(),

                "percentage":
                    candidate_ranking_metrics[
                        f"ndcg_at_{k}"
                    ].mean()
                    * 100
            }
        )


    mean_reciprocal_rank = (
        candidate_ranking_metrics[
            "reciprocal_rank"
        ].mean()
    )

    summary_records.append(
        {
            "metric":
                "MRR",

            "value":
                mean_reciprocal_rank,

            "percentage":
                mean_reciprocal_rank
                * 100
        }
    )


    recommendation_ranking_summary = (
        pd.DataFrame(
            summary_records
        )
    )

    recommendation_ranking_summary[
        "value"
    ] = (
        recommendation_ranking_summary[
            "value"
        ]
        .round(4)
    )

    recommendation_ranking_summary[
        "percentage"
    ] = (
        recommendation_ranking_summary[
            "percentage"
        ]
        .round(2)
    )


    # --------------------------------------------------------
    # 8. DISPLAY RESULTS
    # --------------------------------------------------------

    print(
        "\n✅ Human ranking review complete."
    )

    print(
        "\nRecommendation-ranking evaluation:"
    )

    display(
        recommendation_ranking_summary
    )

    print(
        "\nCandidate-level ranking metrics:"
    )

    display(
        candidate_ranking_metrics.head(10)
    )


    return (
        recommendation_ranking_summary,
        candidate_ranking_metrics,
        evaluation_df
    )


print(
    "✓ Recommendation-ranking evaluation "
    "function created successfully."
)


# ============================================================
# TEST USING CURRENT UNREVIEWED TEMPLATE
# ============================================================

(
    recommendation_ranking_metrics,
    candidate_ranking_evaluation,
    recommendation_ranking_evaluated
) = evaluate_recommendation_ranking(
    recommendation_ground_truth
)

Creating recommendation-ranking evaluation function...
✓ Recommendation-ranking evaluation function created successfully.

Total ranking rows: 300
Human relevance labels completed: 0
Human relevance labels missing: 300
Human relevance grades completed: 0
Human relevance grades missing: 300
Invalid binary relevance labels: 0
Invalid graded relevance labels: 0

Binary-label review completion: 0.00%
Graded-label review completion: 0.00%

⚠️ Recommendation-ranking evaluation is not yet complete.
Top-K, Precision@K, Recall@K, MRR, and NDCG will not be calculated until all human-review labels are completed.


In [159]:
# ============================================================
# CELL 65 — Create Skill-Gap Human-Review Ground-Truth Matrix
# CORRECTED VERSION
# ============================================================

print("Creating skill-gap human-review ground-truth matrix...")


# ------------------------------------------------------------
# 1. USE SAME 20 REVIEW CANDIDATES
# ------------------------------------------------------------

skill_gap_review_candidates = (
    review_candidates.copy()
)

print(
    "Candidates selected for skill-gap review:",
    len(skill_gap_review_candidates)
)


# ------------------------------------------------------------
# 2. GET EACH CANDIDATE'S WEEK 6 TOP-1 OCCUPATION
# ------------------------------------------------------------

top1_review = (
    final_hybrid[
        final_hybrid[
            "candidate_id"
        ].isin(
            skill_gap_review_candidates
        )
    ]
    .sort_values(
        [
            "candidate_id",
            "final_hybrid_rank"
        ]
    )
    .groupby(
        "candidate_id",
        group_keys=False
    )
    .head(1)
    [
        [
            "candidate_id",
            "selected_occupation",
            "final_hybrid_percentage"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. GET TOP-1 OCCUPATION SKILL-GAP ROWS
# ------------------------------------------------------------

skill_gap_review = (
    skill_gap_base
    .merge(
        top1_review,
        on=[
            "candidate_id",
            "selected_occupation"
        ],
        how="inner"
    )
    .copy()
)


# ------------------------------------------------------------
# 4. STANDARDIZE SYSTEM GAP LABEL
# gap_status contains Existing / Missing
# ------------------------------------------------------------

skill_gap_review[
    "system_gap"
] = (
    skill_gap_review[
        "gap_status"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("missing")
    .astype(int)
)


# ------------------------------------------------------------
# 5. CREATE HUMAN REVIEW FIELDS
# ------------------------------------------------------------

skill_gap_review[
    "human_gap"
] = ""

skill_gap_review[
    "human_gap_severity"
] = ""

skill_gap_review[
    "expert_rating"
] = ""

skill_gap_review[
    "reviewer_notes"
] = ""


# ------------------------------------------------------------
# 6. SELECT FINAL REVIEW COLUMNS
# ------------------------------------------------------------

skill_gap_ground_truth = (
    skill_gap_review[
        [
            "candidate_id",
            "selected_occupation",
            "final_hybrid_percentage",
            "onet_soc_code",
            "skill",
            "candidate_evidence",
            "skill_present",
            "gap_status",
            "importance",
            "level",
            "onet_gap_priority",
            "skill_gap_priority",
            "system_gap",
            "human_gap",
            "human_gap_severity",
            "expert_rating",
            "reviewer_notes"
        ]
    ]
    .sort_values(
        [
            "candidate_id",
            "skill"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 7. VALIDATE MATRIX
# ------------------------------------------------------------

skill_count = (
    skill_gap_ground_truth[
        "skill"
    ]
    .nunique()
)

expected_skill_gap_rows = (
    len(skill_gap_review_candidates)
    * skill_count
)

actual_skill_gap_rows = len(
    skill_gap_ground_truth
)

unique_skill_gap_candidates = (
    skill_gap_ground_truth[
        "candidate_id"
    ]
    .nunique()
)

duplicate_candidate_skill_rows = int(
    skill_gap_ground_truth[
        [
            "candidate_id",
            "skill"
        ]
    ]
    .duplicated()
    .sum()
)

system_missing_count = int(
    skill_gap_ground_truth[
        "system_gap"
    ]
    .sum()
)

system_existing_count = int(
    (
        skill_gap_ground_truth[
            "system_gap"
        ] == 0
    ).sum()
)

invalid_gap_status_count = int(
    (
        ~skill_gap_ground_truth[
            "gap_status"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            [
                "existing",
                "missing"
            ]
        )
    ).sum()
)


print(
    "\nSkill dimensions:",
    skill_count
)

print(
    "Expected review rows:",
    expected_skill_gap_rows
)

print(
    "Actual review rows:",
    actual_skill_gap_rows
)

print(
    "Unique candidates:",
    unique_skill_gap_candidates
)

print(
    "Duplicate candidate-skill rows:",
    duplicate_candidate_skill_rows
)

print(
    "System Missing rows:",
    system_missing_count
)

print(
    "System Existing rows:",
    system_existing_count
)

print(
    "Invalid gap-status rows:",
    invalid_gap_status_count
)


# ------------------------------------------------------------
# 8. DISPLAY SAMPLE
# ------------------------------------------------------------

print(
    "\nSkill-gap review sample:"
)

display(
    skill_gap_ground_truth[
        [
            "candidate_id",
            "selected_occupation",
            "skill",
            "candidate_evidence",
            "skill_present",
            "gap_status",
            "importance",
            "level",
            "skill_gap_priority",
            "system_gap",
            "human_gap",
            "human_gap_severity",
            "expert_rating",
            "reviewer_notes"
        ]
    ]
    .head(25)
)


# ------------------------------------------------------------
# 9. EXPORT TEMPLATE
# ------------------------------------------------------------

skill_gap_review_path = (
    WEEK7_OUTPUT_DIR
    / "week7_skill_gap_ground_truth_review.csv"
)

skill_gap_ground_truth.to_csv(
    skill_gap_review_path,
    index=False
)


print(
    "\nSkill-gap ground-truth template exported:"
)

print(
    skill_gap_review_path
)

print(
    "File exists:",
    skill_gap_review_path.exists()
)

print(
    "File size:",
    skill_gap_review_path.stat().st_size,
    "bytes"
)


# ------------------------------------------------------------
# 10. FINAL VALIDATION
# ------------------------------------------------------------

skill_gap_review_pass = (
    expected_skill_gap_rows
    == actual_skill_gap_rows

    and unique_skill_gap_candidates
    == len(skill_gap_review_candidates)

    and duplicate_candidate_skill_rows
    == 0

    and (
        system_missing_count
        + system_existing_count
    )
    == actual_skill_gap_rows

    and invalid_gap_status_count
    == 0
)


if skill_gap_review_pass:

    print(
        "\n✅ PASS — Skill-gap human-review "
        "ground-truth matrix created successfully."
    )

else:

    print(
        "\n❌ FAIL — Review skill-gap "
        "ground-truth matrix construction."
    )


# ------------------------------------------------------------
# 11. HUMAN REVIEW INSTRUCTIONS
# ------------------------------------------------------------

print(
    "\nHUMAN REVIEW INSTRUCTIONS:"
)

print(
    "human_gap = 1 if the candidate genuinely lacks "
    "the required skill for the recommended occupation; "
    "otherwise enter 0."
)

print(
    "human_gap_severity:"
)

print(
    "0 = No gap"
)

print(
    "1 = Minor gap"
)

print(
    "2 = Moderate gap"
)

print(
    "3 = Major gap"
)

print(
    "expert_rating:"
)

print(
    "1 = Very poor system gap assessment"
)

print(
    "2 = Poor"
)

print(
    "3 = Acceptable"
)

print(
    "4 = Good"
)

print(
    "5 = Excellent"
)

print(
    "Do not modify gap_status, system_gap, "
    "skill_present, or skill_gap_priority."
)

print(
    "Precision, Recall, F1, agreement percentage, "
    "and expert-rating metrics will be calculated "
    "only after human review is completed."
)

Creating skill-gap human-review ground-truth matrix...
Candidates selected for skill-gap review: 20

Skill dimensions: 10
Expected review rows: 200
Actual review rows: 200
Unique candidates: 20
Duplicate candidate-skill rows: 0
System Missing rows: 57
System Existing rows: 143
Invalid gap-status rows: 0

Skill-gap review sample:


,candidate_id,selected_occupation,skill,candidate_evidence,skill_present,gap_status,importance,level,skill_gap_priority,system_gap,human_gap,human_gap_severity,expert_rating,reviewer_notes
0,Candidate_001,Software Developer,Active Learning,1.0,1,Existing,3.50,3.62,0.000,0,,,,
1,Candidate_001,Software Developer,Active Listening,0.0,0,Missing,3.38,3.88,0.726,1,,,,
2,Candidate_001,Software Developer,Critical Thinking,1.0,1,Existing,3.88,4.12,0.000,0,,,,
3,Candidate_001,Software Developer,Learning Strategies,1.0,1,Existing,2.62,3.12,0.000,0,,,,
4,Candidate_001,Software Developer,Mathematics,1.0,1,Existing,2.75,3.25,0.000,0,,,,
5,Candidate_001,Software Developer,Monitoring,0.0,0,Missing,3.00,3.50,0.650,1,,,,
6,Candidate_001,Software Developer,Reading Comprehension,1.0,1,Existing,3.50,4.25,0.000,0,,,,
7,Candidate_001,Software Developer,Science,1.0,1,Existing,2.12,1.88,0.000,0,,,,
8,Candidate_001,Software Developer,Speaking,0.0,0,Missing,3.12,3.62,0.674,1,,,,
9,Candidate_001,Software Developer,Writing,1.0,1,Existing,3.25,3.62,0.000,0,,,,



Skill-gap ground-truth template exported:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7\week7_skill_gap_ground_truth_review.csv
File exists: True
File size: 22586 bytes

✅ PASS — Skill-gap human-review ground-truth matrix created successfully.

HUMAN REVIEW INSTRUCTIONS:
human_gap = 1 if the candidate genuinely lacks the required skill for the recommended occupation; otherwise enter 0.
human_gap_severity:
0 = No gap
1 = Minor gap
2 = Moderate gap
3 = Major gap
expert_rating:
1 = Very poor system gap assessment
2 = Poor
3 = Acceptable
4 = Good
5 = Excellent
Do not modify gap_status, system_gap, skill_present, or skill_gap_priority.
Precision, Recall, F1, agreement percentage, and expert-rating metrics will be calculated only after human review is completed.


In [161]:
# ============================================================
# CELL 66 — Skill-Gap Evaluation Function
# ============================================================

print("Creating skill-gap evaluation function...")


def evaluate_skill_gap(
    review_df
):
    """
    Evaluate system-generated skill gaps against
    human-reviewed ground truth.

    Metrics:
    - Precision
    - Recall
    - F1 Score
    - Accuracy
    - Percentage Agreement
    - True Positives
    - False Positives
    - False Negatives
    - True Negatives
    - Mean Expert Rating

    Final metrics are calculated only after
    human review is complete.
    """

    evaluation_df = (
        review_df.copy()
    )


    # --------------------------------------------------------
    # 1. VALIDATE REQUIRED COLUMNS
    # --------------------------------------------------------

    required_columns = [
        "candidate_id",
        "selected_occupation",
        "skill",
        "system_gap",
        "human_gap",
        "human_gap_severity",
        "expert_rating"
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in evaluation_df.columns
    ]

    if missing_columns:

        print(
            "❌ Missing required columns:",
            missing_columns
        )

        return None, None, None


    # --------------------------------------------------------
    # 2. CLEAN HUMAN REVIEW FIELDS
    # --------------------------------------------------------

    evaluation_df[
        "human_gap"
    ] = pd.to_numeric(
        evaluation_df[
            "human_gap"
        ],
        errors="coerce"
    )

    evaluation_df[
        "human_gap_severity"
    ] = pd.to_numeric(
        evaluation_df[
            "human_gap_severity"
        ],
        errors="coerce"
    )

    evaluation_df[
        "expert_rating"
    ] = pd.to_numeric(
        evaluation_df[
            "expert_rating"
        ],
        errors="coerce"
    )


    total_rows = len(
        evaluation_df
    )

    reviewed_gap_rows = int(
        evaluation_df[
            "human_gap"
        ]
        .notna()
        .sum()
    )

    reviewed_severity_rows = int(
        evaluation_df[
            "human_gap_severity"
        ]
        .notna()
        .sum()
    )

    reviewed_expert_rows = int(
        evaluation_df[
            "expert_rating"
        ]
        .notna()
        .sum()
    )


    unreviewed_gap_rows = (
        total_rows
        - reviewed_gap_rows
    )

    unreviewed_severity_rows = (
        total_rows
        - reviewed_severity_rows
    )

    unreviewed_expert_rows = (
        total_rows
        - reviewed_expert_rows
    )


    print(
        "\nTotal skill-gap rows:",
        total_rows
    )

    print(
        "Human gap labels completed:",
        reviewed_gap_rows
    )

    print(
        "Human gap labels missing:",
        unreviewed_gap_rows
    )

    print(
        "Gap severity ratings completed:",
        reviewed_severity_rows
    )

    print(
        "Gap severity ratings missing:",
        unreviewed_severity_rows
    )

    print(
        "Expert ratings completed:",
        reviewed_expert_rows
    )

    print(
        "Expert ratings missing:",
        unreviewed_expert_rows
    )


    # --------------------------------------------------------
    # 3. VALIDATE HUMAN LABEL VALUES
    # --------------------------------------------------------

    reviewed_gap_values = (
        evaluation_df.loc[
            evaluation_df[
                "human_gap"
            ].notna(),
            "human_gap"
        ]
    )

    reviewed_severity_values = (
        evaluation_df.loc[
            evaluation_df[
                "human_gap_severity"
            ].notna(),
            "human_gap_severity"
        ]
    )

    reviewed_expert_values = (
        evaluation_df.loc[
            evaluation_df[
                "expert_rating"
            ].notna(),
            "expert_rating"
        ]
    )


    invalid_gap_labels = int(
        (
            ~reviewed_gap_values.isin(
                [0, 1]
            )
        ).sum()
    )

    invalid_severity_labels = int(
        (
            ~reviewed_severity_values.isin(
                [0, 1, 2, 3]
            )
        ).sum()
    )

    invalid_expert_ratings = int(
        (
            ~reviewed_expert_values.isin(
                [1, 2, 3, 4, 5]
            )
        ).sum()
    )


    print(
        "Invalid human-gap labels:",
        invalid_gap_labels
    )

    print(
        "Invalid severity ratings:",
        invalid_severity_labels
    )

    print(
        "Invalid expert ratings:",
        invalid_expert_ratings
    )


    if (
        invalid_gap_labels > 0
        or invalid_severity_labels > 0
        or invalid_expert_ratings > 0
    ):

        print(
            "\n❌ Invalid human-review values detected."
        )

        print(
            "human_gap must contain only 0 or 1."
        )

        print(
            "human_gap_severity must contain only "
            "0, 1, 2, or 3."
        )

        print(
            "expert_rating must contain only "
            "1, 2, 3, 4, or 5."
        )

        return None, None, evaluation_df


    # --------------------------------------------------------
    # 4. STOP IF REVIEW IS INCOMPLETE
    # --------------------------------------------------------

    if (
        unreviewed_gap_rows > 0
        or unreviewed_severity_rows > 0
        or unreviewed_expert_rows > 0
    ):

        gap_completion = (
            reviewed_gap_rows
            / total_rows
            * 100
        )

        severity_completion = (
            reviewed_severity_rows
            / total_rows
            * 100
        )

        expert_completion = (
            reviewed_expert_rows
            / total_rows
            * 100
        )

        print(
            "\nHuman gap-label completion:",
            f"{gap_completion:.2f}%"
        )

        print(
            "Gap-severity completion:",
            f"{severity_completion:.2f}%"
        )

        print(
            "Expert-rating completion:",
            f"{expert_completion:.2f}%"
        )

        print(
            "\n⚠️ Skill-gap evaluation is not yet complete."
        )

        print(
            "Precision, Recall, F1, agreement, and "
            "expert-rating metrics will not be calculated "
            "until all human-review fields are completed."
        )

        return None, None, evaluation_df


    # --------------------------------------------------------
    # 5. CONVERT VALUES TO INTEGER
    # --------------------------------------------------------

    evaluation_df[
        "system_gap"
    ] = (
        evaluation_df[
            "system_gap"
        ]
        .astype(int)
    )

    evaluation_df[
        "human_gap"
    ] = (
        evaluation_df[
            "human_gap"
        ]
        .astype(int)
    )

    evaluation_df[
        "human_gap_severity"
    ] = (
        evaluation_df[
            "human_gap_severity"
        ]
        .astype(int)
    )

    evaluation_df[
        "expert_rating"
    ] = (
        evaluation_df[
            "expert_rating"
        ]
        .astype(int)
    )


    # --------------------------------------------------------
    # 6. CLASSIFY CONFUSION-MATRIX RESULT
    # --------------------------------------------------------

    conditions = [
        (
            (evaluation_df["system_gap"] == 1)
            &
            (evaluation_df["human_gap"] == 1)
        ),
        (
            (evaluation_df["system_gap"] == 1)
            &
            (evaluation_df["human_gap"] == 0)
        ),
        (
            (evaluation_df["system_gap"] == 0)
            &
            (evaluation_df["human_gap"] == 1)
        ),
        (
            (evaluation_df["system_gap"] == 0)
            &
            (evaluation_df["human_gap"] == 0)
        )
    ]

    labels = [
        "True Positive",
        "False Positive",
        "False Negative",
        "True Negative"
    ]

    evaluation_df[
        "evaluation_result"
    ] = np.select(
        conditions,
        labels,
        default="Unknown"
    )


    # --------------------------------------------------------
    # 7. CONFUSION-MATRIX COUNTS
    # --------------------------------------------------------

    tp = int(
        (
            evaluation_df[
                "evaluation_result"
            ] == "True Positive"
        ).sum()
    )

    fp = int(
        (
            evaluation_df[
                "evaluation_result"
            ] == "False Positive"
        ).sum()
    )

    fn = int(
        (
            evaluation_df[
                "evaluation_result"
            ] == "False Negative"
        ).sum()
    )

    tn = int(
        (
            evaluation_df[
                "evaluation_result"
            ] == "True Negative"
        ).sum()
    )


    # --------------------------------------------------------
    # 8. CALCULATE PERFORMANCE METRICS
    # --------------------------------------------------------

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    accuracy = (
        (tp + tn)
        / (tp + fp + fn + tn)
        if (tp + fp + fn + tn) > 0
        else 0
    )

    agreement_percentage = (
        (
            evaluation_df[
                "system_gap"
            ]
            ==
            evaluation_df[
                "human_gap"
            ]
        )
        .mean()
        * 100
    )

    mean_expert_rating = (
        evaluation_df[
            "expert_rating"
        ]
        .mean()
    )

    mean_gap_severity = (
        evaluation_df[
            "human_gap_severity"
        ]
        .mean()
    )


    # --------------------------------------------------------
    # 9. CREATE SUMMARY TABLE
    # --------------------------------------------------------

    skill_gap_metric_summary = pd.DataFrame(
        [
            {
                "metric": "Precision",
                "value": precision,
                "percentage": precision * 100
            },
            {
                "metric": "Recall",
                "value": recall,
                "percentage": recall * 100
            },
            {
                "metric": "F1 Score",
                "value": f1,
                "percentage": f1 * 100
            },
            {
                "metric": "Accuracy",
                "value": accuracy,
                "percentage": accuracy * 100
            },
            {
                "metric": "Agreement",
                "value": agreement_percentage / 100,
                "percentage": agreement_percentage
            },
            {
                "metric": "Mean Expert Rating",
                "value": mean_expert_rating,
                "percentage": np.nan
            },
            {
                "metric": "Mean Human Gap Severity",
                "value": mean_gap_severity,
                "percentage": np.nan
            }
        ]
    )


    skill_gap_metric_summary[
        "value"
    ] = (
        skill_gap_metric_summary[
            "value"
        ]
        .round(4)
    )

    skill_gap_metric_summary[
        "percentage"
    ] = (
        skill_gap_metric_summary[
            "percentage"
        ]
        .round(2)
    )


    # --------------------------------------------------------
    # 10. CONFUSION SUMMARY
    # --------------------------------------------------------

    skill_gap_confusion_summary = pd.DataFrame(
        [
            {
                "category": "True Positive",
                "count": tp
            },
            {
                "category": "False Positive",
                "count": fp
            },
            {
                "category": "False Negative",
                "count": fn
            },
            {
                "category": "True Negative",
                "count": tn
            }
        ]
    )


    # --------------------------------------------------------
    # 11. DISPLAY RESULTS
    # --------------------------------------------------------

    print(
        "\n✅ Human skill-gap review complete."
    )

    print(
        "\nSkill-gap evaluation metrics:"
    )

    display(
        skill_gap_metric_summary
    )

    print(
        "\nSkill-gap confusion matrix:"
    )

    display(
        skill_gap_confusion_summary
    )


    return (
        skill_gap_metric_summary,
        skill_gap_confusion_summary,
        evaluation_df
    )


print(
    "✓ Skill-gap evaluation function "
    "created successfully."
)


# ============================================================
# TEST USING CURRENT UNREVIEWED TEMPLATE
# ============================================================

(
    skill_gap_metrics,
    skill_gap_confusion,
    skill_gap_evaluated
) = evaluate_skill_gap(
    skill_gap_ground_truth
)

Creating skill-gap evaluation function...
✓ Skill-gap evaluation function created successfully.

Total skill-gap rows: 200
Human gap labels completed: 0
Human gap labels missing: 200
Gap severity ratings completed: 0
Gap severity ratings missing: 200
Expert ratings completed: 0
Expert ratings missing: 200
Invalid human-gap labels: 0
Invalid severity ratings: 0
Invalid expert ratings: 0

Human gap-label completion: 0.00%
Gap-severity completion: 0.00%
Expert-rating completion: 0.00%

⚠️ Skill-gap evaluation is not yet complete.
Precision, Recall, F1, agreement, and expert-rating metrics will not be calculated until all human-review fields are completed.


In [163]:
# ============================================================
# CELL 67 — Create RAG Human-Review Evaluation Template
# ============================================================

print("Creating RAG human-review evaluation template...")


# ------------------------------------------------------------
# 1. USE SAME 20 REVIEW CANDIDATES
# ------------------------------------------------------------

rag_review_candidates = (
    review_candidates.copy()
)

print(
    "Candidates selected for RAG review:",
    len(rag_review_candidates)
)


# ------------------------------------------------------------
# 2. SELECT RAG OUTPUTS FOR REVIEW
# ------------------------------------------------------------

rag_human_review = (
    all_candidate_rag_explanations[
        all_candidate_rag_explanations[
            "candidate_id"
        ].isin(
            rag_review_candidates
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. KEEP IMPORTANT SYSTEM EVIDENCE
# ------------------------------------------------------------

rag_human_review = (
    rag_human_review[
        [
            "candidate_id",
            "recommended_occupation",
            "hybrid_recommendation_score",
            "missing_skill_count",
            "priority_missing_skills",
            "demand_percentage",
            "demand_level",
            "cip_status",
            "education_action",
            "skill_source_ids",
            "task_source_ids",
            "demand_source_ids",
            "education_source_ids",
            "rag_explanation"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 4. ADD HUMAN REVIEW FIELDS
# ------------------------------------------------------------

rag_human_review[
    "relevance_rating"
] = ""

rag_human_review[
    "correctness_rating"
] = ""

rag_human_review[
    "groundedness_rating"
] = ""

rag_human_review[
    "citation_correctness_rating"
] = ""

rag_human_review[
    "completeness_rating"
] = ""

rag_human_review[
    "clarity_rating"
] = ""

rag_human_review[
    "usefulness_rating"
] = ""

rag_human_review[
    "unsupported_claim_count"
] = ""

rag_human_review[
    "reviewer_notes"
] = ""


# ------------------------------------------------------------
# 5. VALIDATE TEMPLATE
# ------------------------------------------------------------

expected_rag_review_rows = (
    len(rag_review_candidates)
)

actual_rag_review_rows = len(
    rag_human_review
)

unique_rag_review_candidates = (
    rag_human_review[
        "candidate_id"
    ]
    .nunique()
)

duplicate_rag_candidate_rows = int(
    rag_human_review[
        "candidate_id"
    ]
    .duplicated()
    .sum()
)

empty_rag_explanations = int(
    rag_human_review[
        "rag_explanation"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)


print(
    "\nExpected RAG review rows:",
    expected_rag_review_rows
)

print(
    "Actual RAG review rows:",
    actual_rag_review_rows
)

print(
    "Unique candidates:",
    unique_rag_review_candidates
)

print(
    "Duplicate candidate rows:",
    duplicate_rag_candidate_rows
)

print(
    "Empty RAG explanations:",
    empty_rag_explanations
)


# ------------------------------------------------------------
# 6. DISPLAY SAMPLE
# ------------------------------------------------------------

print(
    "\nRAG human-review template sample:"
)

display(
    rag_human_review[
        [
            "candidate_id",
            "recommended_occupation",
            "hybrid_recommendation_score",
            "missing_skill_count",
            "demand_percentage",
            "demand_level",
            "cip_status",
            "relevance_rating",
            "correctness_rating",
            "groundedness_rating",
            "citation_correctness_rating",
            "completeness_rating",
            "clarity_rating",
            "usefulness_rating",
            "unsupported_claim_count",
            "reviewer_notes"
        ]
    ]
    .head(10)
)


# ------------------------------------------------------------
# 7. EXPORT TEMPLATE
# ------------------------------------------------------------

rag_review_path = (
    WEEK7_OUTPUT_DIR
    / "week7_rag_human_review.csv"
)

rag_human_review.to_csv(
    rag_review_path,
    index=False
)


print(
    "\nRAG human-review template exported:"
)

print(
    rag_review_path
)

print(
    "File exists:",
    rag_review_path.exists()
)

print(
    "File size:",
    rag_review_path.stat().st_size,
    "bytes"
)


# ------------------------------------------------------------
# 8. FINAL VALIDATION
# ------------------------------------------------------------

rag_review_template_pass = (
    expected_rag_review_rows
    == actual_rag_review_rows

    and unique_rag_review_candidates
    == len(rag_review_candidates)

    and duplicate_rag_candidate_rows
    == 0

    and empty_rag_explanations
    == 0
)


if rag_review_template_pass:

    print(
        "\n✅ PASS — RAG human-review "
        "evaluation template created successfully."
    )

else:

    print(
        "\n❌ FAIL — Review RAG human-review "
        "template construction."
    )


# ------------------------------------------------------------
# 9. HUMAN REVIEW INSTRUCTIONS
# ------------------------------------------------------------

print(
    "\nHUMAN REVIEW INSTRUCTIONS:"
)

print(
    "Rate each criterion from 1 to 5:"
)

print(
    "1 = Very poor"
)

print(
    "2 = Poor"
)

print(
    "3 = Acceptable"
)

print(
    "4 = Good"
)

print(
    "5 = Excellent"
)

print(
    "\nEvaluation criteria:"
)

print(
    "- relevance_rating: explanation addresses "
    "the candidate and recommended occupation."
)

print(
    "- correctness_rating: statements are factually "
    "consistent with retrieved evidence."
)

print(
    "- groundedness_rating: claims are supported "
    "by retrieved project sources."
)

print(
    "- citation_correctness_rating: cited RAG document "
    "IDs correctly support the associated claims."
)

print(
    "- completeness_rating: explanation covers recommendation, "
    "skill gaps, tasks, demand, education, and sources."
)

print(
    "- clarity_rating: explanation is understandable "
    "and well organized."
)

print(
    "- usefulness_rating: explanation provides useful "
    "career guidance."
)

print(
    "- unsupported_claim_count: enter the number of claims "
    "that are not supported by retrieved evidence."
)

print(
    "\nDo not change the generated RAG explanation "
    "or source-ID columns."
)

print(
    "Final RAG evaluation scores will be calculated "
    "only after human review is completed."
)

Creating RAG human-review evaluation template...
Candidates selected for RAG review: 20

Expected RAG review rows: 20
Actual RAG review rows: 20
Unique candidates: 20
Duplicate candidate rows: 0
Empty RAG explanations: 0

RAG human-review template sample:


,candidate_id,recommended_occupation,hybrid_recommendation_score,missing_skill_count,demand_percentage,demand_level,cip_status,relevance_rating,correctness_rating,groundedness_rating,citation_correctness_rating,completeness_rating,clarity_rating,usefulness_rating,unsupported_claim_count,reviewer_notes
0,Candidate_001,Software Developer,69.0545,3,22.91,Lower,CIP pathway available,,,,,,,,,
1,Candidate_002,Office Manager,55.6242,3,52.86,Moderate,CIP pathway available,,,,,,,,,
2,Candidate_003,Administrative Assistant,93.2442,2,43.85,Moderate,CIP pathway available,,,,,,,,,
3,Candidate_004,Software Developer,78.5914,2,22.91,Lower,CIP pathway available,,,,,,,,,
4,Candidate_005,Software Developer,53.3510,3,22.91,Lower,CIP pathway available,,,,,,,,,
5,Candidate_006,Software Developer,90.4630,1,22.91,Lower,CIP pathway available,,,,,,,,,
6,Candidate_007,Food Service Supervisor,100.0000,0,68.33,Moderate,CIP pathway available,,,,,,,,,
7,Candidate_008,Software Developer,68.0845,3,22.91,Lower,CIP pathway available,,,,,,,,,
8,Candidate_009,Administrative Assistant,27.7293,3,43.85,Moderate,CIP pathway available,,,,,,,,,
9,Candidate_010,Office Manager,98.5612,1,52.86,Moderate,CIP pathway available,,,,,,,,,



RAG human-review template exported:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7\week7_rag_human_review.csv
File exists: True
File size: 64676 bytes

✅ PASS — RAG human-review evaluation template created successfully.

HUMAN REVIEW INSTRUCTIONS:
Rate each criterion from 1 to 5:
1 = Very poor
2 = Poor
3 = Acceptable
4 = Good
5 = Excellent

Evaluation criteria:
- relevance_rating: explanation addresses the candidate and recommended occupation.
- correctness_rating: statements are factually consistent with retrieved evidence.
- groundedness_rating: claims are supported by retrieved project sources.
- citation_correctness_rating: cited RAG document IDs correctly support the associated claims.
- completeness_rating: explanation covers recommendation, skill gaps, tasks, demand, education, and sources.
- clarity_rating: explanation is understandable and well organized.
- usefulness_rating: explanation provides useful career guidance.
- unsupported_claim_count: enter the number of cla

In [165]:
# ============================================================
# CELL 68 — RAG Evaluation Function
# ============================================================

print("Creating RAG evaluation function...")


def evaluate_rag_quality(review_df):
    """
    Evaluate human-reviewed RAG explanations.

    Human-rated dimensions:
    - Relevance
    - Correctness
    - Groundedness
    - Citation correctness
    - Completeness
    - Clarity
    - Usefulness

    Additional metric:
    - Unsupported claim count

    Final metrics are calculated only after
    human review is complete.
    """

    evaluation_df = review_df.copy()


    # --------------------------------------------------------
    # 1. DEFINE REQUIRED REVIEW COLUMNS
    # --------------------------------------------------------

    rating_columns = [
        "relevance_rating",
        "correctness_rating",
        "groundedness_rating",
        "citation_correctness_rating",
        "completeness_rating",
        "clarity_rating",
        "usefulness_rating"
    ]

    required_columns = (
        [
            "candidate_id",
            "recommended_occupation",
            "rag_explanation",
            "unsupported_claim_count"
        ]
        + rating_columns
    )

    missing_columns = [
        col
        for col in required_columns
        if col not in evaluation_df.columns
    ]

    if missing_columns:

        print(
            "❌ Missing required columns:",
            missing_columns
        )

        return None, None, None


    # --------------------------------------------------------
    # 2. CONVERT HUMAN REVIEW FIELDS TO NUMERIC
    # --------------------------------------------------------

    for column in rating_columns:

        evaluation_df[column] = pd.to_numeric(
            evaluation_df[column],
            errors="coerce"
        )


    evaluation_df[
        "unsupported_claim_count"
    ] = pd.to_numeric(
        evaluation_df[
            "unsupported_claim_count"
        ],
        errors="coerce"
    )


    total_rows = len(
        evaluation_df
    )


    # --------------------------------------------------------
    # 3. CHECK REVIEW COMPLETION
    # --------------------------------------------------------

    completion_records = []

    for column in rating_columns:

        completed = int(
            evaluation_df[
                column
            ]
            .notna()
            .sum()
        )

        missing = (
            total_rows
            - completed
        )

        completion_records.append(
            {
                "field": column,
                "completed": completed,
                "missing": missing,
                "completion_percentage":
                    (
                        completed
                        / total_rows
                        * 100
                        if total_rows > 0
                        else 0
                    )
            }
        )


    unsupported_completed = int(
        evaluation_df[
            "unsupported_claim_count"
        ]
        .notna()
        .sum()
    )

    unsupported_missing = (
        total_rows
        - unsupported_completed
    )

    completion_records.append(
        {
            "field":
                "unsupported_claim_count",

            "completed":
                unsupported_completed,

            "missing":
                unsupported_missing,

            "completion_percentage":
                (
                    unsupported_completed
                    / total_rows
                    * 100
                    if total_rows > 0
                    else 0
                )
        }
    )


    rag_review_completion = pd.DataFrame(
        completion_records
    )


    print(
        "\nTotal RAG review rows:",
        total_rows
    )

    print(
        "\nHuman-review completion:"
    )

    display(
        rag_review_completion
    )


    # --------------------------------------------------------
    # 4. VALIDATE 1–5 RATINGS
    # --------------------------------------------------------

    invalid_rating_count = 0

    for column in rating_columns:

        reviewed_values = (
            evaluation_df.loc[
                evaluation_df[
                    column
                ].notna(),
                column
            ]
        )

        invalid_rating_count += int(
            (
                ~reviewed_values.isin(
                    [1, 2, 3, 4, 5]
                )
            ).sum()
        )


    reviewed_unsupported = (
        evaluation_df.loc[
            evaluation_df[
                "unsupported_claim_count"
            ].notna(),
            "unsupported_claim_count"
        ]
    )

    invalid_unsupported_count = int(
        (
            (
                reviewed_unsupported < 0
            )
            |
            (
                reviewed_unsupported
                % 1 != 0
            )
        ).sum()
    )


    print(
        "\nInvalid 1–5 ratings:",
        invalid_rating_count
    )

    print(
        "Invalid unsupported-claim counts:",
        invalid_unsupported_count
    )


    if (
        invalid_rating_count > 0
        or invalid_unsupported_count > 0
    ):

        print(
            "\n❌ Invalid RAG human-review "
            "values detected."
        )

        print(
            "All quality ratings must be integers "
            "from 1 to 5."
        )

        print(
            "unsupported_claim_count must be a "
            "non-negative whole number."
        )

        return (
            None,
            rag_review_completion,
            evaluation_df
        )


    # --------------------------------------------------------
    # 5. STOP IF REVIEW IS INCOMPLETE
    # --------------------------------------------------------

    total_missing_fields = int(
        rag_review_completion[
            "missing"
        ].sum()
    )

    if total_missing_fields > 0:

        print(
            "\n⚠️ RAG evaluation is not yet complete."
        )

        print(
            "Final relevance, correctness, groundedness, "
            "citation correctness, completeness, clarity, "
            "usefulness, and unsupported-claim metrics "
            "will not be calculated until all human-review "
            "fields are completed."
        )

        return (
            None,
            rag_review_completion,
            evaluation_df
        )


    # --------------------------------------------------------
    # 6. CONVERT COMPLETED VALUES
    # --------------------------------------------------------

    for column in rating_columns:

        evaluation_df[
            column
        ] = (
            evaluation_df[
                column
            ]
            .astype(int)
        )


    evaluation_df[
        "unsupported_claim_count"
    ] = (
        evaluation_df[
            "unsupported_claim_count"
        ]
        .astype(int)
    )


    # --------------------------------------------------------
    # 7. CANDIDATE-LEVEL OVERALL RAG SCORE
    # --------------------------------------------------------

    evaluation_df[
        "overall_rag_rating"
    ] = (
        evaluation_df[
            rating_columns
        ]
        .mean(
            axis=1
        )
    )


    evaluation_df[
        "overall_rag_percentage"
    ] = (
        evaluation_df[
            "overall_rag_rating"
        ]
        / 5
        * 100
    )


    # --------------------------------------------------------
    # 8. AGGREGATE QUALITY METRICS
    # --------------------------------------------------------

    metric_name_map = {
        "relevance_rating":
            "Relevance",

        "correctness_rating":
            "Correctness",

        "groundedness_rating":
            "Groundedness",

        "citation_correctness_rating":
            "Citation Correctness",

        "completeness_rating":
            "Completeness",

        "clarity_rating":
            "Clarity",

        "usefulness_rating":
            "Usefulness"
    }


    summary_records = []

    for column in rating_columns:

        mean_rating = (
            evaluation_df[
                column
            ]
            .mean()
        )

        summary_records.append(
            {
                "metric":
                    metric_name_map[
                        column
                    ],

                "mean_rating":
                    mean_rating,

                "percentage":
                    (
                        mean_rating
                        / 5
                        * 100
                    )
            }
        )


    overall_mean_rating = (
        evaluation_df[
            "overall_rag_rating"
        ]
        .mean()
    )

    summary_records.append(
        {
            "metric":
                "Overall RAG Quality",

            "mean_rating":
                overall_mean_rating,

            "percentage":
                (
                    overall_mean_rating
                    / 5
                    * 100
                )
        }
    )


    # --------------------------------------------------------
    # 9. UNSUPPORTED CLAIM METRICS
    # --------------------------------------------------------

    total_unsupported_claims = int(
        evaluation_df[
            "unsupported_claim_count"
        ]
        .sum()
    )

    mean_unsupported_claims = (
        evaluation_df[
            "unsupported_claim_count"
        ]
        .mean()
    )

    zero_unsupported_count = int(
        (
            evaluation_df[
                "unsupported_claim_count"
            ] == 0
        ).sum()
    )

    zero_unsupported_rate = (
        zero_unsupported_count
        / total_rows
        if total_rows > 0
        else 0
    )


    # --------------------------------------------------------
    # 10. CREATE SUMMARY DATAFRAME
    # --------------------------------------------------------

    rag_metric_summary = pd.DataFrame(
        summary_records
    )

    rag_metric_summary[
        "mean_rating"
    ] = (
        rag_metric_summary[
            "mean_rating"
        ]
        .round(3)
    )

    rag_metric_summary[
        "percentage"
    ] = (
        rag_metric_summary[
            "percentage"
        ]
        .round(2)
    )


    unsupported_claim_summary = pd.DataFrame(
        [
            {
                "metric":
                    "Total Unsupported Claims",

                "value":
                    total_unsupported_claims
            },
            {
                "metric":
                    "Mean Unsupported Claims per Explanation",

                "value":
                    round(
                        mean_unsupported_claims,
                        3
                    )
            },
            {
                "metric":
                    "Explanations with Zero Unsupported Claims",

                "value":
                    zero_unsupported_count
            },
            {
                "metric":
                    "Zero Unsupported Claim Rate",

                "value":
                    round(
                        zero_unsupported_rate
                        * 100,
                        2
                    )
            }
        ]
    )


    # --------------------------------------------------------
    # 11. DISPLAY RESULTS
    # --------------------------------------------------------

    print(
        "\n✅ Human RAG review complete."
    )

    print(
        "\nRAG quality metrics:"
    )

    display(
        rag_metric_summary
    )

    print(
        "\nUnsupported-claim evaluation:"
    )

    display(
        unsupported_claim_summary
    )

    print(
        "\nCandidate-level RAG evaluation sample:"
    )

    display(
        evaluation_df[
            [
                "candidate_id",
                "recommended_occupation",
                "overall_rag_rating",
                "overall_rag_percentage",
                "unsupported_claim_count"
            ]
        ]
        .head(10)
    )


    return (
        rag_metric_summary,
        unsupported_claim_summary,
        evaluation_df
    )


print(
    "✓ RAG evaluation function "
    "created successfully."
)


# ============================================================
# TEST USING CURRENT UNREVIEWED TEMPLATE
# ============================================================

(
    rag_quality_metrics,
    rag_unsupported_claim_metrics,
    rag_evaluated
) = evaluate_rag_quality(
    rag_human_review
)

Creating RAG evaluation function...
✓ RAG evaluation function created successfully.

Total RAG review rows: 20

Human-review completion:


,field,completed,missing,completion_percentage
0,relevance_rating,0,20,0.0
1,correctness_rating,0,20,0.0
2,groundedness_rating,0,20,0.0
3,citation_correctness_rating,0,20,0.0
4,completeness_rating,0,20,0.0
5,clarity_rating,0,20,0.0
6,usefulness_rating,0,20,0.0
7,unsupported_claim_count,0,20,0.0



Invalid 1–5 ratings: 0
Invalid unsupported-claim counts: 0

⚠️ RAG evaluation is not yet complete.
Final relevance, correctness, groundedness, citation correctness, completeness, clarity, usefulness, and unsupported-claim metrics will not be calculated until all human-review fields are completed.


In [167]:
# ============================================================
# CELL 69 — Inspect Week 6 Scoring Components for Ablation
# ============================================================

print("Inspecting Week 6 scoring components for ablation analysis...")


# ------------------------------------------------------------
# 1. RELEVANT SCORE COLUMNS
# ------------------------------------------------------------

ablation_columns = [
    "candidate_id",
    "selected_occupation",
    "skill_normalized_score",
    "skill_percentage",
    "tfidf_normalized_score",
    "tfidf_percentage",
    "semantic_normalized_score",
    "semantic_percentage",
    "demand_score",
    "demand_percentage",
    "education_score",
    "education_percentage",
    "candidate_match_score",
    "final_hybrid_score",
    "final_hybrid_percentage",
    "final_hybrid_rank",
    "demand_adjusted_score",
    "demand_adjusted_rank",
    "tfidf_demand_score",
    "tfidf_demand_rank",
    "tfidf_education_score",
    "tfidf_education_rank"
]


available_ablation_columns = [
    col
    for col in ablation_columns
    if col in final_hybrid.columns
]

missing_ablation_columns = [
    col
    for col in ablation_columns
    if col not in final_hybrid.columns
]


print(
    "\nAvailable ablation-related columns:"
)

print(
    available_ablation_columns
)


print(
    "\nMissing requested columns:"
)

print(
    missing_ablation_columns
)


# ------------------------------------------------------------
# 2. DISPLAY SAMPLE
# ------------------------------------------------------------

print(
    "\nSample Week 6 scoring data:"
)

display(
    final_hybrid[
        available_ablation_columns
    ]
    .sort_values(
        [
            "candidate_id",
            "final_hybrid_rank"
        ]
    )
    .head(20)
)


# ------------------------------------------------------------
# 3. NUMERIC SUMMARY
# ------------------------------------------------------------

numeric_ablation_columns = [
    col
    for col in available_ablation_columns
    if col not in [
        "candidate_id",
        "selected_occupation"
    ]
]

print(
    "\nScore ranges:"
)

display(
    final_hybrid[
        numeric_ablation_columns
    ]
    .agg(
        [
            "min",
            "max",
            "mean"
        ]
    )
    .T
    .round(4)
)


# ------------------------------------------------------------
# 4. CHECK RELATIONSHIPS BETWEEN EXISTING SCORES
# ------------------------------------------------------------

relationship_columns = [
    col
    for col in [
        "skill_normalized_score",
        "tfidf_normalized_score",
        "semantic_normalized_score",
        "demand_score",
        "education_score",
        "candidate_match_score",
        "final_hybrid_score",
        "demand_adjusted_score",
        "tfidf_demand_score",
        "tfidf_education_score"
    ]
    if col in final_hybrid.columns
]

print(
    "\nCorrelation between existing scoring components:"
)

display(
    final_hybrid[
        relationship_columns
    ]
    .corr()
    .round(4)
)

Inspecting Week 6 scoring components for ablation analysis...

Available ablation-related columns:
['candidate_id', 'selected_occupation', 'skill_normalized_score', 'skill_percentage', 'tfidf_normalized_score', 'tfidf_percentage', 'semantic_normalized_score', 'semantic_percentage', 'demand_score', 'demand_percentage', 'education_score', 'education_percentage', 'candidate_match_score', 'final_hybrid_score', 'final_hybrid_percentage', 'final_hybrid_rank', 'demand_adjusted_score', 'demand_adjusted_rank', 'tfidf_demand_score', 'tfidf_demand_rank', 'tfidf_education_score', 'tfidf_education_rank']

Missing requested columns:
[]

Sample Week 6 scoring data:


,candidate_id,selected_occupation,skill_normalized_score,skill_percentage,tfidf_normalized_score,tfidf_percentage,semantic_normalized_score,semantic_percentage,demand_score,demand_percentage,education_score,education_percentage,candidate_match_score,final_hybrid_score,final_hybrid_percentage,final_hybrid_rank,demand_adjusted_score,demand_adjusted_rank,tfidf_demand_score,tfidf_demand_rank,tfidf_education_score,tfidf_education_rank
0,Candidate_001,Software Developer,1.000000,100.00,0.395807,39.58,1.000000,100.00,0.229086,22.91,1.0,100.0,0.700000,0.845817,84.58,1,0.845817,1,0.362463,3,0.456227,2
1,Candidate_001,Information Technology (IT) Analyst,0.696776,69.68,1.000000,100.00,0.730981,73.10,0.266376,26.64,1.0,100.0,0.499715,0.652990,65.30,2,0.624378,2,0.853275,1,1.000000,1
2,Candidate_001,Bookkeeper,0.612782,61.28,0.230774,23.08,0.654600,65.46,0.299226,29.92,1.0,100.0,0.443584,0.603429,60.34,3,0.566798,3,0.244465,4,0.307697,4
3,Candidate_001,Secondary School Teacher,0.622929,62.29,0.055379,5.54,0.574992,57.50,0.363556,36.36,1.0,100.0,0.419272,0.591983,59.20,4,0.551879,4,0.117015,13,0.149842,5
4,Candidate_001,Office Administrator,0.436160,43.62,0.031502,3.15,0.592570,59.26,0.528644,52.86,0.0,0.0,0.360055,0.465784,46.58,5,0.517220,5,0.130931,11,0.028352,12
5,Candidate_001,Office Manager,0.415372,41.54,0.055075,5.51,0.538183,53.82,0.528644,52.86,0.0,0.0,0.333744,0.439473,43.95,6,0.487151,6,0.149789,10,0.049568,8
6,Candidate_001,Licensed Practical Nurse (L.P.N.),0.515402,51.54,0.028612,2.86,0.197713,19.77,0.303919,30.39,1.0,100.0,0.249590,0.410374,41.04,7,0.346030,12,0.083673,15,0.125750,6
7,Candidate_001,Restaurant Manager,0.268008,26.80,0.000000,0.00,0.467688,46.77,0.586501,58.65,0.0,0.0,0.257494,0.374794,37.48,8,0.411578,7,0.117300,12,0.000000,15
8,Candidate_001,"Driver, Truck",0.253182,25.32,0.016123,1.61,0.230136,23.01,0.888027,88.80,0.0,0.0,0.169161,0.346766,34.68,9,0.370932,11,0.190504,7,0.014510,14
9,Candidate_001,Food Service Supervisor,0.036362,3.64,0.049424,4.94,0.558170,55.82,0.683338,68.33,0.0,0.0,0.208086,0.344754,34.48,10,0.374480,9,0.176207,8,0.044482,9



Score ranges:


,min,max,mean
skill_normalized_score,0.0000,1.0000,0.4517
skill_percentage,0.0000,100.0000,45.1746
tfidf_normalized_score,0.0000,1.0000,0.3229
tfidf_percentage,0.0000,100.0000,32.2852
semantic_normalized_score,0.0000,1.0000,0.3990
semantic_percentage,0.0000,100.0000,39.8953
demand_score,0.2291,0.8880,0.5103
demand_percentage,22.9100,88.8000,51.0307
education_score,0.0000,1.0000,0.2963
education_percentage,0.0000,100.0000,29.6296



Correlation between existing scoring components:


,skill_normalized_score,tfidf_normalized_score,semantic_normalized_score,demand_score,education_score,candidate_match_score,final_hybrid_score,demand_adjusted_score,tfidf_demand_score,tfidf_education_score
skill_normalized_score,1.0000,0.1582,0.2122,-0.0381,0.6148,0.8330,0.8778,0.8599,0.1611,0.2578
tfidf_normalized_score,0.1582,1.0000,0.5669,-0.4178,0.1999,0.4337,0.3311,0.3484,0.9815,0.9852
semantic_normalized_score,0.2122,0.5669,1.0000,-0.4662,0.2737,0.7175,0.5873,0.6324,0.5082,0.5866
demand_score,-0.0381,-0.4178,-0.4662,1.0000,-0.2744,-0.2911,-0.0849,-0.0542,-0.2362,-0.4450
education_score,0.6148,0.1999,0.2737,-0.2744,1.0000,0.5932,0.7077,0.5507,0.1560,0.3648
candidate_match_score,0.8330,0.4337,0.7175,-0.2911,0.5932,1.0000,0.9582,0.9711,0.4026,0.5159
final_hybrid_score,0.8778,0.3311,0.5873,-0.0849,0.7077,0.9582,1.0000,0.9790,0.3363,0.4384
demand_adjusted_score,0.8599,0.3484,0.6324,-0.0542,0.5507,0.9711,0.9790,1.0000,0.3612,0.4274
tfidf_demand_score,0.1611,0.9815,0.5082,-0.2362,0.1560,0.4026,0.3363,0.3612,1.0000,0.9600
tfidf_education_score,0.2578,0.9852,0.5866,-0.4450,0.3648,0.5159,0.4384,0.4274,0.9600,1.0000


In [169]:
# ============================================================
# CELL 70 — Verify Exact Week 6 Scoring Formulas
# ============================================================

print("Verifying exact Week 6 scoring formulas...")


# ------------------------------------------------------------
# 1. TEST CANDIDATE MATCH FORMULA
#
# Observed from Cell 69:
#
# candidate_match_score =
#     0.35 * skill_normalized_score
#   + 0.35 * semantic_normalized_score
#
# Maximum possible value = 0.70
# ------------------------------------------------------------

final_hybrid[
    "candidate_match_reconstructed"
] = (
    0.35
    * final_hybrid[
        "skill_normalized_score"
    ]
    +
    0.35
    * final_hybrid[
        "semantic_normalized_score"
    ]
)


final_hybrid[
    "candidate_match_difference"
] = (
    final_hybrid[
        "candidate_match_score"
    ]
    -
    final_hybrid[
        "candidate_match_reconstructed"
    ]
).abs()


candidate_match_max_difference = (
    final_hybrid[
        "candidate_match_difference"
    ]
    .max()
)


# ------------------------------------------------------------
# 2. TEST FINAL HYBRID FORMULA
#
# Observed from Cell 69:
#
# final_hybrid_score =
#     candidate_match_score
#   + 0.20 * demand_score
#   + 0.10 * education_score
#
# Total theoretical maximum:
#
# 0.70 + 0.20 + 0.10 = 1.00
# ------------------------------------------------------------

final_hybrid[
    "final_hybrid_reconstructed"
] = (
    final_hybrid[
        "candidate_match_score"
    ]
    +
    0.20
    * final_hybrid[
        "demand_score"
    ]
    +
    0.10
    * final_hybrid[
        "education_score"
    ]
)


final_hybrid[
    "final_hybrid_difference"
] = (
    final_hybrid[
        "final_hybrid_score"
    ]
    -
    final_hybrid[
        "final_hybrid_reconstructed"
    ]
).abs()


final_hybrid_max_difference = (
    final_hybrid[
        "final_hybrid_difference"
    ]
    .max()
)


# ------------------------------------------------------------
# 3. CHECK NUMBER OF MATCHING ROWS
# ------------------------------------------------------------

tolerance = 1e-8


candidate_match_exact_rows = int(
    (
        final_hybrid[
            "candidate_match_difference"
        ]
        <= tolerance
    )
    .sum()
)


final_hybrid_exact_rows = int(
    (
        final_hybrid[
            "final_hybrid_difference"
        ]
        <= tolerance
    )
    .sum()
)


total_rows = len(
    final_hybrid
)


print(
    "\nTotal candidate-occupation rows:",
    total_rows
)


print(
    "\nCandidate-match formula:"
)

print(
    "candidate_match_score = "
    "0.35 × skill_normalized_score "
    "+ 0.35 × semantic_normalized_score"
)

print(
    "Maximum absolute difference:",
    candidate_match_max_difference
)

print(
    "Rows matching formula:",
    candidate_match_exact_rows,
    "/",
    total_rows
)


print(
    "\nFinal-hybrid formula:"
)

print(
    "final_hybrid_score = "
    "candidate_match_score "
    "+ 0.20 × demand_score "
    "+ 0.10 × education_score"
)

print(
    "Maximum absolute difference:",
    final_hybrid_max_difference
)

print(
    "Rows matching formula:",
    final_hybrid_exact_rows,
    "/",
    total_rows
)


# ------------------------------------------------------------
# 4. DISPLAY SAMPLE COMPARISON
# ------------------------------------------------------------

print(
    "\nFormula verification sample:"
)

display(
    final_hybrid[
        [
            "candidate_id",
            "selected_occupation",
            "skill_normalized_score",
            "semantic_normalized_score",
            "candidate_match_score",
            "candidate_match_reconstructed",
            "candidate_match_difference",
            "demand_score",
            "education_score",
            "final_hybrid_score",
            "final_hybrid_reconstructed",
            "final_hybrid_difference"
        ]
    ]
    .head(20)
)


# ------------------------------------------------------------
# 5. FINAL VALIDATION
# ------------------------------------------------------------

formula_validation_pass = (
    candidate_match_max_difference
    <= tolerance

    and final_hybrid_max_difference
    <= tolerance

    and candidate_match_exact_rows
    == total_rows

    and final_hybrid_exact_rows
    == total_rows
)


if formula_validation_pass:

    print(
        "\n✅ PASS — Exact Week 6 scoring formulas "
        "confirmed across all rows."
    )

else:

    print(
        "\n⚠️ Formula differences detected."
    )

    print(
        "Do not proceed with ablation scoring until "
        "the differences are reviewed."
    )

Verifying exact Week 6 scoring formulas...

Total candidate-occupation rows: 945

Candidate-match formula:
candidate_match_score = 0.35 × skill_normalized_score + 0.35 × semantic_normalized_score
Maximum absolute difference: 1.1102230246251565e-16
Rows matching formula: 945 / 945

Final-hybrid formula:
final_hybrid_score = candidate_match_score + 0.20 × demand_score + 0.10 × education_score
Maximum absolute difference: 1.1102230246251565e-16
Rows matching formula: 945 / 945

Formula verification sample:


,candidate_id,selected_occupation,skill_normalized_score,semantic_normalized_score,candidate_match_score,candidate_match_reconstructed,candidate_match_difference,demand_score,education_score,final_hybrid_score,final_hybrid_reconstructed,final_hybrid_difference
0,Candidate_001,Software Developer,1.000000,1.000000,0.700000,0.700000,0.000000e+00,0.229086,1.0,0.845817,0.845817,0.000000e+00
1,Candidate_001,Information Technology (IT) Analyst,0.696776,0.730981,0.499715,0.499715,0.000000e+00,0.266376,1.0,0.652990,0.652990,0.000000e+00
2,Candidate_001,Bookkeeper,0.612782,0.654600,0.443584,0.443584,0.000000e+00,0.299226,1.0,0.603429,0.603429,0.000000e+00
3,Candidate_001,Secondary School Teacher,0.622929,0.574992,0.419272,0.419272,0.000000e+00,0.363556,1.0,0.591983,0.591983,0.000000e+00
4,Candidate_001,Office Administrator,0.436160,0.592570,0.360055,0.360055,0.000000e+00,0.528644,0.0,0.465784,0.465784,0.000000e+00
5,Candidate_001,Office Manager,0.415372,0.538183,0.333744,0.333744,5.551115e-17,0.528644,0.0,0.439473,0.439473,0.000000e+00
6,Candidate_001,Licensed Practical Nurse (L.P.N.),0.515402,0.197713,0.249590,0.249590,2.775558e-17,0.303919,1.0,0.410374,0.410374,0.000000e+00
7,Candidate_001,Restaurant Manager,0.268008,0.467688,0.257494,0.257494,5.551115e-17,0.586501,0.0,0.374794,0.374794,5.551115e-17
8,Candidate_001,"Driver, Truck",0.253182,0.230136,0.169161,0.169161,2.775558e-17,0.888027,0.0,0.346766,0.346766,5.551115e-17
9,Candidate_001,Food Service Supervisor,0.036362,0.558170,0.208086,0.208086,5.551115e-17,0.683338,0.0,0.344754,0.344754,0.000000e+00



✅ PASS — Exact Week 6 scoring formulas confirmed across all rows.


In [171]:
# ============================================================
# CELL 71 — Build Dataset Ablation Model Configurations
# ============================================================

print("Building Week 7 dataset-ablation configurations...")


# ------------------------------------------------------------
# 1. CREATE ABLATION DATAFRAME
# ------------------------------------------------------------

ablation_results = (
    final_hybrid[
        [
            "candidate_id",
            "selected_occupation",
            "skill_normalized_score",
            "semantic_normalized_score",
            "demand_score",
            "education_score",
            "candidate_match_score",
            "final_hybrid_score",
            "final_hybrid_rank"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 2. CONFIGURATION A — O*NET ONLY
#
# O*NET-derived candidate match:
# 0.35 skill + 0.35 semantic
#
# Theoretical maximum = 0.70
# ------------------------------------------------------------

ablation_results[
    "onet_only_score"
] = (
    0.35
    * ablation_results[
        "skill_normalized_score"
    ]
    +
    0.35
    * ablation_results[
        "semantic_normalized_score"
    ]
)


# Normalized to 0–1 only for easier score interpretation.
# This does NOT change ranking.

ablation_results[
    "onet_only_normalized"
] = (
    ablation_results[
        "onet_only_score"
    ]
    / 0.70
)


# ------------------------------------------------------------
# 3. CONFIGURATION B — O*NET + JOB BANK
#
# Adds Canadian labour-demand evidence.
#
# Theoretical maximum = 0.90
# ------------------------------------------------------------

ablation_results[
    "onet_jobbank_score"
] = (
    ablation_results[
        "onet_only_score"
    ]
    +
    0.20
    * ablation_results[
        "demand_score"
    ]
)


ablation_results[
    "onet_jobbank_normalized"
] = (
    ablation_results[
        "onet_jobbank_score"
    ]
    / 0.90
)


# ------------------------------------------------------------
# 4. CONFIGURATION C — O*NET + CIP
#
# Adds education-alignment evidence.
#
# Theoretical maximum = 0.80
# ------------------------------------------------------------

ablation_results[
    "onet_cip_score"
] = (
    ablation_results[
        "onet_only_score"
    ]
    +
    0.10
    * ablation_results[
        "education_score"
    ]
)


ablation_results[
    "onet_cip_normalized"
] = (
    ablation_results[
        "onet_cip_score"
    ]
    / 0.80
)


# ------------------------------------------------------------
# 5. CONFIGURATION D — ALL DATASETS
#
# O*NET + Job Bank + CIP
#
# Theoretical maximum = 1.00
# ------------------------------------------------------------

ablation_results[
    "all_datasets_score"
] = (
    ablation_results[
        "onet_only_score"
    ]
    +
    0.20
    * ablation_results[
        "demand_score"
    ]
    +
    0.10
    * ablation_results[
        "education_score"
    ]
)


ablation_results[
    "all_datasets_normalized"
] = (
    ablation_results[
        "all_datasets_score"
    ]
)


# ------------------------------------------------------------
# 6. CREATE RANKS WITHIN EACH CANDIDATE
#
# method="first" gives deterministic ranks if scores tie.
# ------------------------------------------------------------

score_rank_pairs = [
    (
        "onet_only_score",
        "onet_only_rank"
    ),
    (
        "onet_jobbank_score",
        "onet_jobbank_rank"
    ),
    (
        "onet_cip_score",
        "onet_cip_rank"
    ),
    (
        "all_datasets_score",
        "all_datasets_rank"
    )
]


for score_column, rank_column in score_rank_pairs:

    ablation_results[
        rank_column
    ] = (
        ablation_results
        .groupby(
            "candidate_id"
        )[
            score_column
        ]
        .rank(
            method="first",
            ascending=False
        )
        .astype(int)
    )


# ------------------------------------------------------------
# 7. VERIFY RECONSTRUCTED SCORES
# ------------------------------------------------------------

ablation_results[
    "onet_formula_difference"
] = (
    ablation_results[
        "onet_only_score"
    ]
    -
    ablation_results[
        "candidate_match_score"
    ]
).abs()


ablation_results[
    "all_formula_difference"
] = (
    ablation_results[
        "all_datasets_score"
    ]
    -
    ablation_results[
        "final_hybrid_score"
    ]
).abs()


max_onet_difference = (
    ablation_results[
        "onet_formula_difference"
    ]
    .max()
)

max_all_difference = (
    ablation_results[
        "all_formula_difference"
    ]
    .max()
)


# ------------------------------------------------------------
# 8. VERIFY FULL-MODEL RANK
# ------------------------------------------------------------

full_rank_difference_count = int(
    (
        ablation_results[
            "all_datasets_rank"
        ]
        !=
        ablation_results[
            "final_hybrid_rank"
        ]
    )
    .sum()
)


# ------------------------------------------------------------
# 9. BASIC STRUCTURAL VALIDATION
# ------------------------------------------------------------

total_ablation_rows = len(
    ablation_results
)

unique_ablation_candidates = (
    ablation_results[
        "candidate_id"
    ]
    .nunique()
)

unique_ablation_occupations = (
    ablation_results[
        "selected_occupation"
    ]
    .nunique()
)


print(
    "\nTotal candidate-occupation rows:",
    total_ablation_rows
)

print(
    "Unique candidates:",
    unique_ablation_candidates
)

print(
    "Unique occupations:",
    unique_ablation_occupations
)


print(
    "\nMaximum O*NET formula difference:",
    max_onet_difference
)

print(
    "Maximum full-model formula difference:",
    max_all_difference
)

print(
    "Full-model rank differences:",
    full_rank_difference_count
)


# ------------------------------------------------------------
# 10. CHECK SCORE RANGES
# ------------------------------------------------------------

ablation_score_summary = pd.DataFrame(
    [
        {
            "configuration":
                "O*NET Only",

            "datasets":
                "O*NET",

            "raw_score_maximum":
                0.70,

            "observed_min":
                ablation_results[
                    "onet_only_score"
                ].min(),

            "observed_max":
                ablation_results[
                    "onet_only_score"
                ].max(),

            "observed_mean":
                ablation_results[
                    "onet_only_score"
                ].mean()
        },
        {
            "configuration":
                "O*NET + Job Bank",

            "datasets":
                "O*NET + Job Bank",

            "raw_score_maximum":
                0.90,

            "observed_min":
                ablation_results[
                    "onet_jobbank_score"
                ].min(),

            "observed_max":
                ablation_results[
                    "onet_jobbank_score"
                ].max(),

            "observed_mean":
                ablation_results[
                    "onet_jobbank_score"
                ].mean()
        },
        {
            "configuration":
                "O*NET + CIP",

            "datasets":
                "O*NET + CIP",

            "raw_score_maximum":
                0.80,

            "observed_min":
                ablation_results[
                    "onet_cip_score"
                ].min(),

            "observed_max":
                ablation_results[
                    "onet_cip_score"
                ].max(),

            "observed_mean":
                ablation_results[
                    "onet_cip_score"
                ].mean()
        },
        {
            "configuration":
                "All Datasets",

            "datasets":
                "O*NET + Job Bank + CIP",

            "raw_score_maximum":
                1.00,

            "observed_min":
                ablation_results[
                    "all_datasets_score"
                ].min(),

            "observed_max":
                ablation_results[
                    "all_datasets_score"
                ].max(),

            "observed_mean":
                ablation_results[
                    "all_datasets_score"
                ].mean()
        }
    ]
)


for column in [
    "observed_min",
    "observed_max",
    "observed_mean"
]:

    ablation_score_summary[
        column
    ] = (
        ablation_score_summary[
            column
        ]
        .round(4)
    )


print(
    "\nAblation configuration score summary:"
)

display(
    ablation_score_summary
)


# ------------------------------------------------------------
# 11. DISPLAY CANDIDATE_001 EXAMPLE
# ------------------------------------------------------------

print(
    "\nCandidate_001 ablation ranking comparison:"
)

display(
    ablation_results[
        ablation_results[
            "candidate_id"
        ]
        ==
        "Candidate_001"
    ]
    [
        [
            "candidate_id",
            "selected_occupation",
            "onet_only_score",
            "onet_only_rank",
            "onet_jobbank_score",
            "onet_jobbank_rank",
            "onet_cip_score",
            "onet_cip_rank",
            "all_datasets_score",
            "all_datasets_rank"
        ]
    ]
    .sort_values(
        "all_datasets_rank"
    )
)


# ------------------------------------------------------------
# 12. FINAL VALIDATION
# ------------------------------------------------------------

tolerance = 1e-8


ablation_configuration_pass = (
    total_ablation_rows
    == len(final_hybrid)

    and unique_ablation_candidates
    == 63

    and unique_ablation_occupations
    == 15

    and max_onet_difference
    <= tolerance

    and max_all_difference
    <= tolerance

    and full_rank_difference_count
    == 0
)


if ablation_configuration_pass:

    print(
        "\n✅ PASS — All four dataset-ablation "
        "configurations created and validated."
    )

else:

    print(
        "\n❌ FAIL — Review ablation configuration "
        "construction before continuing."
    )

Building Week 7 dataset-ablation configurations...

Total candidate-occupation rows: 945
Unique candidates: 63
Unique occupations: 15

Maximum O*NET formula difference: 1.1102230246251565e-16
Maximum full-model formula difference: 1.1102230246251565e-16
Full-model rank differences: 0

Ablation configuration score summary:


,configuration,datasets,raw_score_maximum,observed_min,observed_max,observed_mean
0,O*NET Only,O*NET,0.7,0.0000,0.7000,0.2977
1,O*NET + Job Bank,O*NET + Job Bank,0.9,0.0731,0.7458,0.3998
2,O*NET + CIP,O*NET + CIP,0.8,0.0123,0.8000,0.3274
3,All Datasets,O*NET + Job Bank + CIP,1.0,0.0731,0.8458,0.4294



Candidate_001 ablation ranking comparison:


,candidate_id,selected_occupation,onet_only_score,onet_only_rank,onet_jobbank_score,onet_jobbank_rank,onet_cip_score,onet_cip_rank,all_datasets_score,all_datasets_rank
0,Candidate_001,Software Developer,0.700000,1,0.745817,1,0.800000,1,0.845817,1
1,Candidate_001,Information Technology (IT) Analyst,0.499715,2,0.552990,2,0.599715,2,0.652990,2
2,Candidate_001,Bookkeeper,0.443584,3,0.503429,3,0.543584,3,0.603429,3
3,Candidate_001,Secondary School Teacher,0.419272,4,0.491983,4,0.519272,4,0.591983,4
4,Candidate_001,Office Administrator,0.360055,5,0.465784,5,0.360055,5,0.465784,5
5,Candidate_001,Office Manager,0.333744,6,0.439473,6,0.333744,7,0.439473,6
6,Candidate_001,Licensed Practical Nurse (L.P.N.),0.249590,9,0.310374,14,0.349590,6,0.410374,7
7,Candidate_001,Restaurant Manager,0.257494,7,0.374794,7,0.257494,8,0.374794,8
8,Candidate_001,"Driver, Truck",0.169161,12,0.346766,8,0.169161,12,0.346766,9
9,Candidate_001,Food Service Supervisor,0.208086,11,0.344754,9,0.208086,11,0.344754,10



✅ PASS — All four dataset-ablation configurations created and validated.


In [173]:
# ============================================================
# CELL 72 — Dataset Contribution / Ablation Ranking Analysis
# ============================================================

print("Analyzing dataset contribution and ranking changes...")


# ------------------------------------------------------------
# 1. CONFIGURATION DEFINITIONS
# ------------------------------------------------------------

ablation_configurations = {
    "O*NET Only": "onet_only_rank",
    "O*NET + Job Bank": "onet_jobbank_rank",
    "O*NET + CIP": "onet_cip_rank",
    "All Datasets": "all_datasets_rank"
}


# ------------------------------------------------------------
# 2. EXTRACT TOP-1 RECOMMENDATION FOR EACH CONFIGURATION
# ------------------------------------------------------------

top1_tables = {}

for config_name, rank_column in ablation_configurations.items():

    temp = (
        ablation_results[
            ablation_results[
                rank_column
            ] == 1
        ]
        [
            [
                "candidate_id",
                "selected_occupation"
            ]
        ]
        .copy()
    )

    temp = temp.rename(
        columns={
            "selected_occupation":
                config_name
        }
    )

    top1_tables[
        config_name
    ] = temp


# ------------------------------------------------------------
# 3. COMBINE TOP-1 RESULTS
# ------------------------------------------------------------

ablation_top1_comparison = (
    top1_tables[
        "O*NET Only"
    ]
    .merge(
        top1_tables[
            "O*NET + Job Bank"
        ],
        on="candidate_id",
        how="inner"
    )
    .merge(
        top1_tables[
            "O*NET + CIP"
        ],
        on="candidate_id",
        how="inner"
    )
    .merge(
        top1_tables[
            "All Datasets"
        ],
        on="candidate_id",
        how="inner"
    )
    .sort_values(
        "candidate_id"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 4. TOP-1 CHANGE FLAGS RELATIVE TO O*NET ONLY
# ------------------------------------------------------------

ablation_top1_comparison[
    "jobbank_changed_top1"
] = (
    ablation_top1_comparison[
        "O*NET Only"
    ]
    !=
    ablation_top1_comparison[
        "O*NET + Job Bank"
    ]
)


ablation_top1_comparison[
    "cip_changed_top1"
] = (
    ablation_top1_comparison[
        "O*NET Only"
    ]
    !=
    ablation_top1_comparison[
        "O*NET + CIP"
    ]
)


ablation_top1_comparison[
    "all_changed_top1"
] = (
    ablation_top1_comparison[
        "O*NET Only"
    ]
    !=
    ablation_top1_comparison[
        "All Datasets"
    ]
)


# ------------------------------------------------------------
# 5. FUNCTION FOR RANK-CHANGE ANALYSIS
# ------------------------------------------------------------

def calculate_ablation_rank_metrics(
    df,
    comparison_rank,
    baseline_rank="onet_only_rank"
):

    temp = df[
        [
            "candidate_id",
            "selected_occupation",
            baseline_rank,
            comparison_rank
        ]
    ].copy()

    temp[
        "absolute_rank_change"
    ] = (
        temp[
            comparison_rank
        ]
        -
        temp[
            baseline_rank
        ]
    ).abs()

    mean_absolute_rank_change = (
        temp[
            "absolute_rank_change"
        ]
        .mean()
    )

    maximum_rank_change = (
        temp[
            "absolute_rank_change"
        ]
        .max()
    )

    unchanged_rank_rate = (
        (
            temp[
                comparison_rank
            ]
            ==
            temp[
                baseline_rank
            ]
        )
        .mean()
        * 100
    )

    return (
        mean_absolute_rank_change,
        maximum_rank_change,
        unchanged_rank_rate
    )


# ------------------------------------------------------------
# 6. TOP-K OVERLAP FUNCTION
# ------------------------------------------------------------

def calculate_topk_overlap(
    df,
    comparison_rank,
    k,
    baseline_rank="onet_only_rank"
):

    overlap_scores = []

    for candidate_id, group in df.groupby(
        "candidate_id"
    ):

        baseline_topk = set(
            group.loc[
                group[
                    baseline_rank
                ] <= k,
                "selected_occupation"
            ]
        )

        comparison_topk = set(
            group.loc[
                group[
                    comparison_rank
                ] <= k,
                "selected_occupation"
            ]
        )

        overlap = (
            len(
                baseline_topk.intersection(
                    comparison_topk
                )
            )
            / k
        )

        overlap_scores.append(
            overlap
        )

    return (
        np.mean(
            overlap_scores
        )
        * 100
    )


# ------------------------------------------------------------
# 7. BUILD DATASET-CONTRIBUTION SUMMARY
# ------------------------------------------------------------

comparison_definitions = [
    (
        "O*NET Only",
        "onet_only_rank",
        "Baseline"
    ),
    (
        "O*NET + Job Bank",
        "onet_jobbank_rank",
        "Adds Canadian labour demand"
    ),
    (
        "O*NET + CIP",
        "onet_cip_rank",
        "Adds education alignment"
    ),
    (
        "All Datasets",
        "all_datasets_rank",
        "Adds labour demand + education"
    )
]


ablation_summary_records = []


for (
    config_name,
    rank_column,
    contribution
) in comparison_definitions:

    if rank_column == "onet_only_rank":

        mean_rank_change = 0.0
        max_rank_change = 0
        unchanged_rate = 100.0
        top1_change_count = 0
        top1_change_percentage = 0.0
        top3_overlap = 100.0
        top5_overlap = 100.0

    else:

        (
            mean_rank_change,
            max_rank_change,
            unchanged_rate
        ) = calculate_ablation_rank_metrics(
            ablation_results,
            rank_column
        )


        config_top1_column = config_name

        top1_change_count = int(
            (
                ablation_top1_comparison[
                    "O*NET Only"
                ]
                !=
                ablation_top1_comparison[
                    config_top1_column
                ]
            )
            .sum()
        )


        top1_change_percentage = (
            top1_change_count
            /
            len(
                ablation_top1_comparison
            )
            * 100
        )


        top3_overlap = (
            calculate_topk_overlap(
                ablation_results,
                rank_column,
                k=3
            )
        )


        top5_overlap = (
            calculate_topk_overlap(
                ablation_results,
                rank_column,
                k=5
            )
        )


    ablation_summary_records.append(
        {
            "configuration":
                config_name,

            "dataset_contribution":
                contribution,

            "top1_change_count_vs_onet":
                top1_change_count,

            "top1_change_percentage_vs_onet":
                top1_change_percentage,

            "mean_absolute_rank_change_vs_onet":
                mean_rank_change,

            "maximum_rank_change_vs_onet":
                max_rank_change,

            "unchanged_rank_percentage_vs_onet":
                unchanged_rate,

            "top3_overlap_percentage_vs_onet":
                top3_overlap,

            "top5_overlap_percentage_vs_onet":
                top5_overlap
        }
    )


ablation_contribution_summary = pd.DataFrame(
    ablation_summary_records
)


# ------------------------------------------------------------
# 8. ROUND NUMERIC RESULTS
# ------------------------------------------------------------

round_columns = [
    "top1_change_percentage_vs_onet",
    "mean_absolute_rank_change_vs_onet",
    "unchanged_rank_percentage_vs_onet",
    "top3_overlap_percentage_vs_onet",
    "top5_overlap_percentage_vs_onet"
]


for column in round_columns:

    ablation_contribution_summary[
        column
    ] = (
        ablation_contribution_summary[
            column
        ]
        .round(2)
    )


# ------------------------------------------------------------
# 9. TOP-1 DISTRIBUTION BY CONFIGURATION
# ------------------------------------------------------------

top1_distribution_records = []


for config_name in ablation_configurations.keys():

    distribution = (
        ablation_top1_comparison[
            config_name
        ]
        .value_counts()
    )

    for occupation, count in distribution.items():

        top1_distribution_records.append(
            {
                "configuration":
                    config_name,

                "selected_occupation":
                    occupation,

                "top1_candidate_count":
                    int(count)
            }
        )


ablation_top1_distribution = pd.DataFrame(
    top1_distribution_records
)


# ------------------------------------------------------------
# 10. VALIDATION
# ------------------------------------------------------------

expected_candidates = (
    final_hybrid[
        "candidate_id"
    ]
    .nunique()
)


top1_candidate_count = len(
    ablation_top1_comparison
)


duplicate_top1_candidates = int(
    ablation_top1_comparison[
        "candidate_id"
    ]
    .duplicated()
    .sum()
)


all_matches_final_hybrid = int(
    (
        ablation_top1_comparison[
            "All Datasets"
        ]
        ==
        final_hybrid.loc[
            final_hybrid[
                "final_hybrid_rank"
            ] == 1
        ]
        .set_index(
            "candidate_id"
        )
        .loc[
            ablation_top1_comparison[
                "candidate_id"
            ],
            "selected_occupation"
        ]
        .values
    )
    .sum()
)


print(
    "\nCandidates in Top-1 comparison:",
    top1_candidate_count
)

print(
    "Expected candidates:",
    expected_candidates
)

print(
    "Duplicate candidate rows:",
    duplicate_top1_candidates
)

print(
    "All-datasets Top-1 matches Week 6 final model:",
    all_matches_final_hybrid,
    "/",
    expected_candidates
)


# ------------------------------------------------------------
# 11. DISPLAY CONTRIBUTION SUMMARY
# ------------------------------------------------------------

print(
    "\nDataset contribution / ablation summary:"
)

display(
    ablation_contribution_summary
)


# ------------------------------------------------------------
# 12. DISPLAY TOP-1 COMPARISON SAMPLE
# ------------------------------------------------------------

print(
    "\nCandidate-level Top-1 comparison sample:"
)

display(
    ablation_top1_comparison.head(20)
)


# ------------------------------------------------------------
# 13. DISPLAY TOP-1 OCCUPATION DISTRIBUTIONS
# ------------------------------------------------------------

print(
    "\nTop-1 occupation distribution by configuration:"
)

display(
    ablation_top1_distribution
    .sort_values(
        [
            "configuration",
            "top1_candidate_count"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 14. EXPORT ABLATION RESULTS
# ------------------------------------------------------------

ablation_results_path = (
    WEEK7_OUTPUT_DIR
    / "week7_ablation_candidate_occupation_scores.csv"
)

ablation_summary_path = (
    WEEK7_OUTPUT_DIR
    / "week7_ablation_contribution_summary.csv"
)

ablation_top1_path = (
    WEEK7_OUTPUT_DIR
    / "week7_ablation_top1_comparison.csv"
)

ablation_distribution_path = (
    WEEK7_OUTPUT_DIR
    / "week7_ablation_top1_distribution.csv"
)


ablation_results.to_csv(
    ablation_results_path,
    index=False
)

ablation_contribution_summary.to_csv(
    ablation_summary_path,
    index=False
)

ablation_top1_comparison.to_csv(
    ablation_top1_path,
    index=False
)

ablation_top1_distribution.to_csv(
    ablation_distribution_path,
    index=False
)


print(
    "\nAblation files exported:"
)

print(
    ablation_results_path
)

print(
    ablation_summary_path
)

print(
    ablation_top1_path
)

print(
    ablation_distribution_path
)


# ------------------------------------------------------------
# 15. FINAL PASS / FAIL
# ------------------------------------------------------------

ablation_analysis_pass = (
    top1_candidate_count
    == expected_candidates

    and duplicate_top1_candidates
    == 0

    and all_matches_final_hybrid
    == expected_candidates

    and len(
        ablation_contribution_summary
    )
    == 4
)


if ablation_analysis_pass:

    print(
        "\n✅ PASS — Dataset contribution / "
        "ablation ranking analysis completed successfully."
    )

else:

    print(
        "\n❌ FAIL — Review ablation analysis."
    )


# ------------------------------------------------------------
# 16. METHODOLOGICAL NOTE
# ------------------------------------------------------------

print(
    "\nMethodological note:"
)

print(
    "This ablation analysis measures how Job Bank "
    "and CIP evidence change recommendation rankings "
    "relative to the O*NET-only baseline."
)

print(
    "It does not claim that a configuration is more "
    "accurate until human relevance labels are available."
)

Analyzing dataset contribution and ranking changes...

Candidates in Top-1 comparison: 63
Expected candidates: 63
Duplicate candidate rows: 0
All-datasets Top-1 matches Week 6 final model: 63 / 63

Dataset contribution / ablation summary:


,configuration,dataset_contribution,top1_change_count_vs_onet,top1_change_percentage_vs_onet,mean_absolute_rank_change_vs_onet,maximum_rank_change_vs_onet,unchanged_rank_percentage_vs_onet,top3_overlap_percentage_vs_onet,top5_overlap_percentage_vs_onet
0,O*NET Only,Baseline,0,0.00,0.00,0,100.00,100.00,100.00
1,O*NET + Job Bank,Adds Canadian labour demand,13,20.63,1.22,6,39.05,83.60,83.17
2,O*NET + CIP,Adds education alignment,1,1.59,0.47,7,72.70,89.95,86.98
3,All Datasets,Adds labour demand + education,12,19.05,1.23,9,38.20,84.66,80.63



Candidate-level Top-1 comparison sample:


,candidate_id,O*NET Only,O*NET + Job Bank,O*NET + CIP,All Datasets,jobbank_changed_top1,cip_changed_top1,all_changed_top1
0,Candidate_001,Software Developer,Software Developer,Software Developer,Software Developer,False,False,False
1,Candidate_002,Software Developer,Software Developer,Software Developer,Software Developer,False,False,False
2,Candidate_003,Administrative Assistant,Administrative Assistant,Administrative Assistant,Administrative Assistant,False,False,False
3,Candidate_004,Software Developer,Software Developer,Software Developer,Software Developer,False,False,False
4,Candidate_005,Software Developer,Software Developer,Software Developer,Software Developer,False,False,False
5,Candidate_006,Software Developer,Software Developer,Software Developer,Software Developer,False,False,False
6,Candidate_007,Administrative Assistant,Food Service Supervisor,Administrative Assistant,Food Service Supervisor,True,False,True
7,Candidate_008,Software Developer,Software Developer,Software Developer,Software Developer,False,False,False
8,Candidate_009,Administrative Assistant,Administrative Assistant,Administrative Assistant,Administrative Assistant,False,False,False
9,Candidate_010,Administrative Assistant,Food Service Supervisor,Administrative Assistant,Food Service Supervisor,True,False,True



Top-1 occupation distribution by configuration:


,configuration,selected_occupation,top1_candidate_count
0,All Datasets,Software Developer,35
1,All Datasets,Administrative Assistant,13
2,All Datasets,Food Service Supervisor,11
3,All Datasets,Office Manager,4
4,O*NET + CIP,Software Developer,38
5,O*NET + CIP,Administrative Assistant,21
6,O*NET + CIP,Food Service Supervisor,3
7,O*NET + CIP,Office Manager,1
8,O*NET + Job Bank,Software Developer,35
9,O*NET + Job Bank,Administrative Assistant,11



Ablation files exported:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7\week7_ablation_candidate_occupation_scores.csv
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7\week7_ablation_contribution_summary.csv
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7\week7_ablation_top1_comparison.csv
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7\week7_ablation_top1_distribution.csv

✅ PASS — Dataset contribution / ablation ranking analysis completed successfully.

Methodological note:
This ablation analysis measures how Job Bank and CIP evidence change recommendation rankings relative to the O*NET-only baseline.
It does not claim that a configuration is more accurate until human relevance labels are available.


In [175]:
# ============================================================
# CELL 73 — Formal Error-Analysis Framework
# ============================================================

print("Creating Week 7 formal error-analysis framework...")


def build_week7_error_analysis(
    skill_extraction_review,
    ranking_review,
    skill_gap_review,
    rag_review
):
    """
    Build formal Week 7 error-analysis tables.

    Human-dependent errors are produced only when the
    corresponding human-review labels are available.

    Sections:
    1. Skill extraction false positives / false negatives
    2. Recommendation ranking failures
    3. Skill-gap false positives / false negatives
    4. RAG low-quality / unsupported-claim cases
    """

    error_summary_records = []


    # ========================================================
    # 1. SKILL EXTRACTION ERROR ANALYSIS
    # ========================================================

    skill_eval = skill_extraction_review.copy()

    skill_eval[
        "human_skill_present"
    ] = pd.to_numeric(
        skill_eval[
            "human_skill_present"
        ],
        errors="coerce"
    )


    skill_review_complete = (
        skill_eval[
            "human_skill_present"
        ]
        .notna()
        .all()
    )


    if skill_review_complete:

        skill_eval[
            "human_skill_present"
        ] = (
            skill_eval[
                "human_skill_present"
            ]
            .astype(int)
        )

        skill_extraction_false_positives = (
            skill_eval[
                (
                    skill_eval[
                        "system_extracted"
                    ] == 1
                )
                &
                (
                    skill_eval[
                        "human_skill_present"
                    ] == 0
                )
            ]
            .copy()
        )

        skill_extraction_false_negatives = (
            skill_eval[
                (
                    skill_eval[
                        "system_extracted"
                    ] == 0
                )
                &
                (
                    skill_eval[
                        "human_skill_present"
                    ] == 1
                )
            ]
            .copy()
        )

        error_summary_records.append(
            {
                "evaluation_area":
                    "Skill Extraction",

                "error_type":
                    "False Positive",

                "error_count":
                    len(
                        skill_extraction_false_positives
                    ),

                "review_status":
                    "Complete"
            }
        )

        error_summary_records.append(
            {
                "evaluation_area":
                    "Skill Extraction",

                "error_type":
                    "False Negative",

                "error_count":
                    len(
                        skill_extraction_false_negatives
                    ),

                "review_status":
                    "Complete"
            }
        )

    else:

        skill_extraction_false_positives = pd.DataFrame()
        skill_extraction_false_negatives = pd.DataFrame()

        error_summary_records.append(
            {
                "evaluation_area":
                    "Skill Extraction",

                "error_type":
                    "FP/FN analysis",

                "error_count":
                    np.nan,

                "review_status":
                    "Pending human review"
            }
        )


    # ========================================================
    # 2. RECOMMENDATION RANKING ERROR ANALYSIS
    # ========================================================

    ranking_eval = ranking_review.copy()

    ranking_eval[
        "human_relevant"
    ] = pd.to_numeric(
        ranking_eval[
            "human_relevant"
        ],
        errors="coerce"
    )

    ranking_eval[
        "human_relevance_grade"
    ] = pd.to_numeric(
        ranking_eval[
            "human_relevance_grade"
        ],
        errors="coerce"
    )


    ranking_review_complete = (
        ranking_eval[
            [
                "human_relevant",
                "human_relevance_grade"
            ]
        ]
        .notna()
        .all()
        .all()
    )


    if ranking_review_complete:

        ranking_eval[
            "human_relevant"
        ] = (
            ranking_eval[
                "human_relevant"
            ]
            .astype(int)
        )

        ranking_eval[
            "human_relevance_grade"
        ] = (
            ranking_eval[
                "human_relevance_grade"
            ]
            .astype(int)
        )


        # ----------------------------------------------------
        # Candidates where Top-1 recommendation was irrelevant
        # ----------------------------------------------------

        ranking_top1_failures = (
            ranking_eval[
                (
                    ranking_eval[
                        "system_rank"
                    ] == 1
                )
                &
                (
                    ranking_eval[
                        "human_relevant"
                    ] == 0
                )
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Relevant occupations missed from Top-5
        # ----------------------------------------------------

        ranking_relevant_below_top5 = (
            ranking_eval[
                (
                    ranking_eval[
                        "system_rank"
                    ] > 5
                )
                &
                (
                    ranking_eval[
                        "human_relevant"
                    ] == 1
                )
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Highly relevant occupations ranked below Top-5
        #
        # Grade 3 = highly relevant
        # ----------------------------------------------------

        ranking_high_relevance_misses = (
            ranking_eval[
                (
                    ranking_eval[
                        "system_rank"
                    ] > 5
                )
                &
                (
                    ranking_eval[
                        "human_relevance_grade"
                    ] == 3
                )
            ]
            .copy()
        )


        error_summary_records.append(
            {
                "evaluation_area":
                    "Recommendation Ranking",

                "error_type":
                    "Irrelevant Top-1",

                "error_count":
                    len(
                        ranking_top1_failures
                    ),

                "review_status":
                    "Complete"
            }
        )

        error_summary_records.append(
            {
                "evaluation_area":
                    "Recommendation Ranking",

                "error_type":
                    "Relevant occupation below Top-5",

                "error_count":
                    len(
                        ranking_relevant_below_top5
                    ),

                "review_status":
                    "Complete"
            }
        )

        error_summary_records.append(
            {
                "evaluation_area":
                    "Recommendation Ranking",

                "error_type":
                    "Highly relevant occupation below Top-5",

                "error_count":
                    len(
                        ranking_high_relevance_misses
                    ),

                "review_status":
                    "Complete"
            }
        )

    else:

        ranking_top1_failures = pd.DataFrame()
        ranking_relevant_below_top5 = pd.DataFrame()
        ranking_high_relevance_misses = pd.DataFrame()

        error_summary_records.append(
            {
                "evaluation_area":
                    "Recommendation Ranking",

                "error_type":
                    "Ranking error analysis",

                "error_count":
                    np.nan,

                "review_status":
                    "Pending human review"
            }
        )


    # ========================================================
    # 3. SKILL-GAP ERROR ANALYSIS
    # ========================================================

    gap_eval = skill_gap_review.copy()

    gap_eval[
        "human_gap"
    ] = pd.to_numeric(
        gap_eval[
            "human_gap"
        ],
        errors="coerce"
    )


    gap_review_complete = (
        gap_eval[
            "human_gap"
        ]
        .notna()
        .all()
    )


    if gap_review_complete:

        gap_eval[
            "human_gap"
        ] = (
            gap_eval[
                "human_gap"
            ]
            .astype(int)
        )


        skill_gap_false_positives = (
            gap_eval[
                (
                    gap_eval[
                        "system_gap"
                    ] == 1
                )
                &
                (
                    gap_eval[
                        "human_gap"
                    ] == 0
                )
            ]
            .copy()
        )


        skill_gap_false_negatives = (
            gap_eval[
                (
                    gap_eval[
                        "system_gap"
                    ] == 0
                )
                &
                (
                    gap_eval[
                        "human_gap"
                    ] == 1
                )
            ]
            .copy()
        )


        error_summary_records.append(
            {
                "evaluation_area":
                    "Skill Gap",

                "error_type":
                    "False Positive Gap",

                "error_count":
                    len(
                        skill_gap_false_positives
                    ),

                "review_status":
                    "Complete"
            }
        )

        error_summary_records.append(
            {
                "evaluation_area":
                    "Skill Gap",

                "error_type":
                    "False Negative Gap",

                "error_count":
                    len(
                        skill_gap_false_negatives
                    ),

                "review_status":
                    "Complete"
            }
        )

    else:

        skill_gap_false_positives = pd.DataFrame()
        skill_gap_false_negatives = pd.DataFrame()

        error_summary_records.append(
            {
                "evaluation_area":
                    "Skill Gap",

                "error_type":
                    "FP/FN gap analysis",

                "error_count":
                    np.nan,

                "review_status":
                    "Pending human review"
            }
        )


    # ========================================================
    # 4. RAG ERROR ANALYSIS
    # ========================================================

    rag_eval = rag_review.copy()

    rag_rating_columns = [
        "relevance_rating",
        "correctness_rating",
        "groundedness_rating",
        "citation_correctness_rating",
        "completeness_rating",
        "clarity_rating",
        "usefulness_rating"
    ]


    for column in rag_rating_columns:

        rag_eval[
            column
        ] = pd.to_numeric(
            rag_eval[
                column
            ],
            errors="coerce"
        )


    rag_eval[
        "unsupported_claim_count"
    ] = pd.to_numeric(
        rag_eval[
            "unsupported_claim_count"
        ],
        errors="coerce"
    )


    rag_review_complete = (
        rag_eval[
            rag_rating_columns
            +
            [
                "unsupported_claim_count"
            ]
        ]
        .notna()
        .all()
        .all()
    )


    if rag_review_complete:

        rag_eval[
            "mean_rag_rating"
        ] = (
            rag_eval[
                rag_rating_columns
            ]
            .mean(
                axis=1
            )
        )


        # ----------------------------------------------------
        # Low-quality explanations:
        # mean human rating below 3
        # ----------------------------------------------------

        rag_low_quality_cases = (
            rag_eval[
                rag_eval[
                    "mean_rag_rating"
                ] < 3
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Grounding failures:
        # groundedness <= 2
        # ----------------------------------------------------

        rag_grounding_failures = (
            rag_eval[
                rag_eval[
                    "groundedness_rating"
                ] <= 2
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Citation failures:
        # citation correctness <= 2
        # ----------------------------------------------------

        rag_citation_failures = (
            rag_eval[
                rag_eval[
                    "citation_correctness_rating"
                ] <= 2
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Unsupported claims
        # ----------------------------------------------------

        rag_unsupported_cases = (
            rag_eval[
                rag_eval[
                    "unsupported_claim_count"
                ] > 0
            ]
            .copy()
        )


        error_summary_records.append(
            {
                "evaluation_area":
                    "RAG",

                "error_type":
                    "Low-quality explanation",

                "error_count":
                    len(
                        rag_low_quality_cases
                    ),

                "review_status":
                    "Complete"
            }
        )

        error_summary_records.append(
            {
                "evaluation_area":
                    "RAG",

                "error_type":
                    "Low groundedness",

                "error_count":
                    len(
                        rag_grounding_failures
                    ),

                "review_status":
                    "Complete"
            }
        )

        error_summary_records.append(
            {
                "evaluation_area":
                    "RAG",

                "error_type":
                    "Low citation correctness",

                "error_count":
                    len(
                        rag_citation_failures
                    ),

                "review_status":
                    "Complete"
            }
        )

        error_summary_records.append(
            {
                "evaluation_area":
                    "RAG",

                "error_type":
                    "Contains unsupported claim",

                "error_count":
                    len(
                        rag_unsupported_cases
                    ),

                "review_status":
                    "Complete"
            }
        )

    else:

        rag_low_quality_cases = pd.DataFrame()
        rag_grounding_failures = pd.DataFrame()
        rag_citation_failures = pd.DataFrame()
        rag_unsupported_cases = pd.DataFrame()

        error_summary_records.append(
            {
                "evaluation_area":
                    "RAG",

                "error_type":
                    "RAG error analysis",

                "error_count":
                    np.nan,

                "review_status":
                    "Pending human review"
            }
        )


    # ========================================================
    # 5. BUILD SUMMARY
    # ========================================================

    error_analysis_summary = pd.DataFrame(
        error_summary_records
    )


    print(
        "\nWeek 7 error-analysis status:"
    )

    display(
        error_analysis_summary
    )


    # ========================================================
    # 6. COMPLETION STATUS
    # ========================================================

    print(
        "\nHuman-review completion status:"
    )

    print(
        "Skill extraction:",
        "Complete"
        if skill_review_complete
        else "Pending"
    )

    print(
        "Recommendation ranking:",
        "Complete"
        if ranking_review_complete
        else "Pending"
    )

    print(
        "Skill-gap evaluation:",
        "Complete"
        if gap_review_complete
        else "Pending"
    )

    print(
        "RAG evaluation:",
        "Complete"
        if rag_review_complete
        else "Pending"
    )


    if (
        skill_review_complete
        and ranking_review_complete
        and gap_review_complete
        and rag_review_complete
    ):

        print(
            "\n✅ Full human-grounded error analysis "
            "completed successfully."
        )

    else:

        print(
            "\n⚠️ Formal error analysis is ready, "
            "but final error counts are intentionally "
            "withheld until human review is complete."
        )


    # ========================================================
    # 7. RETURN RESULTS
    # ========================================================

    return {
        "summary":
            error_analysis_summary,

        "skill_extraction_false_positives":
            skill_extraction_false_positives,

        "skill_extraction_false_negatives":
            skill_extraction_false_negatives,

        "ranking_top1_failures":
            ranking_top1_failures,

        "ranking_relevant_below_top5":
            ranking_relevant_below_top5,

        "ranking_high_relevance_misses":
            ranking_high_relevance_misses,

        "skill_gap_false_positives":
            skill_gap_false_positives,

        "skill_gap_false_negatives":
            skill_gap_false_negatives,

        "rag_low_quality_cases":
            rag_low_quality_cases,

        "rag_grounding_failures":
            rag_grounding_failures,

        "rag_citation_failures":
            rag_citation_failures,

        "rag_unsupported_cases":
            rag_unsupported_cases
    }


print(
    "✓ Week 7 error-analysis framework "
    "created successfully."
)


# ============================================================
# TEST WITH CURRENT HUMAN-REVIEW TEMPLATES
# ============================================================

week7_error_analysis = build_week7_error_analysis(
    skill_extraction_ground_truth,
    recommendation_ground_truth,
    skill_gap_ground_truth,
    rag_human_review
)

Creating Week 7 formal error-analysis framework...
✓ Week 7 error-analysis framework created successfully.

Week 7 error-analysis status:


,evaluation_area,error_type,error_count,review_status
0,Skill Extraction,FP/FN analysis,NaN,Pending human review
1,Recommendation Ranking,Ranking error analysis,NaN,Pending human review
2,Skill Gap,FP/FN gap analysis,NaN,Pending human review
3,RAG,RAG error analysis,NaN,Pending human review



Human-review completion status:
Skill extraction: Pending
Recommendation ranking: Pending
Skill-gap evaluation: Pending
RAG evaluation: Pending

⚠️ Formal error analysis is ready, but final error counts are intentionally withheld until human review is complete.


In [177]:
# ============================================================
# CELL 74 — Formal Week 7 Project Limitations
# ============================================================

print("Creating formal Week 7 project limitations table...")


week7_limitations = pd.DataFrame(
    [
        {
            "limitation_area":
                "Recommendation Scope",

            "limitation":
                "The recommendation system is restricted to the 15 occupations selected for the capstone analysis.",

            "potential_impact":
                "A candidate may have a strong match to an occupation outside the predefined occupation set.",

            "mitigation_or_future_work":
                "Expand the occupation catalogue and repeat the crosswalk, skill-profile, demand, and education integration pipeline."
        },

        {
            "limitation_area":
                "Model Type",

            "limitation":
                "The system is a content-based ranking system rather than a supervised career-classification model.",

            "potential_impact":
                "Recommendation quality cannot be interpreted as classification accuracy without independent human relevance labels.",

            "mitigation_or_future_work":
                "Use the human-reviewed ranking matrix to calculate Top-K metrics, MRR, and NDCG and later develop supervised models if sufficient labelled data become available."
        },

        {
            "limitation_area":
                "Skill Representation",

            "limitation":
                "The current O*NET candidate-skill matrix evaluates 10 selected O*NET skill dimensions.",

            "potential_impact":
                "Candidate capabilities outside these dimensions may not be fully represented in the skill-gap calculation.",

            "mitigation_or_future_work":
                "Expand the skill taxonomy using additional O*NET, ESCO, domain-specific, and occupation-specific skill dimensions."
        },

        {
            "limitation_area":
                "Skill Extraction",

            "limitation":
                "Candidate skill evidence is automatically extracted from CV text and may contain missed skills or incorrectly detected skills.",

            "potential_impact":
                "Extraction errors can propagate into candidate matching and skill-gap recommendations.",

            "mitigation_or_future_work":
                "Complete the human-reviewed skill-extraction matrix and use Precision, Recall, F1, false-positive, and false-negative analysis to refine extraction rules."
        },

        {
            "limitation_area":
                "Skill-Gap Inference",

            "limitation":
                "A skill is treated as existing or missing according to evidence detected in the candidate CV.",

            "potential_impact":
                "Absence of a skill from a CV does not necessarily mean that the candidate does not possess that skill.",

            "mitigation_or_future_work":
                "Combine CV evidence with candidate questionnaires, portfolios, certifications, assessments, and human validation."
        },

        {
            "limitation_area":
                "Labour Demand",

            "limitation":
                "Job Bank labour-demand scores are derived from the project dataset and represent the available observation period rather than continuously updated labour-market conditions.",

            "potential_impact":
                "Demand rankings may change as vacancies, locations, and labour-market conditions change.",

            "mitigation_or_future_work":
                "Refresh Job Bank data periodically and introduce time-aware demand indicators."
        },

        {
            "limitation_area":
                "Geographic Demand",

            "limitation":
                "The demand component summarizes Canadian labour-market evidence and does not model every candidate's specific relocation preferences or local commuting constraints.",

            "potential_impact":
                "A high-demand occupation nationally may not be equally attractive in the candidate's preferred location.",

            "mitigation_or_future_work":
                "Add province, city, remote-work, and candidate-location preference filters."
        },

        {
            "limitation_area":
                "Education Alignment",

            "limitation":
                "CIP education recommendations depend on available occupation-to-CIP crosswalk mappings.",

            "potential_impact":
                "Some candidate-occupation pairs have no mapped CIP pathway even when relevant training may exist.",

            "mitigation_or_future_work":
                "Expand and manually validate CIP mappings and incorporate Canadian institutional program data."
        },

        {
            "limitation_area":
                "Crosswalk Mapping",

            "limitation":
                "The project integrates occupational datasets through NOC, SOC, O*NET-SOC, and CIP crosswalk relationships.",

            "potential_impact":
                "Many-to-many or imperfect crosswalk relationships can introduce ambiguity between occupational classifications.",

            "mitigation_or_future_work":
                "Perform additional expert validation of ambiguous mappings and maintain mapping-confidence indicators."
        },

        {
            "limitation_area":
                "Hybrid Weighting",

            "limitation":
                "The final recommendation score uses project-defined weights: 0.35 skill, 0.35 semantic similarity, 0.20 labour demand, and 0.10 education alignment.",

            "potential_impact":
                "Different weighting choices may change occupation rankings.",

            "mitigation_or_future_work":
                "Perform sensitivity analysis and optimize weights using completed human relevance judgments."
        },

        {
            "limitation_area":
                "RAG Retrieval",

            "limitation":
                "RAG explanations are grounded only in the project's indexed O*NET, Job Bank, CIP, task, skill, and integrated-profile documents.",

            "potential_impact":
                "Information not represented in the knowledge base cannot reliably support the generated explanation.",

            "mitigation_or_future_work":
                "Expand the knowledge base with validated occupation, training, certification, and labour-market sources."
        },

        {
            "limitation_area":
                "RAG Generation",

            "limitation":
                "Retrieved evidence reduces unsupported generation risk but does not guarantee that every generated statement is correct or fully supported.",

            "potential_impact":
                "Explanations may contain incomplete, weakly supported, or incorrectly interpreted statements.",

            "mitigation_or_future_work":
                "Complete the human RAG evaluation for relevance, correctness, groundedness, citation correctness, completeness, clarity, usefulness, and unsupported claims."
        },

        {
            "limitation_area":
                "Human Evaluation Sample",

            "limitation":
                "The formal human-review templates currently use a 20-candidate evaluation subset.",

            "potential_impact":
                "Evaluation results from the review subset may not fully represent all 63 candidates.",

            "mitigation_or_future_work":
                "Increase the reviewed sample or conduct full-candidate evaluation where resources permit."
        },

        {
            "limitation_area":
                "Human Ground Truth",

            "limitation":
                "Human-review fields are currently blank, so ground-truth-dependent evaluation metrics are intentionally not reported yet.",

            "potential_impact":
                "Precision, Recall, F1, Top-K accuracy, MRR, NDCG, skill-gap agreement, and final RAG quality scores cannot yet be interpreted.",

            "mitigation_or_future_work":
                "Complete the human-review templates before calculating or reporting final evaluation metrics."
        }
    ]
)


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

required_limitation_columns = [
    "limitation_area",
    "limitation",
    "potential_impact",
    "mitigation_or_future_work"
]


missing_limitation_columns = [
    column
    for column in required_limitation_columns
    if column not in week7_limitations.columns
]


empty_limitation_rows = int(
    week7_limitations[
        required_limitation_columns
    ]
    .fillna("")
    .astype(str)
    .apply(
        lambda column:
            column.str.strip().eq("")
    )
    .any(
        axis=1
    )
    .sum()
)


duplicate_limitation_areas = int(
    week7_limitations[
        "limitation_area"
    ]
    .duplicated()
    .sum()
)


print(
    "\nNumber of documented limitations:",
    len(week7_limitations)
)

print(
    "Missing required columns:",
    missing_limitation_columns
)

print(
    "Rows with empty required fields:",
    empty_limitation_rows
)

print(
    "Duplicate limitation areas:",
    duplicate_limitation_areas
)


print(
    "\nWeek 7 limitations:"
)

display(
    week7_limitations
)


# ------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------

week7_limitations_path = (
    WEEK7_OUTPUT_DIR
    / "week7_project_limitations.csv"
)


week7_limitations.to_csv(
    week7_limitations_path,
    index=False
)


print(
    "\nLimitations file exported:"
)

print(
    week7_limitations_path
)

print(
    "File exists:",
    week7_limitations_path.exists()
)

print(
    "File size:",
    week7_limitations_path.stat().st_size,
    "bytes"
)


# ------------------------------------------------------------
# FINAL CHECK
# ------------------------------------------------------------

limitations_pass = (
    len(week7_limitations) > 0
    and len(missing_limitation_columns) == 0
    and empty_limitation_rows == 0
    and duplicate_limitation_areas == 0
    and week7_limitations_path.exists()
)


if limitations_pass:

    print(
        "\n✅ PASS — Week 7 project limitations "
        "documented and exported successfully."
    )

else:

    print(
        "\n❌ FAIL — Review Week 7 limitation documentation."
    )

Creating formal Week 7 project limitations table...

Number of documented limitations: 14
Missing required columns: []
Rows with empty required fields: 0
Duplicate limitation areas: 0

Week 7 limitations:


,limitation_area,limitation,potential_impact,mitigation_or_future_work
0,Recommendation Scope,The recommendation system is restricted to the 15 occupations selected for the capstone analysis.,A candidate may have a strong match to an occupation outside the predefined occupation set.,"Expand the occupation catalogue and repeat the crosswalk, skill-profile, demand, and education integration pipeline."
1,Model Type,The system is a content-based ranking system rather than a supervised career-classification model.,Recommendation quality cannot be interpreted as classification accuracy without independent human relevance labels.,"Use the human-reviewed ranking matrix to calculate Top-K metrics, MRR, and NDCG and later develop supervised models if sufficient labelled data become available."
2,Skill Representation,The current O*NET candidate-skill matrix evaluates 10 selected O*NET skill dimensions.,Candidate capabilities outside these dimensions may not be fully represented in the skill-gap calculation.,"Expand the skill taxonomy using additional O*NET, ESCO, domain-specific, and occupation-specific skill dimensions."
3,Skill Extraction,Candidate skill evidence is automatically extracted from CV text and may contain missed skills or incorrectly detected skills.,Extraction errors can propagate into candidate matching and skill-gap recommendations.,"Complete the human-reviewed skill-extraction matrix and use Precision, Recall, F1, false-positive, and false-negative analysis to refine extraction rules."
4,Skill-Gap Inference,A skill is treated as existing or missing according to evidence detected in the candidate CV.,Absence of a skill from a CV does not necessarily mean that the candidate does not possess that skill.,"Combine CV evidence with candidate questionnaires, portfolios, certifications, assessments, and human validation."
5,Labour Demand,Job Bank labour-demand scores are derived from the project dataset and represent the available observation period rather than continuously updated labour-market conditions.,"Demand rankings may change as vacancies, locations, and labour-market conditions change.",Refresh Job Bank data periodically and introduce time-aware demand indicators.
6,Geographic Demand,The demand component summarizes Canadian labour-market evidence and does not model every candidate's specific relocation preferences or local commuting constraints.,A high-demand occupation nationally may not be equally attractive in the candidate's preferred location.,"Add province, city, remote-work, and candidate-location preference filters."
7,Education Alignment,CIP education recommendations depend on available occupation-to-CIP crosswalk mappings.,Some candidate-occupation pairs have no mapped CIP pathway even when relevant training may exist.,Expand and manually validate CIP mappings and incorporate Canadian institutional program data.
8,Crosswalk Mapping,"The project integrates occupational datasets through NOC, SOC, O*NET-SOC, and CIP crosswalk relationships.",Many-to-many or imperfect crosswalk relationships can introduce ambiguity between occupational classifications.,Perform additional expert validation of ambiguous mappings and maintain mapping-confidence indicators.
9,Hybrid Weighting,"The final recommendation score uses project-defined weights: 0.35 skill, 0.35 semantic similarity, 0.20 labour demand, and 0.10 education alignment.",Different weighting choices may change occupation rankings.,Perform sensitivity analysis and optimize weights using completed human relevance judgments.



Limitations file exported:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7\week7_project_limitations.csv
File exists: True
File size: 4955 bytes

✅ PASS — Week 7 project limitations documented and exported successfully.


In [179]:
# ============================================================
# CELL 75 — MASTER WEEK 7 DELIVERABLE VALIDATION
# ============================================================

print("=" * 70)
print("WEEK 7 — MASTER DELIVERABLE AND REQUIREMENT VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. DEFINE REQUIRED WEEK 7 OUTPUT FILES
# ------------------------------------------------------------

required_week7_files = {

    # --------------------------------------------------------
    # Prescriptive Analytics
    # --------------------------------------------------------

    "Missing Skill Priorities":
        "week7_missing_skill_priorities.csv",

    "Candidate Prescriptive Recommendations":
        "week7_candidate_prescriptive_recommendations.csv",

    "Top-1 Prescriptive Skill Gaps":
        "week7_top1_prescriptive_skill_gaps.csv",

    "Final Prescriptive Recommendations":
        "week7_final_prescriptive_recommendations.csv",

    "Prescriptive Validation Summary":
        "week7_prescriptive_validation_summary.csv",


    # --------------------------------------------------------
    # RAG
    # --------------------------------------------------------

    "RAG Knowledge Base":
        "week7_rag_knowledge_base.csv",

    "RAG Embeddings":
        "week7_rag_embeddings.npy",

    "FAISS Index":
        "week7_rag_faiss.index",

    "Candidate RAG Explanations":
        "week7_candidate_rag_explanations.csv",

    "RAG Validation Summary":
        "week7_rag_validation_summary.csv",


    # --------------------------------------------------------
    # Human Review / Evaluation
    # --------------------------------------------------------

    "Skill Extraction Positive Review":
        "week7_skill_extraction_human_review.csv",

    "Skill Extraction Ground Truth":
        "week7_skill_extraction_ground_truth_review.csv",

    "Recommendation Top-5 Review":
        "week7_recommendation_ranking_human_review.csv",

    "Recommendation Ranking Ground Truth":
        "week7_recommendation_ranking_ground_truth_review.csv",

    "Skill Gap Ground Truth":
        "week7_skill_gap_ground_truth_review.csv",

    "RAG Human Review":
        "week7_rag_human_review.csv",


    # --------------------------------------------------------
    # Ablation Analysis
    # --------------------------------------------------------

    "Ablation Candidate-Occupation Scores":
        "week7_ablation_candidate_occupation_scores.csv",

    "Ablation Contribution Summary":
        "week7_ablation_contribution_summary.csv",

    "Ablation Top-1 Comparison":
        "week7_ablation_top1_comparison.csv",

    "Ablation Top-1 Distribution":
        "week7_ablation_top1_distribution.csv",


    # --------------------------------------------------------
    # Limitations
    # --------------------------------------------------------

    "Project Limitations":
        "week7_project_limitations.csv"
}


# ------------------------------------------------------------
# 2. CHECK FILE EXISTENCE AND SIZE
# ------------------------------------------------------------

file_validation_records = []


for deliverable, filename in required_week7_files.items():

    file_path = (
        WEEK7_OUTPUT_DIR
        / filename
    )

    file_exists = (
        file_path.exists()
    )

    file_size = (
        file_path.stat().st_size
        if file_exists
        else 0
    )

    file_validation_records.append(
        {
            "deliverable":
                deliverable,

            "filename":
                filename,

            "exists":
                file_exists,

            "file_size_bytes":
                file_size,

            "status":
                (
                    "PASS"
                    if (
                        file_exists
                        and file_size > 0
                    )
                    else "FAIL"
                )
        }
    )


week7_file_validation = pd.DataFrame(
    file_validation_records
)


print(
    "\n1. WEEK 7 OUTPUT FILE VALIDATION"
)

display(
    week7_file_validation
)


existing_file_count = int(
    week7_file_validation[
        "exists"
    ]
    .sum()
)

passing_file_count = int(
    (
        week7_file_validation[
            "status"
        ] == "PASS"
    )
    .sum()
)

required_file_count = len(
    week7_file_validation
)


print(
    "\nRequired files:",
    required_file_count
)

print(
    "Files found:",
    existing_file_count
)

print(
    "Files passing validation:",
    passing_file_count
)


# ------------------------------------------------------------
# 3. HUMAN-REVIEW COMPLETION STATUS
# ------------------------------------------------------------

def calculate_review_completion(
    dataframe,
    columns
):

    if dataframe is None or len(dataframe) == 0:

        return 0.0

    temp = dataframe[
        columns
    ].copy()

    for column in columns:

        temp[column] = pd.to_numeric(
            temp[column],
            errors="coerce"
        )

    completed_cells = int(
        temp
        .notna()
        .sum()
        .sum()
    )

    total_cells = (
        len(temp)
        * len(columns)
    )

    return (
        completed_cells
        / total_cells
        * 100
        if total_cells > 0
        else 0.0
    )


skill_extraction_review_completion = (
    calculate_review_completion(
        skill_extraction_ground_truth,
        [
            "human_skill_present"
        ]
    )
)


recommendation_review_completion = (
    calculate_review_completion(
        recommendation_ground_truth,
        [
            "human_relevant",
            "human_relevance_grade"
        ]
    )
)


skill_gap_review_completion = (
    calculate_review_completion(
        skill_gap_ground_truth,
        [
            "human_gap",
            "human_gap_severity",
            "expert_rating"
        ]
    )
)


rag_review_completion_percentage = (
    calculate_review_completion(
        rag_human_review,
        [
            "relevance_rating",
            "correctness_rating",
            "groundedness_rating",
            "citation_correctness_rating",
            "completeness_rating",
            "clarity_rating",
            "usefulness_rating",
            "unsupported_claim_count"
        ]
    )
)


human_review_status = pd.DataFrame(
    [
        {
            "evaluation":
                "Skill Extraction",

            "review_rows":
                len(
                    skill_extraction_ground_truth
                ),

            "completion_percentage":
                skill_extraction_review_completion,

            "status":
                (
                    "Complete"
                    if skill_extraction_review_completion
                    == 100
                    else "Pending Human Review"
                )
        },

        {
            "evaluation":
                "Recommendation Ranking",

            "review_rows":
                len(
                    recommendation_ground_truth
                ),

            "completion_percentage":
                recommendation_review_completion,

            "status":
                (
                    "Complete"
                    if recommendation_review_completion
                    == 100
                    else "Pending Human Review"
                )
        },

        {
            "evaluation":
                "Skill Gap",

            "review_rows":
                len(
                    skill_gap_ground_truth
                ),

            "completion_percentage":
                skill_gap_review_completion,

            "status":
                (
                    "Complete"
                    if skill_gap_review_completion
                    == 100
                    else "Pending Human Review"
                )
        },

        {
            "evaluation":
                "RAG",

            "review_rows":
                len(
                    rag_human_review
                ),

            "completion_percentage":
                rag_review_completion_percentage,

            "status":
                (
                    "Complete"
                    if rag_review_completion_percentage
                    == 100
                    else "Pending Human Review"
                )
        }
    ]
)


human_review_status[
    "completion_percentage"
] = (
    human_review_status[
        "completion_percentage"
    ]
    .round(2)
)


print(
    "\n2. HUMAN-REVIEW STATUS"
)

display(
    human_review_status
)


# ------------------------------------------------------------
# 4. PROFESSOR REQUIREMENT COVERAGE
# ------------------------------------------------------------

requirement_coverage = pd.DataFrame(
    [
        {
            "requirement":
                "1. Prescriptive skill-gap analysis",

            "implementation":
                "Candidate vs occupation O*NET skills; existing/missing skills; importance; level; demand; education integration.",

            "status":
                "COMPLETE"
        },

        {
            "requirement":
                "2. Skill-priority model",

            "implementation":
                "O*NET importance/level gap priority plus Canadian labour-demand adjustment.",

            "status":
                "COMPLETE"
        },

        {
            "requirement":
                "3. Career-action recommendations",

            "implementation":
                "Priority missing skills, demand level, CIP education pathway status, and candidate career actions.",

            "status":
                "COMPLETE"
        },

        {
            "requirement":
                "4. RAG retrieval",

            "implementation":
                "707 grounded documents from O*NET occupations, skills, tasks, Job Bank demand, CIP, and integrated profiles; MiniLM embeddings and FAISS retrieval.",

            "status":
                "COMPLETE"
        },

        {
            "requirement":
                "5. Grounded RAG explanations",

            "implementation":
                "63 candidate explanations with recommendation evidence, exact skill-gap evidence, tasks, demand, education, and source document IDs.",

            "status":
                "COMPLETE"
        },

        {
            "requirement":
                "6. Skill-extraction evaluation",

            "implementation":
                "200-row human ground-truth matrix and Precision/Recall/F1/FP/FN evaluation function.",

            "status":
                (
                    "FRAMEWORK COMPLETE — HUMAN REVIEW PENDING"
                    if skill_extraction_review_completion < 100
                    else "COMPLETE"
                )
        },

        {
            "requirement":
                "7. Recommendation ranking evaluation",

            "implementation":
                "300-row full-ranking ground-truth matrix with Top-1/3/5, Precision@K, Recall@K, MRR, and NDCG evaluation function.",

            "status":
                (
                    "FRAMEWORK COMPLETE — HUMAN REVIEW PENDING"
                    if recommendation_review_completion < 100
                    else "COMPLETE"
                )
        },

        {
            "requirement":
                "8. Skill-gap evaluation",

            "implementation":
                "200-row human gap-review matrix with Precision, Recall, F1, agreement, severity, and expert-rating evaluation.",

            "status":
                (
                    "FRAMEWORK COMPLETE — HUMAN REVIEW PENDING"
                    if skill_gap_review_completion < 100
                    else "COMPLETE"
                )
        },

        {
            "requirement":
                "9. RAG evaluation",

            "implementation":
                "20-candidate human review for relevance, correctness, groundedness, citation correctness, completeness, clarity, usefulness, and unsupported claims.",

            "status":
                (
                    "FRAMEWORK COMPLETE — HUMAN REVIEW PENDING"
                    if rag_review_completion_percentage < 100
                    else "COMPLETE"
                )
        },

        {
            "requirement":
                "10. Dataset contribution / ablation",

            "implementation":
                "O*NET only, O*NET + Job Bank, O*NET + CIP, and all-dataset configurations with ranking-change and Top-K overlap analysis.",

            "status":
                "COMPLETE"
        },

        {
            "requirement":
                "11. Error analysis and limitations",

            "implementation":
                "Human-grounded FP/FN/ranking/RAG error-analysis framework plus 14 documented project limitations.",

            "status":
                (
                    "FRAMEWORK COMPLETE — HUMAN REVIEW PENDING"
                    if (
                        skill_extraction_review_completion < 100
                        or recommendation_review_completion < 100
                        or skill_gap_review_completion < 100
                        or rag_review_completion_percentage < 100
                    )
                    else "COMPLETE"
                )
        }
    ]
)


print(
    "\n3. PROFESSOR REQUIREMENT COVERAGE"
)

display(
    requirement_coverage
)


# ------------------------------------------------------------
# 5. CORE DATASET VALIDATION
# ------------------------------------------------------------

core_validation = pd.DataFrame(
    [
        {
            "validation":
                "Candidate-occupation pairs",

            "expected":
                945,

            "actual":
                len(
                    final_hybrid
                ),

            "status":
                (
                    "PASS"
                    if len(
                        final_hybrid
                    ) == 945
                    else "FAIL"
                )
        },

        {
            "validation":
                "Candidates",

            "expected":
                63,

            "actual":
                final_hybrid[
                    "candidate_id"
                ].nunique(),

            "status":
                (
                    "PASS"
                    if final_hybrid[
                        "candidate_id"
                    ].nunique() == 63
                    else "FAIL"
                )
        },

        {
            "validation":
                "Occupations",

            "expected":
                15,

            "actual":
                final_hybrid[
                    "selected_occupation"
                ].nunique(),

            "status":
                (
                    "PASS"
                    if final_hybrid[
                        "selected_occupation"
                    ].nunique() == 15
                    else "FAIL"
                )
        },

        {
            "validation":
                "Skill-gap rows",

            "expected":
                9450,

            "actual":
                len(
                    skill_gap_base
                ),

            "status":
                (
                    "PASS"
                    if len(
                        skill_gap_base
                    ) == 9450
                    else "FAIL"
                )
        },

        {
            "validation":
                "RAG knowledge documents",

            "expected":
                707,

            "actual":
                len(
                    rag_documents
                ),

            "status":
                (
                    "PASS"
                    if len(
                        rag_documents
                    ) == 707
                    else "FAIL"
                )
        },

        {
            "validation":
                "Candidate RAG explanations",

            "expected":
                63,

            "actual":
                len(
                    all_candidate_rag_explanations
                ),

            "status":
                (
                    "PASS"
                    if len(
                        all_candidate_rag_explanations
                    ) == 63
                    else "FAIL"
                )
        },

        {
            "validation":
                "Ablation candidate-occupation rows",

            "expected":
                945,

            "actual":
                len(
                    ablation_results
                ),

            "status":
                (
                    "PASS"
                    if len(
                        ablation_results
                    ) == 945
                    else "FAIL"
                )
        }
    ]
)


print(
    "\n4. CORE DATA VALIDATION"
)

display(
    core_validation
)


# ------------------------------------------------------------
# 6. MASTER VALIDATION SUMMARY
# ------------------------------------------------------------

all_files_pass = (
    passing_file_count
    == required_file_count
)


all_core_checks_pass = (
    core_validation[
        "status"
    ]
    .eq(
        "PASS"
    )
    .all()
)


implementation_requirements_ready = (
    requirement_coverage[
        "status"
    ]
    .isin(
        [
            "COMPLETE",
            "FRAMEWORK COMPLETE — HUMAN REVIEW PENDING"
        ]
    )
    .all()
)


week7_master_validation = pd.DataFrame(
    [
        {
            "validation_area":
                "Required Week 7 Files",

            "status":
                (
                    "PASS"
                    if all_files_pass
                    else "FAIL"
                )
        },

        {
            "validation_area":
                "Core Data Integrity",

            "status":
                (
                    "PASS"
                    if all_core_checks_pass
                    else "FAIL"
                )
        },

        {
            "validation_area":
                "Professor Requirement Coverage",

            "status":
                (
                    "PASS"
                    if implementation_requirements_ready
                    else "FAIL"
                )
        },

        {
            "validation_area":
                "Human Ground-Truth Review",

            "status":
                (
                    "COMPLETE"
                    if (
                        skill_extraction_review_completion == 100
                        and recommendation_review_completion == 100
                        and skill_gap_review_completion == 100
                        and rag_review_completion_percentage == 100
                    )
                    else "PENDING — DO NOT INVENT METRICS"
                )
        }
    ]
)


print(
    "\n5. MASTER WEEK 7 VALIDATION"
)

display(
    week7_master_validation
)


# ------------------------------------------------------------
# 7. EXPORT VALIDATION TABLES
# ------------------------------------------------------------

week7_file_validation.to_csv(
    WEEK7_OUTPUT_DIR
    / "week7_master_file_validation.csv",
    index=False
)


human_review_status.to_csv(
    WEEK7_OUTPUT_DIR
    / "week7_human_review_status.csv",
    index=False
)


requirement_coverage.to_csv(
    WEEK7_OUTPUT_DIR
    / "week7_requirement_coverage.csv",
    index=False
)


core_validation.to_csv(
    WEEK7_OUTPUT_DIR
    / "week7_core_validation.csv",
    index=False
)


week7_master_validation.to_csv(
    WEEK7_OUTPUT_DIR
    / "week7_master_validation.csv",
    index=False
)


# ------------------------------------------------------------
# 8. FINAL WEEK 7 STATUS
# ------------------------------------------------------------

if (
    all_files_pass
    and all_core_checks_pass
    and implementation_requirements_ready
):

    print(
        "\n"
        + "=" * 70
    )

    print(
        "✅ WEEK 7 IMPLEMENTATION VALIDATION PASSED"
    )

    print(
        "=" * 70
    )

    print(
        "\nPrescriptive analytics: COMPLETE"
    )

    print(
        "RAG knowledge base and retrieval: COMPLETE"
    )

    print(
        "Grounded candidate explanations: COMPLETE"
    )

    print(
        "Evaluation frameworks: COMPLETE"
    )

    print(
        "Dataset ablation analysis: COMPLETE"
    )

    print(
        "Error-analysis framework: COMPLETE"
    )

    print(
        "Project limitations: COMPLETE"
    )

    print(
        "\nHuman ground-truth evaluation:"
    )

    if (
        skill_extraction_review_completion == 100
        and recommendation_review_completion == 100
        and skill_gap_review_completion == 100
        and rag_review_completion_percentage == 100
    ):

        print(
            "COMPLETE"
        )

    else:

        print(
            "PENDING — final human-dependent metrics "
            "must not be reported until review is completed."
        )

else:

    print(
        "\n❌ WEEK 7 VALIDATION FAILED"
    )

    print(
        "Review failed validation areas before "
        "moving to Week 8."
    )

WEEK 7 — MASTER DELIVERABLE AND REQUIREMENT VALIDATION

1. WEEK 7 OUTPUT FILE VALIDATION


,deliverable,filename,exists,file_size_bytes,status
0,Missing Skill Priorities,week7_missing_skill_priorities.csv,True,456045,PASS
1,Candidate Prescriptive Recommendations,week7_candidate_prescriptive_recommendations.csv,True,759048,PASS
2,Top-1 Prescriptive Skill Gaps,week7_top1_prescriptive_skill_gaps.csv,True,60603,PASS
3,Final Prescriptive Recommendations,week7_final_prescriptive_recommendations.csv,True,15387,PASS
4,Prescriptive Validation Summary,week7_prescriptive_validation_summary.csv,True,382,PASS
5,RAG Knowledge Base,week7_rag_knowledge_base.csv,True,219274,PASS
6,RAG Embeddings,week7_rag_embeddings.npy,True,1086080,PASS
7,FAISS Index,week7_rag_faiss.index,True,1085997,PASS
8,Candidate RAG Explanations,week7_candidate_rag_explanations.csv,True,202789,PASS
9,RAG Validation Summary,week7_rag_validation_summary.csv,True,425,PASS



Required files: 21
Files found: 21
Files passing validation: 21

2. HUMAN-REVIEW STATUS


,evaluation,review_rows,completion_percentage,status
0,Skill Extraction,200,0.0,Pending Human Review
1,Recommendation Ranking,300,0.0,Pending Human Review
2,Skill Gap,200,0.0,Pending Human Review
3,RAG,20,0.0,Pending Human Review



3. PROFESSOR REQUIREMENT COVERAGE


...


4. CORE DATA VALIDATION


,validation,expected,actual,status
0,Candidate-occupation pairs,945,945,PASS
1,Candidates,63,63,PASS
2,Occupations,15,15,PASS
3,Skill-gap rows,9450,9450,PASS
4,RAG knowledge documents,707,707,PASS
5,Candidate RAG explanations,63,63,PASS
6,Ablation candidate-occupation rows,945,945,PASS



5. MASTER WEEK 7 VALIDATION


,validation_area,status
0,Required Week 7 Files,PASS
1,Core Data Integrity,PASS
2,Professor Requirement Coverage,PASS
3,Human Ground-Truth Review,PENDING — DO NOT INVENT METRICS



✅ WEEK 7 IMPLEMENTATION VALIDATION PASSED

Prescriptive analytics: COMPLETE
RAG knowledge base and retrieval: COMPLETE
Grounded candidate explanations: COMPLETE
Evaluation frameworks: COMPLETE
Dataset ablation analysis: COMPLETE
Error-analysis framework: COMPLETE
Project limitations: COMPLETE

Human ground-truth evaluation:
PENDING — final human-dependent metrics must not be reported until review is completed.
